In [15]:
import pandas as pd
import numpy as np
import os
import zipfile
import json
import re
import warnings

from pathlib import Path

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Dataset directory
DATA_DIR = Path(r"C:\Users\pavan")

print("PRESSURE-X | Data Preparation")
print("Directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

PRESSURE-X | Data Preparation
Directory: C:\Users\pavan
Directory exists: True


In [16]:
# Existing datasets
files = {
    "ball_by_ball": "ball_by_ball.csv",
    "matches": "matches.csv",
    "playing_eleven": "playing_eleven.csv",
    "competitions": "competitions.csv",
    "competition_mapping": "competition_team_player_mapping.csv",
    "teams": "teams.csv",
    "smat": "SMAT_ball_by_ball.csv"
}

data = {}

# Load existing CSV files
for name, filename in files.items():
    path = DATA_DIR / filename

    if path.exists():
        data[name] = pd.read_csv(path, low_memory=False)
    else:
        print(f"Missing: {filename}")

# Load BBL and SA20 from ZIP archives
archives = {
    "bbl": "BBL_csv.zip",
    "sa20": "SA20_csv.zip"
}

for name, filename in archives.items():
    path = DATA_DIR / filename

    if not path.exists():
        print(f"Missing archive: {filename}")
        continue

    with zipfile.ZipFile(path, "r") as z:
        csv_files = [
            f for f in z.namelist()
            if Path(f).name.lower() == "all_matches.csv"
        ]

        if csv_files:
            data[name] = pd.read_csv(
                z.open(csv_files[0]),
                low_memory=False
            )
        else:
            print(f"No all_matches.csv found in {filename}")

# Dataset summary
print("\nDATASET SUMMARY")
print("-" * 55)

for name, df in data.items():
    print(f"{name:22} {df.shape}")

print("-" * 55)
print("Total datasets loaded:", len(data))

Missing archive: BBL_csv.zip
Missing archive: SA20_csv.zip

DATASET SUMMARY
-------------------------------------------------------
ball_by_ball           (200999, 33)
matches                (883, 22)
playing_eleven         (28381, 9)
competitions           (141, 4)
competition_mapping    (6413, 6)
teams                  (353, 3)
smat                   (158839, 28)
-------------------------------------------------------
Total datasets loaded: 7


In [14]:
import zipfile
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"C:\Users\pavan")

archives = {
    "bbl": DATA_DIR / "BBL_csv.zip",
    "sa20": DATA_DIR / "SA20_csv.zip"
}

for name, path in archives.items():
    if not path.exists():
        print(f"{name.upper()}: ZIP not found")
        continue

    with zipfile.ZipFile(path) as z:
        csv_files = [f for f in z.namelist()
                     if f.lower().endswith(".csv")]

        print(f"\n{name.upper()} CSV files:", len(csv_files))
        print("Sample files:", csv_files[:5])

        consolidated = [
            f for f in csv_files
            if Path(f).name.lower() == "all_matches.csv"
        ]

        if consolidated:
            data[name] = pd.read_csv(
                z.open(consolidated[0]),
                low_memory=False
            )
            print("Loaded:", data[name].shape)
        else:
            print("No all_matches.csv found")

BBL: ZIP not found
SA20: ZIP not found


In [17]:
from pathlib import Path

search_dirs = [
    Path(r"C:\Users\pavan"),
    Path(r"C:\Users\pavan\Downloads")
]

for folder in search_dirs:
    print(f"\nSearching: {folder}")

    for file in folder.glob("*.zip"):
        name = file.name.lower()

        if any(x in name for x in ["bbl", "sa20", "sat"]):
            print(file)


Searching: C:\Users\pavan
C:\Users\pavan\bbl_csv2.zip
C:\Users\pavan\sat_csv2.zip

Searching: C:\Users\pavan\Downloads
C:\Users\pavan\Downloads\bbl_csv2.zip
C:\Users\pavan\Downloads\sat_csv2.zip


In [18]:
archives = {
    "bbl": DATA_DIR / "bbl_csv2.zip",
    "sa20": DATA_DIR / "sat_csv2.zip"
}

for name, path in archives.items():

    with zipfile.ZipFile(path, "r") as z:

        csv_files = [
            f for f in z.namelist()
            if Path(f).name.lower() == "all_matches.csv"
        ]

        if not csv_files:
            print(f"{name.upper()}: all_matches.csv not found")
            continue

        df = pd.read_csv(
            z.open(csv_files[0]),
            low_memory=False
        )

        data[name] = df

        print(f"{name.upper()}: {df.shape}")
        print("Matches:", df["match_id"].nunique())

print("\nTotal datasets loaded:", len(data))

BBL: (153250, 27)
Matches: 662
SA20: (29020, 27)
Matches: 130

Total datasets loaded: 9


In [19]:
# Compare ball-by-ball dataset schemas
ball_datasets = [
    "ball_by_ball",
    "smat",
    "bbl",
    "sa20"
]

for name in ball_datasets:
    df = data[name]

    print(f"\n{'='*55}")
    print(f"{name.upper()}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    print("\nSample:")
    display(df.head(2))


BALL_BY_BALL
Shape: (200999, 33)
Columns: ['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']

Sample:


,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,non_striker_name,bowler_id,bowler_name,batting_type,bowling_type,batting_order,runs,is_four,is_six,is_wicket,dismissal_type_id,dismissal_type,dismissed_player_id,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time
0,13511,1,1,1,1,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,0,False,False,False,NaN,NaN,NaN,NaN,0,0,Pace,0.00,Cover Drive,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC
1,13511,1,1,2,2,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,0,False,False,False,NaN,NaN,NaN,NaN,0,0,Pace,0.00,Flick,Inswing,Over,On-side,2022-01-23 00:00:00 UTC



SMAT
Shape: (158839, 28)
Columns: ['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3', 'competition']

Sample:


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3,competition
0,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.10,0.10,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy
1,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.20,0.20,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy



BBL
Shape: (153250, 27)
Columns: ['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3']

Sample:


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
0,1023581,2016/17,2016-12-20,Sydney Showground Stadium,1,0.10,0.10,Sydney Thunder,Sydney Sixers,KR Patterson,RJ Gibson,SNJ O'Keefe,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1023581,2016/17,2016-12-20,Sydney Showground Stadium,1,0.20,0.20,Sydney Thunder,Sydney Sixers,KR Patterson,RJ Gibson,SNJ O'Keefe,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



SA20
Shape: (29020, 27)
Columns: ['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3']

Sample:


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
0,1343941,2022/23,2023-01-10,"Newlands, Cape Town",1,0.10,0.10,Paarl Royals,MI Cape Town,WJ Lubbe,JC Buttler,GF Linde,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1343941,2022/23,2023-01-10,"Newlands, Cape Town",1,0.20,0.20,Paarl Royals,MI Cape Town,JC Buttler,WJ Lubbe,GF Linde,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# ==========================================
# PRESSURE-X: COMPLETE DATASET AUDIT
# ==========================================

import pandas as pd
import numpy as np

ball_datasets = {
    "IPL_T20": data["ball_by_ball"],
    "SMAT": data["smat"],
    "BBL": data["bbl"],
    "SA20": data["sa20"]
}

summary = []
player_coverage = []
missing_report = []

for name, df in ball_datasets.items():

    # Standardise column names for inspection only
    cols = {c.lower().strip(): c for c in df.columns}

    batter_col = cols.get("striker") or cols.get("batter")
    bowler_col = cols.get("bowler")
    match_col = cols.get("match_id")
    season_col = cols.get("season")
    team_col = cols.get("batting_team")

    # Dataset summary
    summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Matches": df[match_col].nunique() if match_col else np.nan,
        "Batters": df[batter_col].nunique() if batter_col else np.nan,
        "Bowlers": df[bowler_col].nunique() if bowler_col else np.nan,
        "Teams": df[team_col].nunique() if team_col else np.nan,
        "Seasons": df[season_col].nunique() if season_col else np.nan,
        "Duplicate_Rows": df.duplicated().sum(),
        "Missing_Cells": df.isna().sum().sum()
    })

    # Missing-value report
    for col, count in df.isna().sum().items():
        if count > 0:
            missing_report.append({
                "Dataset": name,
                "Column": col,
                "Missing": count,
                "Missing_Percent": round(count / len(df) * 100, 2)
            })

    # Player delivery coverage
    if batter_col:
        counts = df[batter_col].value_counts()

        for player, deliveries in counts.items():
            if pd.notna(player):
                player_coverage.append({
                    "Dataset": name,
                    "Player": player,
                    "Batting_Deliveries": deliveries
                })

# Create audit tables
audit_summary = pd.DataFrame(summary)

player_coverage_df = pd.DataFrame(player_coverage)

missing_df = pd.DataFrame(missing_report)

# Display results
print("=" * 70)
print("PRESSURE-X | DATASET SUMMARY")
print("=" * 70)
display(audit_summary)

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)
display(missing_df)

print("\n" + "=" * 70)
print("TOP 15 PLAYERS BY DELIVERIES IN EACH DATASET")
print("=" * 70)

for name in ball_datasets:
    print(f"\n{name}")
    display(
        player_coverage_df[
            player_coverage_df["Dataset"] == name
        ].nlargest(15, "Batting_Deliveries")
    )

# Save audit outputs
audit_summary.to_csv(DATA_DIR / "dataset_audit.csv", index=False)
player_coverage_df.to_csv(DATA_DIR / "player_coverage.csv", index=False)
missing_df.to_csv(DATA_DIR / "missing_values_audit.csv", index=False)

print("\nAudit completed. Three reports saved.")

PRESSURE-X | DATASET SUMMARY


,Dataset,Rows,Columns,Matches,Batters,Bowlers,Teams,Seasons,Duplicate_Rows,Missing_Cells
0,IPL_T20,200999,33,867,NaN,NaN,NaN,NaN,0,766466
1,SMAT,158839,28,695,1300.00,970.00,39.00,9.00,0,2035311
2,BBL,153250,27,662,503.00,372.00,8.00,15.00,0,1961698
3,SA20,29020,27,130,199.00,144.00,6.00,4.00,0,371300



MISSING VALUES


,Dataset,Column,Missing,Missing_Percent
0,IPL_T20,dismissal_type_id,190118,94.59
1,IPL_T20,dismissal_type,190118,94.59
2,IPL_T20,dismissed_player_id,190107,94.58
3,IPL_T20,extras_type,189167,94.11
4,IPL_T20,stroke,6158,3.06
5,IPL_T20,delivery_style,391,0.19
6,IPL_T20,over_or_around,407,0.20
7,SMAT,wides,154816,97.47
8,SMAT,noballs,158405,99.73
9,SMAT,byes,158392,99.72



TOP 15 PLAYERS BY DELIVERIES IN EACH DATASET

IPL_T20


,Dataset,Player,Batting_Deliveries


... (3 outputs left out)

,Dataset,Player,Batting_Deliveries
1300,BBL,CA Lynn,2812
1301,BBL,MC Henriques,2654
1302,BBL,AJ Finch,2616
1303,BBL,MP Stoinis,2476
1304,BBL,JW Wells,2470
1305,BBL,DJM Short,2468
1306,BBL,BR McDermott,2313
1307,BBL,SE Marsh,2208
1308,BBL,GJ Maxwell,2154
1309,BBL,MS Wade,2147



SA20


,Dataset,Player,Batting_Deliveries
1803,SA20,AK Markram,956
1804,SA20,HE van der Dussen,932
1805,SA20,RD Rickelton,867
1806,SA20,J Hermann,822
1807,SA20,Q de Kock,775
1808,SA20,MP Breetzke,772
1809,SA20,RR Hendricks,760
1810,SA20,F du Plessis,733
1811,SA20,D Brevis,725
1812,SA20,H Klaasen,716



Audit completed. Three reports saved.


In [21]:
display(data["competitions"].to_string(index=False))

' comp_id                                               comp_name match_format  season\n    3395            South Africa in New Zealand T20I Series 2026         T20I    2026\n    3405                Afghanistan v Sri Lanka T20I Series 2026         T20I    2026\n    3465                      India in Zimbabwe T20I Series 2026         T20I    2026\n    3568                 Bangladesh in Zimbabwe T20I Series 2026         T20I    2026\n    3482                Australia in Bangladesh T20I Series 2026         T20I    2026\n    3464               Sri Lanka in West Indies T20I Series 2026         T20I    2026\n    3765                  Sri Lanka in Pakistan T20I Series 2026         T20I    2026\n    3739                          India in Japan T20I Match 2026         T20I    2026\n    3787                 Afghanistan v Zimbabwe T20I Series 2026         T20I    2026\n    3713          Afghanistan vs India in India T20I Series 2026         T20I    2026\n    3690                   India in New Ze

In [22]:
# Inspect relationships between matches and competitions

print("MATCHES COLUMNS:")
print(data["matches"].columns.tolist())

print("\nBALL-BY-BALL COLUMNS:")
print(data["ball_by_ball"].columns.tolist())

print("\nCOMPETITION MAPPING COLUMNS:")
print(data["competition_mapping"].columns.tolist())

# Check match-level competition coverage
matches = data["matches"]
competitions = data["competitions"]

print("\nMATCH DATA SAMPLE:")
display(matches.head())

print("\nCOMPETITION DATA SAMPLE:")
display(competitions.head())

MATCHES COLUMNS:
['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result']

BALL-BY-BALL COLUMNS:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']

COMPETITION MAPPING COLUMNS:
['comp_id', 'comp_name', 'team_id', 'team_name', 'p

,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,away_team,home_team_captain_id,away_team_captain_id,toss_win_team_id,toss_opted,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result
0,13511,1331,1,2022-01-22,League,54,West Indies,28,West Indies,22,England,NaN,NaN,28.00,Fielding,0,NaN,NaN,NaN,0,28.00,West Indies Won by 9 Wickets
1,13512,1331,2,2022-01-23,League,54,West Indies,28,West Indies,22,England,494.00,478.00,28.00,Fielding,0,NaN,NaN,NaN,0,22.00,England Won by 1 Run
2,13513,1331,3,2022-01-26,League,54,West Indies,28,West Indies,22,England,494.00,1596.00,22.00,Fielding,4986,NaN,NaN,NaN,0,28.00,West Indies Won by 20 runs
3,13514,1331,4,2022-01-29,League,54,West Indies,28,West Indies,22,England,494.00,1596.00,28.00,Fielding,0,NaN,NaN,NaN,0,22.00,England Won by 34 runs
4,13515,1331,5,2022-01-30,League,54,West Indies,28,West Indies,22,England,494.00,NaN,28.00,Batting,0,NaN,NaN,NaN,0,28.00,West Indies Won by 17 runs



COMPETITION DATA SAMPLE:


,comp_id,comp_name,match_format,season
0,3395,South Africa in New Zealand T20I Series 2026,T20I,2026
1,3405,Afghanistan v Sri Lanka T20I Series 2026,T20I,2026
2,3465,India in Zimbabwe T20I Series 2026,T20I,2026
3,3568,Bangladesh in Zimbabwe T20I Series 2026,T20I,2026
4,3482,Australia in Bangladesh T20I Series 2026,T20I,2026


In [23]:
# ==========================================
# PRESSURE-X: IPL AND T20I VERIFICATION
# ==========================================

matches = data["matches"].copy()
competitions = data["competitions"].copy()
balls = data["ball_by_ball"].copy()

# Standardise column names
for df in [matches, competitions, balls]:
    df.columns = df.columns.str.strip().str.lower()

# Match competition metadata
match_comp = matches.merge(
    competitions[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left"
)

# Match IDs by format
match_comp["match_format"] = match_comp["match_format"].astype(str).str.upper()

for fmt in ["T20", "T20I"]:
    selected = match_comp[match_comp["match_format"] == fmt]
    match_ids = selected["match_id"].unique()

    deliveries = balls[balls["match_id"].isin(match_ids)]

    print(f"\n{'='*45}")
    print(f"{fmt} COVERAGE")
    print("Matches:", selected["match_id"].nunique())
    print("Deliveries:", len(deliveries))
    print("Seasons:", sorted(selected["season"].dropna().astype(str).unique()))

    display(
        selected.groupby("season")
        .agg(Matches=("match_id", "nunique"))
        .reset_index()
    )

# Save verified match metadata
match_comp.to_csv(DATA_DIR / "verified_match_metadata.csv", index=False)


T20 COVERAGE
Matches: 401
Deliveries: 94672
Seasons: ['2022', '2023', '2024', '2025', '2026']


,season,Matches
0,2022,74
1,2023,74
2,2024,89
3,2025,90
4,2026,74



T20I COVERAGE
Matches: 482
Deliveries: 106327
Seasons: ['2022', '2022/23', '2023', '2023/24', '2024', '2024/25', '2025', '2026']


,season,Matches
0,2022,94
1,2022/23,12
2,2023,63
3,2023/24,3
4,2024,109
5,2024/25,3
6,2025,126
7,2026,72


In [24]:
# ==========================================
# PRESSURE-X: CONSOLIDATED BALL-BY-BALL DATA
# ==========================================

# Prepare source datasets
datasets = {
    "IPL_T20_T20I": data["ball_by_ball"].copy(),
    "SMAT": data["smat"].copy(),
    "BBL": data["bbl"].copy(),
    "SA20": data["sa20"].copy()
}

combined = []

for competition, df in datasets.items():

    df.columns = df.columns.str.strip().str.lower()

    # Preserve original match ID
    df["original_match_id"] = df["match_id"].astype(str)

    # Prevent match ID collisions across competitions
    df["match_id"] = competition + "_" + df["original_match_id"]

    # Add source competition
    df["source_competition"] = competition

    combined.append(df)

# Combine all datasets
master = pd.concat(combined, ignore_index=True, sort=False)

# Attach IPL/T20I competition metadata
metadata = data["verified_match_metadata"].copy() if "verified_match_metadata" in data else pd.read_csv(
    DATA_DIR / "verified_match_metadata.csv"
)

metadata.columns = metadata.columns.str.strip().str.lower()
metadata["match_id"] = "IPL_T20_T20I_" + metadata["match_id"].astype(str)

master = master.merge(
    metadata[["match_id", "comp_name", "match_format", "season"]].drop_duplicates("match_id"),
    on="match_id",
    how="left"
)

# Assign competition names to other datasets
master.loc[master["source_competition"] == "SMAT", "comp_name"] = "Syed Mushtaq Ali Trophy"
master.loc[master["source_competition"] == "BBL", "comp_name"] = "Big Bash League"
master.loc[master["source_competition"] == "SA20", "comp_name"] = "SA20"

# Save master dataset
master.to_csv(DATA_DIR / "PRESSURE_X_Master_Ball_by_Ball.csv", index=False)

# Summary
print("MASTER DATASET CREATED")
print("Shape:", master.shape)
print("Unique matches:", master["match_id"].nunique())
print("Total deliveries:", len(master))

print("\nCompetition coverage:")
display(
    master.groupby("comp_name")
    .agg(
        Deliveries=("match_id", "size"),
        Matches=("match_id", "nunique")
    )
    .sort_values("Matches", ascending=False)
)

print("\nSaved: PRESSURE_X_Master_Ball_by_Ball.csv")

MASTER DATASET CREATED
Shape: (542108, 64)
Unique matches: 2354
Total deliveries: 542108

Competition coverage:


,Deliveries,Matches
comp_name,,
Syed Mushtaq Ali Trophy,158839,695
Big Bash League,153250,662
SA20,29020,130
IPL 2025,17285,74
IPL 2023,17863,74
...,...,...
Pakistan in England T20I Series 2024,461,2
West Indies in Australia T20I Series 2022,504,2
South Africa vs Ireland T20I Series 2022,501,2



Saved: PRESSURE_X_Master_Ball_by_Ball.csv


In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load master dataset
master = pd.read_csv(DATA_DIR / "PRESSURE_X_Master_Ball_by_Ball.csv", low_memory=False)
master.columns = master.columns.str.strip().str.lower()

# Detect available column names
def find_col(candidates, required=True):
    for col in candidates:
        if col in master.columns:
            return col
    if required:
        raise ValueError(
            f"Missing required column. Tried: {candidates}\n"
            f"Available columns: {master.columns.tolist()}"
        )
    return None

batter_col = find_col(["striker", "batter", "batsman"])
bowler_col = find_col(["bowler"])
runs_col = find_col(["runs_off_bat", "runs_batter", "batter_runs", "runs_scored"])
extras_col = find_col(["extras", "total_extras"], required=False)
wides_col = find_col(["wides", "wide_runs"], required=False)
noballs_col = find_col(["noballs", "no_balls", "noball_runs"], required=False)
wicket_col = find_col(["player_dismissed", "player_out", "dismissed_player"], required=False)
wicket_type_col = find_col(["wicket_type", "dismissal_kind"], required=False)

# Standardize columns
df = master.copy()

df["batter"] = df[batter_col].astype("string").str.strip()
df["bowler_name"] = df[bowler_col].astype("string").str.strip()

df["bat_runs"] = pd.to_numeric(df[runs_col], errors="coerce").fillna(0)

df["wides"] = (
    pd.to_numeric(df[wides_col], errors="coerce").fillna(0)
    if wides_col else 0
)
df["noballs"] = (
    pd.to_numeric(df[noballs_col], errors="coerce").fillna(0)
    if noballs_col else 0
)

df["is_legal_ball"] = (df["wides"] == 0) & (df["noballs"] == 0)
df["is_batter_ball"] = df["wides"] == 0

df["dismissed_player"] = (
    df[wicket_col].astype("string").str.strip()
    if wicket_col else pd.NA
)

df["dismissal_type"] = (
    df[wicket_type_col].astype("string").str.lower().str.strip()
    if wicket_type_col else ""
)

df["is_wicket"] = df["dismissed_player"].notna()

# Wickets credited to the bowler
non_bowler_dismissals = [
    "run out", "retired hurt", "retired out",
    "obstructing the field", "retired"
]
df["bowler_wicket"] = (
    df["is_wicket"] &
    ~df["dismissal_type"].isin(non_bowler_dismissals)
)

# Competition and match information
df["competition"] = (
    df["comp_name"].fillna(df["source_competition"])
    if "comp_name" in df.columns else df["source_competition"]
)
df["match_id"] = df["match_id"].astype(str)

# ---------------- BATTING STATISTICS ----------------

batting = df[df["batter"].notna()].groupby("batter").agg(
    runs=("bat_runs", "sum"),
    balls_faced=("is_batter_ball", "sum"),
    innings=("match_id", "nunique"),
    fours=("bat_runs", lambda x: (x == 4).sum()),
    sixes=("bat_runs", lambda x: (x == 6).sum()),
    dismissals=("batter", lambda x: 0)
).reset_index()

# Actual dismissals for each batter
dismissal_counts = (
    df[df["dismissed_player"].notna()]
    .groupby("dismissed_player")
    .size()
    .rename("dismissals")
)

batting = batting.drop(columns="dismissals").merge(
    dismissal_counts, left_on="batter", right_index=True, how="left"
)
batting["dismissals"] = batting["dismissals"].fillna(0).astype(int)

batting["strike_rate"] = np.where(
    batting["balls_faced"] > 0,
    batting["runs"] / batting["balls_faced"] * 100,
    0
)
batting["average"] = np.where(
    batting["dismissals"] > 0,
    batting["runs"] / batting["dismissals"],
    np.nan
)
batting["boundary_runs"] = batting["fours"] * 4 + batting["sixes"] * 6
batting["boundary_percentage"] = np.where(
    batting["runs"] > 0,
    batting["boundary_runs"] / batting["runs"] * 100,
    0
)

# Competition-level batting performance
batting_by_competition = df[df["batter"].notna()].groupby(
    ["batter", "competition"]
).agg(
    runs=("bat_runs", "sum"),
    balls_faced=("is_batter_ball", "sum"),
    matches=("match_id", "nunique")
).reset_index()

batting_by_competition["strike_rate"] = np.where(
    batting_by_competition["balls_faced"] > 0,
    batting_by_competition["runs"] / batting_by_competition["balls_faced"] * 100,
    0
)

# ---------------- BOWLING STATISTICS ----------------

bowling = df[df["bowler_name"].notna()].groupby("bowler_name").agg(
    balls=("is_legal_ball", "sum"),
    runs_off_bat_conceded=("bat_runs", "sum"),
    wickets=("bowler_wicket", "sum"),
    matches=("match_id", "nunique")
).reset_index()

# Extras charged to bowler: wides and no-balls
bowler_extras = df.groupby("bowler_name").agg(
    wides=("wides", "sum"),
    noballs=("noballs", "sum")
).reset_index()

bowling = bowling.merge(bowler_extras, on="bowler_name", how="left")
bowling["runs_conceded"] = (
    bowling["runs_off_bat_conceded"] +
    bowling["wides"] +
    bowling["noballs"]
)

bowling["overs"] = bowling["balls"] / 6
bowling["economy"] = np.where(
    bowling["balls"] > 0,
    bowling["runs_conceded"] / bowling["overs"],
    0
)
bowling["bowling_average"] = np.where(
    bowling["wickets"] > 0,
    bowling["runs_conceded"] / bowling["wickets"],
    np.nan
)
bowling["strike_rate"] = np.where(
    bowling["wickets"] > 0,
    bowling["balls"] / bowling["wickets"],
    np.nan
)

# Competition-level bowling performance
bowling_by_competition = df[df["bowler_name"].notna()].groupby(
    ["bowler_name", "competition"]
).agg(
    balls=("is_legal_ball", "sum"),
    runs=("bat_runs", "sum"),
    wickets=("bowler_wicket", "sum"),
    matches=("match_id", "nunique")
).reset_index()

bowling_by_competition["overs"] = bowling_by_competition["balls"] / 6
bowling_by_competition["economy"] = np.where(
    bowling_by_competition["overs"] > 0,
    bowling_by_competition["runs"] / bowling_by_competition["overs"],
    0
)

# Save outputs
outputs = {
    "PRESSURE_X_Batting_Stats.csv": batting,
    "PRESSURE_X_Bowling_Stats.csv": bowling,
    "PRESSURE_X_Batting_By_Competition.csv": batting_by_competition,
    "PRESSURE_X_Bowling_By_Competition.csv": bowling_by_competition
}

for filename, table in outputs.items():
    table.to_csv(DATA_DIR / filename, index=False)

print("PLAYER STATISTICS CREATED")
print("Unique batters:", len(batting))
print("Unique bowlers:", len(bowling))
print("Batting records:", len(batting_by_competition))
print("Bowling records:", len(bowling_by_competition))
print("\nTop 10 run scorers:")
display(batting.sort_values("runs", ascending=False).head(10))
print("\nTop 10 wicket takers:")
display(bowling.sort_values("wickets", ascending=False).head(10))
print("\nSaved all four player-statistics files.")

PLAYER STATISTICS CREATED
Unique batters: 1935
Unique bowlers: 1442
Batting records: 2002
Bowling records: 1486

Top 10 run scorers:


,batter,runs,balls_faced,innings,fours,sixes,dismissals,strike_rate,average,boundary_runs,boundary_percentage
370,CA Lynn,4088.00,2735,133,343,225,116,149.47,35.24,2722,66.59
990,MC Henriques,3356.00,2593,143,246,102,121,129.43,27.74,1596,47.56
1026,MP Stoinis,3336.00,2508,121,294,123,100,133.01,33.36,1914,57.37
59,AJ Finch,3310.00,2522,104,292,118,95,131.25,34.84,1876,56.68
567,GJ Maxwell,3196.00,2058,116,263,149,93,155.30,34.37,1946,60.89
323,BR McDermott,3150.00,2279,112,220,149,93,138.22,33.87,1774,56.32
469,DJM Short,3138.00,2387,102,305,107,92,131.46,34.11,1862,59.34
1038,MS Wade,3012.00,2118,114,261,104,101,142.21,29.82,1668,55.38
760,JW Wells,2935.00,2412,123,237,51,87,121.68,33.74,1254,42.73
1507,SE Marsh,2802.00,2157,77,255,89,67,129.90,41.82,1554,55.46



Top 10 wicket takers:


,bowler_name,balls,runs_off_bat_conceded,wickets,matches,wides,noballs,runs_conceded,overs,economy,bowling_average,strike_rate
1106,SA Abbott,2673,3644.00,185,129,73.00,16.00,3733.00,445.50,8.38,20.18,14.45
243,BJ Dwarshuis,2690,3502.00,164,126,121.00,7.00,3630.00,448.33,8.10,22.13,16.40
48,AJ Tye,2507,3240.00,160,116,88.00,19.00,3347.00,417.83,8.01,20.92,15.67
552,JP Behrendorff,2509,2919.00,147,115,67.00,12.00,2998.00,418.17,7.17,20.39,17.07
624,KW Richardson,2476,3197.00,138,113,64.00,7.00,3268.00,412.67,7.92,23.68,17.94
19,A Zampa,2575,3100.00,130,116,33.00,3.00,3136.00,429.17,7.31,24.12,19.81
376,DR Sams,1950,2722.00,128,96,88.00,9.00,2819.00,325.00,8.67,22.02,15.23
912,PM Siddle,2124,2601.00,127,103,65.00,9.00,2675.00,354.00,7.56,21.06,16.72
1055,Rashid Khan,2216,2400.00,125,95,57.00,2.00,2459.00,369.33,6.66,19.67,17.73
1013,RP Meredith,1946,2576.00,117,90,113.00,14.00,2703.00,324.33,8.33,23.10,16.63



Saved all four player-statistics files.


In [27]:
import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.str.strip().str.lower()

# Standardize columns
df["batter"] = df["striker"].astype("string").str.strip()
df["bowler_name"] = df["bowler"].astype("string").str.strip()
df["runs"] = pd.to_numeric(df["runs_off_bat"], errors="coerce").fillna(0)

# Handle extras
for col in ["wides", "noballs", "byes", "legbyes"]:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Derive season safely
if "season" not in df.columns:
    if "start_date" in df.columns:
        df["season"] = pd.to_datetime(
            df["start_date"], errors="coerce"
        ).dt.year
    else:
        raise ValueError("Neither season nor start_date exists in master.")

df["season"] = df["season"].astype("string")

# Competition and innings
df["competition"] = (
    df["comp_name"].fillna(df["source_competition"])
    if "comp_name" in df.columns else df["source_competition"]
)

df["innings_key"] = (
    df["match_id"].astype(str) + "_" + df["innings"].astype(str)
)

# Delivery calculations
df["is_legal_ball"] = (df["wides"] == 0) & (df["noballs"] == 0)
df["batter_ball"] = df["wides"] == 0

df["dismissed_player"] = df["player_dismissed"].astype("string").str.strip()
df["dismissal_type"] = df["wicket_type"].astype("string").str.lower().str.strip()

df["is_wicket"] = df["dismissed_player"].notna()

non_bowler_dismissals = [
    "run out", "retired hurt", "retired out",
    "obstructing the field", "retired"
]

df["bowler_wicket"] = (
    df["is_wicket"] &
    ~df["dismissal_type"].isin(non_bowler_dismissals)
)

# ---------------- BATTING ----------------

batting = df[df["batter"].notna()].groupby("batter").agg(
    runs=("runs", "sum"),
    balls_faced=("batter_ball", "sum"),
    matches=("match_id", "nunique"),
    innings=("innings_key", "nunique"),
    fours=("runs", lambda x: (x == 4).sum()),
    sixes=("runs", lambda x: (x == 6).sum())
).reset_index()

dismissals = (
    df[df["is_wicket"]]
    .groupby("dismissed_player")
    .size()
    .rename("dismissals")
)

batting = batting.merge(
    dismissals, left_on="batter", right_index=True, how="left"
)
batting["dismissals"] = batting["dismissals"].fillna(0)

batting["average"] = np.where(
    batting["dismissals"] > 0,
    batting["runs"] / batting["dismissals"], np.nan
)

batting["strike_rate"] = np.where(
    batting["balls_faced"] > 0,
    batting["runs"] / batting["balls_faced"] * 100, 0
)

batting["boundary_percentage"] = np.where(
    batting["runs"] > 0,
    (batting["fours"] * 4 + batting["sixes"] * 6) / batting["runs"] * 100,
    0
)

# Season-wise batting
batting_season = df[df["batter"].notna()].groupby(
    ["batter", "competition", "season"]
).agg(
    runs=("runs", "sum"),
    balls_faced=("batter_ball", "sum"),
    matches=("match_id", "nunique"),
    innings=("innings_key", "nunique"),
    fours=("runs", lambda x: (x == 4).sum()),
    sixes=("runs", lambda x: (x == 6).sum())
).reset_index()

batting_season["strike_rate"] = np.where(
    batting_season["balls_faced"] > 0,
    batting_season["runs"] / batting_season["balls_faced"] * 100, 0
)

# ---------------- BOWLING ----------------

df["bowler_runs_conceded"] = df["runs"] + df["wides"] + df["noballs"]

bowling = df[df["bowler_name"].notna()].groupby("bowler_name").agg(
    balls=("is_legal_ball", "sum"),
    runs_conceded=("bowler_runs_conceded", "sum"),
    wickets=("bowler_wicket", "sum"),
    matches=("match_id", "nunique")
).reset_index()

bowling["overs"] = bowling["balls"] / 6

bowling["economy"] = np.where(
    bowling["overs"] > 0,
    bowling["runs_conceded"] / bowling["overs"], 0
)

bowling["average"] = np.where(
    bowling["wickets"] > 0,
    bowling["runs_conceded"] / bowling["wickets"], np.nan
)

bowling["strike_rate"] = np.where(
    bowling["wickets"] > 0,
    bowling["balls"] / bowling["wickets"], np.nan
)

# Season-wise bowling
bowling_season = df[df["bowler_name"].notna()].groupby(
    ["bowler_name", "competition", "season"]
).agg(
    balls=("is_legal_ball", "sum"),
    runs_conceded=("bowler_runs_conceded", "sum"),
    wickets=("bowler_wicket", "sum"),
    matches=("match_id", "nunique")
).reset_index()

bowling_season["overs"] = bowling_season["balls"] / 6

bowling_season["economy"] = np.where(
    bowling_season["overs"] > 0,
    bowling_season["runs_conceded"] / bowling_season["overs"], 0
)

# Save all outputs
outputs = {
    "PRESSURE_X_Batting_Stats_Corrected.csv": batting,
    "PRESSURE_X_Bowling_Stats_Corrected.csv": bowling,
    "PRESSURE_X_Batting_Season.csv": batting_season,
    "PRESSURE_X_Bowling_Season.csv": bowling_season
}

for filename, table in outputs.items():
    table.to_csv(DATA_DIR / filename, index=False)

print("CORRECTED ANALYTICS COMPLETE")
print("Batters:", len(batting))
print("Bowlers:", len(bowling))
print("Batting season records:", len(batting_season))
print("Bowling season records:", len(bowling_season))
print("Season range:", df["season"].min(), "-", df["season"].max())

display(batting.sort_values("runs", ascending=False).head(10))
display(bowling.sort_values("wickets", ascending=False).head(10))

CORRECTED ANALYTICS COMPLETE
Batters: 1935
Bowlers: 1442
Batting season records: 5460
Bowling season records: 3863
Season range: 2011.0 - 2026.0


,batter,runs,balls_faced,matches,innings,fours,sixes,dismissals,average,strike_rate,boundary_percentage
370,CA Lynn,4088.00,2735,133,133,343,225,116.00,35.24,149.47,66.59
990,MC Henriques,3356.00,2593,143,147,246,102,121.00,27.74,129.43,47.56
1026,MP Stoinis,3336.00,2508,121,121,294,123,100.00,33.36,133.01,57.37
59,AJ Finch,3310.00,2522,104,104,292,118,95.00,34.84,131.25,56.68
567,GJ Maxwell,3196.00,2058,116,116,263,149,93.00,34.37,155.30,60.89
323,BR McDermott,3150.00,2279,112,112,220,149,93.00,33.87,138.22,56.32
469,DJM Short,3138.00,2387,102,102,305,107,92.00,34.11,131.46,59.34
1038,MS Wade,3012.00,2118,114,114,261,104,101.00,29.82,142.21,55.38
760,JW Wells,2935.00,2412,123,123,237,51,87.00,33.74,121.68,42.73
1507,SE Marsh,2802.00,2157,77,77,255,89,67.00,41.82,129.90,55.46


,bowler_name,balls,runs_conceded,wickets,matches,overs,economy,average,strike_rate
1106,SA Abbott,2673,3733.00,185,129,445.50,8.38,20.18,14.45
243,BJ Dwarshuis,2690,3630.00,164,126,448.33,8.10,22.13,16.40
48,AJ Tye,2507,3347.00,160,116,417.83,8.01,20.92,15.67
552,JP Behrendorff,2509,2998.00,147,115,418.17,7.17,20.39,17.07
624,KW Richardson,2476,3268.00,138,113,412.67,7.92,23.68,17.94
19,A Zampa,2575,3136.00,130,116,429.17,7.31,24.12,19.81
376,DR Sams,1950,2819.00,128,96,325.00,8.67,22.02,15.23
912,PM Siddle,2124,2675.00,127,103,354.00,7.56,21.06,16.72
1055,Rashid Khan,2216,2459.00,125,95,369.33,6.66,19.67,17.73
1013,RP Meredith,1946,2703.00,117,90,324.33,8.33,23.10,16.63


In [28]:
# Correct individual batting innings
batting_innings = (
    df[df["batter"].notna()]
    .groupby("batter")["innings_key"]
    .nunique()
    .rename("innings")
)

batting["innings"] = batting["batter"].map(batting_innings)

# Correct season-wise individual innings
season_innings = (
    df[df["batter"].notna()]
    .groupby(["batter", "competition", "season"])["innings_key"]
    .nunique()
    .rename("innings")
)

batting_season = batting_season.drop(columns="innings").merge(
    season_innings.reset_index(),
    on=["batter", "competition", "season"],
    how="left"
)

batting.to_csv(DATA_DIR / "PRESSURE_X_Batting_Stats_Corrected.csv", index=False)
batting_season.to_csv(DATA_DIR / "PRESSURE_X_Batting_Season.csv", index=False)

print("Individual batting innings corrected.")

Individual batting innings corrected.


In [29]:
import pandas as pd
import numpy as np

# Use existing master dataframe
smat_df = df[
    df["competition"].astype(str).str.contains(
        "Syed Mushtaq Ali Trophy", case=False, na=False
    )
].copy()

# Recent seasons
smat_df["season_year"] = pd.to_numeric(
    smat_df["season"], errors="coerce"
)

recent = smat_df[smat_df["season_year"].between(2024, 2026)].copy()

print("SMAT deliveries:", len(smat_df))
print("Recent SMAT deliveries:", len(recent))
print("Seasons:", sorted(recent["season_year"].dropna().unique()))

# ---------------- BATTER CANDIDATES ----------------

bat = recent[recent["batter"].notna()].copy()

batting_candidates = bat.groupby("batter").agg(
    runs=("runs", "sum"),
    balls=("batter_ball", "sum"),
    matches=("match_id", "nunique"),
    innings=("innings_key", "nunique"),
    fours=("runs", lambda x: (x == 4).sum()),
    sixes=("runs", lambda x: (x == 6).sum())
).reset_index()

bat_dismissals = (
    recent[recent["is_wicket"]]
    .groupby("dismissed_player")
    .size()
    .rename("dismissals")
)

batting_candidates = batting_candidates.merge(
    bat_dismissals,
    left_on="batter",
    right_index=True,
    how="left"
)

batting_candidates["dismissals"] = (
    batting_candidates["dismissals"].fillna(0)
)

batting_candidates["average"] = np.where(
    batting_candidates["dismissals"] > 0,
    batting_candidates["runs"] / batting_candidates["dismissals"],
    np.nan
)

batting_candidates["strike_rate"] = np.where(
    batting_candidates["balls"] > 0,
    batting_candidates["runs"] / batting_candidates["balls"] * 100,
    0
)

batting_candidates["boundary_percentage"] = np.where(
    batting_candidates["runs"] > 0,
    (4 * batting_candidates["fours"] +
     6 * batting_candidates["sixes"]) /
    batting_candidates["runs"] * 100,
    0
)

# Minimum sample requirement
batting_candidates = batting_candidates[
    (batting_candidates["matches"] >= 5) &
    (batting_candidates["innings"] >= 5) &
    (batting_candidates["balls"] >= 100)
].copy()

# Percentile-based scouting score
for col in ["runs", "average", "strike_rate", "boundary_percentage"]:
    batting_candidates[col + "_pct"] = (
        batting_candidates[col].rank(pct=True) * 100
    )

batting_candidates["scouting_score"] = (
    0.20 * batting_candidates["runs_pct"] +
    0.30 * batting_candidates["average_pct"] +
    0.35 * batting_candidates["strike_rate_pct"] +
    0.15 * batting_candidates["boundary_percentage_pct"]
)

batting_candidates = batting_candidates.sort_values(
    "scouting_score", ascending=False
)

# ---------------- BOWLER CANDIDATES ----------------

bowl = recent[recent["bowler_name"].notna()].copy()

bowling_candidates = bowl.groupby("bowler_name").agg(
    balls=("is_legal_ball", "sum"),
    runs_conceded=("bowler_runs_conceded", "sum"),
    wickets=("bowler_wicket", "sum"),
    matches=("match_id", "nunique")
).reset_index()

bowling_candidates["overs"] = bowling_candidates["balls"] / 6

bowling_candidates["economy"] = np.where(
    bowling_candidates["overs"] > 0,
    bowling_candidates["runs_conceded"] /
    bowling_candidates["overs"],
    np.nan
)

bowling_candidates["average"] = np.where(
    bowling_candidates["wickets"] > 0,
    bowling_candidates["runs_conceded"] /
    bowling_candidates["wickets"],
    np.nan
)

bowling_candidates["strike_rate"] = np.where(
    bowling_candidates["wickets"] > 0,
    bowling_candidates["balls"] /
    bowling_candidates["wickets"],
    np.nan
)

# Minimum sample requirement
bowling_candidates = bowling_candidates[
    (bowling_candidates["matches"] >= 5) &
    (bowling_candidates["overs"] >= 10) &
    (bowling_candidates["wickets"] >= 5)
].copy()

# Percentile scores: lower economy and average are better
bowling_candidates["wickets_pct"] = (
    bowling_candidates["wickets"].rank(pct=True) * 100
)

bowling_candidates["economy_pct"] = (
    bowling_candidates["economy"].rank(pct=True, ascending=False) * 100
)

bowling_candidates["average_pct"] = (
    bowling_candidates["average"].rank(pct=True, ascending=False) * 100
)

bowling_candidates["strike_rate_pct"] = (
    bowling_candidates["strike_rate"].rank(
        pct=True, ascending=False
    ) * 100
)

bowling_candidates["scouting_score"] = (
    0.35 * bowling_candidates["wickets_pct"] +
    0.25 * bowling_candidates["economy_pct"] +
    0.20 * bowling_candidates["average_pct"] +
    0.20 * bowling_candidates["strike_rate_pct"]
)

bowling_candidates = bowling_candidates.sort_values(
    "scouting_score", ascending=False
)

# ---------------- EXPORT ----------------

batting_candidates.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Batter_Candidates.csv",
    index=False
)

bowling_candidates.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Bowler_Candidates.csv",
    index=False
)

print("\nSMAT SCOUTING POOLS CREATED")
print("Eligible batting candidates:", len(batting_candidates))
print("Eligible bowling candidates:", len(bowling_candidates))

print("\nTOP 25 BATTER CANDIDATES")
display(batting_candidates.head(25))

print("\nTOP 25 BOWLER CANDIDATES")
display(bowling_candidates.head(25))

SMAT deliveries: 158839
Recent SMAT deliveries: 1907
Seasons: [np.float64(2024.0)]

SMAT SCOUTING POOLS CREATED
Eligible batting candidates: 0
Eligible bowling candidates: 0

TOP 25 BATTER CANDIDATES


,batter,runs,balls,matches,innings,fours,sixes,dismissals,average,strike_rate,boundary_percentage,runs_pct,average_pct,strike_rate_pct,boundary_percentage_pct,scouting_score



TOP 25 BOWLER CANDIDATES


,bowler_name,balls,runs_conceded,wickets,matches,overs,economy,average,strike_rate,wickets_pct,economy_pct,average_pct,strike_rate_pct,scouting_score


In [30]:
# PRESSURE-X: SMAT season and date audit

smat_raw = data["smat"].copy()
smat_raw.columns = smat_raw.columns.str.strip().str.lower()

print("SMAT DATASET SHAPE:", smat_raw.shape)
print("\nCOLUMNS:")
print(smat_raw.columns.tolist())

# Inspect original season and date fields
if "season" in smat_raw.columns:
    print("\nORIGINAL SEASON DISTRIBUTION:")
    display(smat_raw["season"].value_counts(dropna=False).sort_index())

if "start_date" in smat_raw.columns:
    smat_raw["parsed_date"] = pd.to_datetime(
        smat_raw["start_date"], errors="coerce"
    )
    smat_raw["year"] = smat_raw["parsed_date"].dt.year

    print("\nYEAR DISTRIBUTION:")
    display(smat_raw["year"].value_counts(dropna=False).sort_index())

    print("\nDATE EXAMPLES:")
    display(smat_raw[["start_date", "parsed_date"]].head(10))

# Inspect master SMAT records
print("\nMASTER SMAT COMPETITION LABELS:")
display(
    master["comp_name"].value_counts(dropna=False).head(15)
)

print("\nMASTER SMAT YEAR DISTRIBUTION:")
smat_master = master[
    master["comp_name"].astype(str).str.contains(
        "Syed Mushtaq Ali", case=False, na=False
    )
].copy()

if "start_date" in smat_master.columns:
    smat_master["parsed_date"] = pd.to_datetime(
        smat_master["start_date"], errors="coerce"
    )
    display(
        smat_master["parsed_date"].dt.year.value_counts(
            dropna=False
        ).sort_index()
    )

print("\nSMAT UNIQUE MATCHES:", smat_raw["match_id"].nunique())
print("MASTER SMAT MATCHES:", smat_master["match_id"].nunique())

SMAT DATASET SHAPE: (158839, 28)

COLUMNS:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3', 'competition']

ORIGINAL SEASON DISTRIBUTION:


season
2015/16    19328
2017/18    18593
2018/19    29101
2019/20    28937
2020/21    21217
2021/22    20804
2022/23    10184
2023/24     8768
2024/25     1907
Name: count, dtype: int64


YEAR DISTRIBUTION:


year
2016    19328
2018    18593
2019    58038
2021    42021
2022    10184
2023     8768
2024     1907
Name: count, dtype: int64


DATE EXAMPLES:


,start_date,parsed_date
0,2018-01-07,2018-01-07
1,2018-01-07,2018-01-07
2,2018-01-07,2018-01-07
3,2018-01-07,2018-01-07
4,2018-01-07,2018-01-07
5,2018-01-07,2018-01-07
6,2018-01-07,2018-01-07
7,2018-01-07,2018-01-07
8,2018-01-07,2018-01-07
9,2018-01-07,2018-01-07



MASTER SMAT COMPETITION LABELS:


comp_name
Syed Mushtaq Ali Trophy                      158839
Big Bash League                              153250
SA20                                          29020
IPL 2022                                      17909
IPL 2023                                      17863
IPL 2026                                      17527
IPL 2025                                      17285
IPL 2024                                      17103
Asia Cup 2025                                  4437
Mens T20 Emerging Teams Asia Cup 2024          3519
Asia Cup Rising Stars 2025                     3466
Mens Twenty20 Asia Cup 2022                    3071
United Arab Emirates T20I Tri-Series 2025      1721
New Zealand T20I Tri Series 2022/23            1682
England in Pakistan T20I Series 2022           1678
Name: count, dtype: int64


MASTER SMAT YEAR DISTRIBUTION:


parsed_date
2016    19328
2018    18593
2019    58038
2021    42021
2022    10184
2023     8768
2024     1907
Name: count, dtype: int64


SMAT UNIQUE MATCHES: 695
MASTER SMAT MATCHES: 695


In [31]:
# PRESSURE-X: SMAT Candidate Identification (2021-2024)

import pandas as pd
import numpy as np

# Filter SMAT
smat_df = df[
    df["competition"].astype(str).str.contains(
        "Syed Mushtaq Ali Trophy", case=False, na=False
    )
].copy()

smat_df["season_year"] = pd.to_numeric(
    smat_df["season"], errors="coerce"
)

# Available recent seasons
recent = smat_df[
    smat_df["season_year"].between(2021, 2024)
].copy()

print("SMAT deliveries:", len(smat_df))
print("Scouting deliveries:", len(recent))
print("Seasons:", sorted(recent["season_year"].dropna().unique()))

# ---------------- BATTING ----------------

bat = recent[recent["batter"].notna()].copy()

batting_candidates = bat.groupby("batter").agg(
    runs=("runs", "sum"),
    balls=("batter_ball", "sum"),
    matches=("match_id", "nunique"),
    innings=("innings_key", "nunique"),
    fours=("runs", lambda x: (x == 4).sum()),
    sixes=("runs", lambda x: (x == 6).sum())
).reset_index()

dismissals = (
    recent[recent["is_wicket"]]
    .groupby("dismissed_player")
    .size()
    .rename("dismissals")
)

batting_candidates = batting_candidates.merge(
    dismissals, left_on="batter", right_index=True, how="left"
)

batting_candidates["dismissals"] = (
    batting_candidates["dismissals"].fillna(0)
)

batting_candidates["average"] = np.where(
    batting_candidates["dismissals"] > 0,
    batting_candidates["runs"] / batting_candidates["dismissals"],
    np.nan
)

batting_candidates["strike_rate"] = np.where(
    batting_candidates["balls"] > 0,
    batting_candidates["runs"] / batting_candidates["balls"] * 100,
    0
)

batting_candidates["boundary_percentage"] = np.where(
    batting_candidates["runs"] > 0,
    (4 * batting_candidates["fours"] +
     6 * batting_candidates["sixes"]) /
    batting_candidates["runs"] * 100,
    0
)

batting_candidates = batting_candidates[
    (batting_candidates["matches"] >= 5) &
    (batting_candidates["innings"] >= 5) &
    (batting_candidates["balls"] >= 100)
].copy()

# Batting score
for col in ["runs", "average", "strike_rate", "boundary_percentage"]:
    batting_candidates[col + "_pct"] = (
        batting_candidates[col].rank(pct=True) * 100
    )

batting_candidates["scouting_score"] = (
    0.20 * batting_candidates["runs_pct"] +
    0.30 * batting_candidates["average_pct"] +
    0.35 * batting_candidates["strike_rate_pct"] +
    0.15 * batting_candidates["boundary_percentage_pct"]
)

batting_candidates = batting_candidates.sort_values(
    "scouting_score", ascending=False
)

# ---------------- BOWLING ----------------

bowl = recent[recent["bowler_name"].notna()].copy()

bowling_candidates = bowl.groupby("bowler_name").agg(
    balls=("is_legal_ball", "sum"),
    runs_conceded=("bowler_runs_conceded", "sum"),
    wickets=("bowler_wicket", "sum"),
    matches=("match_id", "nunique")
).reset_index()

bowling_candidates["overs"] = bowling_candidates["balls"] / 6

bowling_candidates["economy"] = np.where(
    bowling_candidates["overs"] > 0,
    bowling_candidates["runs_conceded"] /
    bowling_candidates["overs"], np.nan
)

bowling_candidates["average"] = np.where(
    bowling_candidates["wickets"] > 0,
    bowling_candidates["runs_conceded"] /
    bowling_candidates["wickets"], np.nan
)

bowling_candidates["strike_rate"] = np.where(
    bowling_candidates["wickets"] > 0,
    bowling_candidates["balls"] /
    bowling_candidates["wickets"], np.nan
)

bowling_candidates = bowling_candidates[
    (bowling_candidates["matches"] >= 5) &
    (bowling_candidates["overs"] >= 10) &
    (bowling_candidates["wickets"] >= 5)
].copy()

# Bowling score
bowling_candidates["wickets_pct"] = (
    bowling_candidates["wickets"].rank(pct=True) * 100
)

bowling_candidates["economy_pct"] = (
    bowling_candidates["economy"].rank(
        pct=True, ascending=False
    ) * 100
)

bowling_candidates["average_pct"] = (
    bowling_candidates["average"].rank(
        pct=True, ascending=False
    ) * 100
)

bowling_candidates["strike_rate_pct"] = (
    bowling_candidates["strike_rate"].rank(
        pct=True, ascending=False
    ) * 100
)

bowling_candidates["scouting_score"] = (
    0.35 * bowling_candidates["wickets_pct"] +
    0.25 * bowling_candidates["economy_pct"] +
    0.20 * bowling_candidates["average_pct"] +
    0.20 * bowling_candidates["strike_rate_pct"]
)

bowling_candidates = bowling_candidates.sort_values(
    "scouting_score", ascending=False
)

# Export
batting_candidates.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Batter_Candidates.csv", index=False
)

bowling_candidates.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Bowler_Candidates.csv", index=False
)

print("\nSCOUTING POOLS CREATED")
print("Batters:", len(batting_candidates))
print("Bowlers:", len(bowling_candidates))

print("\nTOP 25 BATTERS")
display(batting_candidates.head(25))

print("\nTOP 25 BOWLERS")
display(bowling_candidates.head(25))

SMAT deliveries: 158839
Scouting deliveries: 62880
Seasons: [np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0)]

SCOUTING POOLS CREATED
Batters: 213
Bowlers: 183

TOP 25 BATTERS


,batter,runs,balls,matches,innings,fours,sixes,dismissals,average,strike_rate,boundary_percentage,runs_pct,average_pct,strike_rate_pct,boundary_percentage_pct,scouting_score
570,RK Singh,452.00,265,12,12,33,25,6.00,75.33,170.57,62.39,91.08,99.77,97.18,76.06,93.57
16,AA Barot,283.00,153,5,5,32,12,5.00,56.60,184.97,70.67,70.89,95.31,98.59,94.84,91.50
571,RM Patidar,668.00,417,18,18,56,36,16.00,41.75,160.19,65.87,98.12,83.10,95.77,86.85,91.10
295,JM Sharma,394.00,212,17,17,30,28,10.00,39.40,185.85,73.10,86.85,79.34,99.06,97.65,90.49
554,R Parag,595.00,397,16,16,36,38,11.00,54.09,149.87,62.52,96.71,93.43,89.67,76.53,90.23
549,Punit Bisht,235.00,120,7,7,18,18,4.00,58.75,195.83,76.60,57.75,95.77,99.53,99.53,90.05
116,Anmolpreet Singh,625.00,395,17,17,62,26,15.00,41.67,158.23,64.64,97.18,82.63,95.31,82.16,89.91
460,N Rana,679.00,472,18,18,69,30,15.00,45.27,143.86,67.16,98.59,88.26,84.98,90.14,89.46
587,Rahul Singh,374.00,251,10,10,36,15,6.00,62.33,149.00,62.57,84.04,97.18,89.20,77.00,88.73
6,A Manohar,282.00,185,8,8,19,18,4.00,70.50,152.43,65.25,70.42,98.59,92.02,85.21,88.65



TOP 25 BOWLERS


,bowler_name,balls,runs_conceded,wickets,matches,overs,economy,average,strike_rate,wickets_pct,economy_pct,average_pct,strike_rate_pct,scouting_score
93,Ashutosh Aman,264,232.00,21,12,44.00,5.27,11.05,12.57,90.71,98.91,98.36,93.44,94.84
540,Shivank Vashisth,174,163.00,16,8,29.00,5.62,10.19,10.88,82.24,94.54,100.00,99.45,92.31
128,C Sakariya,312,327.00,25,13,52.00,6.29,13.08,12.48,95.36,81.97,95.08,93.99,91.68
107,B Kumar,177,171.00,16,8,29.50,5.80,10.69,11.06,82.24,91.26,98.91,98.91,91.16
453,Ravi Teja,311,335.00,25,14,51.83,6.46,13.40,12.44,95.36,75.96,94.54,94.54,90.18
135,Cheepurapalli Stephen,244,260.00,21,11,40.67,6.39,12.38,11.62,90.71,78.14,97.27,96.72,90.08
451,Ravi Bishnoi,354,337.00,23,15,59.00,5.71,14.65,15.39,93.17,92.35,90.16,72.13,88.16
346,NA Rathva,324,327.00,22,20,54.00,6.06,14.86,14.73,92.08,86.89,88.52,79.78,87.61
262,Khrievitso Kense,185,168.00,13,9,30.83,5.45,12.92,14.23,72.40,97.81,96.17,82.51,85.53
62,Amit Ali,125,114.00,11,7,20.83,5.47,10.36,11.36,58.47,96.17,99.45,97.81,83.96


In [32]:
# STAGE 2: PLAYER MASTER - SOURCE SCHEMA AUDIT

from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"C:\Users\pavan")

# Load candidate pools
batters = pd.read_csv(DATA_DIR / "PRESSURE_X_SMAT_Batter_Candidates.csv")
bowlers = pd.read_csv(DATA_DIR / "PRESSURE_X_SMAT_Bowler_Candidates.csv")

# Metadata datasets already loaded in data dictionary
metadata_tables = {
    "playing_eleven": data["playing_eleven"],
    "competition_mapping": data["competition_mapping"],
    "teams": data["teams"],
    "competitions": data["competitions"]
}

print("=" * 75)
print("CANDIDATE POOLS")
print("=" * 75)

print("Batters:", batters.shape)
print("Bowlers:", bowlers.shape)

print("\n" + "=" * 75)
print("METADATA SCHEMA AUDIT")
print("=" * 75)

for name, table in metadata_tables.items():
    print(f"\n{name.upper()}")
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())
    print("\nSample records:")
    display(table.head(3))

print("\n" + "=" * 75)
print("CANDIDATE NAME CHECK")
print("=" * 75)

print("Batting candidate columns:", batters.columns.tolist())
print("Bowling candidate columns:", bowlers.columns.tolist())

print("\nSample batter names:")
display(batters[["batter"]].head(10))

print("\nSample bowler names:")
display(bowlers[["bowler_name"]].head(10))

CANDIDATE POOLS
Batters: (213, 16)
Bowlers: (183, 14)

METADATA SCHEMA AUDIT

PLAYING_ELEVEN
Shape: (28381, 9)
Columns: ['match_id', 'team_id', 'player_id', 'batting_order', 'is_played', 'is_captain', 'is_wk', 'is_impact', 'is_impact_off']

Sample records:


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
0,13511,22,1187,1.00,True,False,False,0,0
1,13511,22,6273,2.00,True,False,False,0,0
2,13511,22,1210,3.00,True,False,False,0,0



COMPETITION_MAPPING
Shape: (6413, 6)
Columns: ['comp_id', 'comp_name', 'team_id', 'team_name', 'player_id', 'player_name']

Sample records:


,comp_id,comp_name,team_id,team_name,player_id,player_name
0,1331,England in West Indies T20I Series 2022,22,England,478,Eoin Morgan
1,1331,England in West Indies T20I Series 2022,22,England,479,Adil Rashid
2,1331,England in West Indies T20I Series 2022,22,England,722,Liam Dawson



TEAMS
Shape: (353, 3)
Columns: ['comp_id', 'team_id', 'team_name']

Sample records:


,comp_id,team_id,team_name
0,1331,22,England
1,1331,28,West Indies
2,1335,18,India


... (2 outputs left out)


CANDIDATE NAME CHECK
Batting candidate columns: ['batter', 'runs', 'balls', 'matches', 'innings', 'fours', 'sixes', 'dismissals', 'average', 'strike_rate', 'boundary_percentage', 'runs_pct', 'average_pct', 'strike_rate_pct', 'boundary_percentage_pct', 'scouting_score']
Bowling candidate columns: ['bowler_name', 'balls', 'runs_conceded', 'wickets', 'matches', 'overs', 'economy', 'average', 'strike_rate', 'wickets_pct', 'economy_pct', 'average_pct', 'strike_rate_pct', 'scouting_score']

Sample batter names:


,batter
0,RK Singh
1,AA Barot
2,RM Patidar
3,JM Sharma
4,R Parag
5,Punit Bisht
6,Anmolpreet Singh
7,N Rana
8,Rahul Singh
9,A Manohar



Sample bowler names:


,bowler_name
0,Ashutosh Aman
1,Shivank Vashisth
2,C Sakariya
3,B Kumar
4,Ravi Teja
5,Cheepurapalli Stephen
6,Ravi Bishnoi
7,NA Rathva
8,Khrievitso Kense
9,Amit Ali


In [33]:
# STAGE 2A: VERIFY SMAT PLAYER ID MAPPING

# 1. Inspect master dataset identifiers
print("=" * 70)
print("MASTER DATASET PLAYER COLUMNS")
print("=" * 70)

print([
    col for col in master.columns
    if any(x in col.lower() for x in ["player", "striker", "batter", "bowler", "team"])
])

# 2. Extract SMAT competition mapping
smat_mapping = data["competition_mapping"][
    data["competition_mapping"]["comp_name"]
    .str.contains("Syed Mushtaq Ali Trophy", case=False, na=False)
].copy()

print("\nSMAT MAPPING")
print("Shape:", smat_mapping.shape)
print("Unique players:", smat_mapping["player_id"].nunique())
print("Unique teams:", smat_mapping["team_id"].nunique())
print("Unique competitions:", smat_mapping["comp_id"].nunique())

display(smat_mapping.head(15))

# 3. Inspect SMAT team names
print("\nSMAT TEAMS")
display(
    smat_mapping[["team_id", "team_name"]]
    .drop_duplicates()
    .sort_values("team_name")
)

# 4. Compare candidate names with mapping names
candidate_names = set(batters["batter"].dropna()) | set(
    bowlers["bowler_name"].dropna()
)

mapping_names = set(smat_mapping["player_name"].dropna())

print("\nNAME MATCHING AUDIT")
print("Candidate names:", len(candidate_names))
print("Mapping full names:", len(mapping_names))
print("Exact name matches:", len(candidate_names & mapping_names))

print("\nCandidate examples:")
print(list(sorted(candidate_names))[:20])

print("\nMapping examples:")
print(smat_mapping["player_name"].drop_duplicates().head(20).tolist())


MASTER DATASET PLAYER COLUMNS
['batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'dismissed_player_id', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'player_dismissed', 'other_player_dismissed']

SMAT MAPPING
Shape: (0, 6)
Unique players: 0
Unique teams: 0
Unique competitions: 0


,comp_id,comp_name,team_id,team_name,player_id,player_name



SMAT TEAMS


,team_id,team_name



NAME MATCHING AUDIT
Candidate names: 364
Mapping full names: 0
Exact name matches: 0

Candidate examples:
['A Badoni', 'A Choudhary', 'A Manohar', 'A Nagwaswalla', 'A Palkar', 'A Sheth', 'AA Barot', 'AA Herwadkar', 'AA Sarwate', 'AKS Bains', 'AL Menaria', 'AM Rahane', 'AP Tare', 'AS Rajpoot', 'AT Rayudu', 'AV Rajput', 'AV Wankhade', 'Abdul Samad', 'Abhijeet Saket', 'Abhimanyu Rana']

Mapping examples:
[]


In [34]:
# STAGE 2B: BUILD PLAYER IDENTITY AND TEAM MAPPING FROM MASTER

# Filter SMAT records
smat_df = master[
    master["comp_name"].str.contains(
        "Syed Mushtaq Ali Trophy", case=False, na=False
    )
].copy()

print("SMAT deliveries:", len(smat_df))

# Build player-team records from batting and bowling data
batter_records = smat_df[[
    "striker_id", "striker_name",
    "batting_team_id", "batting_team"
]].rename(columns={
    "striker_id": "player_id",
    "striker_name": "player_name",
    "batting_team_id": "team_id",
    "batting_team": "team_name"
})

non_striker_records = smat_df[[
    "non_striker_id", "non_striker_name",
    "batting_team_id", "batting_team"
]].rename(columns={
    "non_striker_id": "player_id",
    "non_striker_name": "player_name",
    "batting_team_id": "team_id",
    "batting_team": "team_name"
})

bowler_records = smat_df[[
    "bowler_id", "bowler_name",
    "bowling_team_id", "bowling_team"
]].rename(columns={
    "bowler_id": "player_id",
    "bowler_name": "player_name",
    "bowling_team_id": "team_id",
    "bowling_team": "team_name"
})

# Combine all player records
player_team_mapping = pd.concat(
    [batter_records, non_striker_records, bowler_records],
    ignore_index=True
)

player_team_mapping = (
    player_team_mapping
    .dropna(subset=["player_id", "player_name"])
    .drop_duplicates()
)

# Count matches played for each player-team combination
player_team_matches = (
    pd.concat([
        smat_df[["match_id", "striker_id", "striker_name",
                 "batting_team_id", "batting_team"]].rename(
            columns={"striker_id": "player_id",
                     "striker_name": "player_name",
                     "batting_team_id": "team_id",
                     "batting_team": "team_name"}
        ),
        smat_df[["match_id", "non_striker_id", "non_striker_name",
                 "batting_team_id", "batting_team"]].rename(
            columns={"non_striker_id": "player_id",
                     "non_striker_name": "player_name",
                     "batting_team_id": "team_id",
                     "batting_team": "team_name"}
        ),
        smat_df[["match_id", "bowler_id", "bowler_name",
                 "bowling_team_id", "bowling_team"]].rename(
            columns={"bowler_id": "player_id",
                     "bowler_name": "player_name",
                     "bowling_team_id": "team_id",
                     "bowling_team": "team_name"}
        )
    ], ignore_index=True)
    .dropna(subset=["player_id", "player_name"])
    .drop_duplicates(["match_id", "player_id", "team_id"])
    .groupby(["player_id", "player_name", "team_id", "team_name"])
    ["match_id"].nunique()
    .reset_index(name="smat_matches")
)

# Check identity coverage
print("\nPLAYER MAPPING")
print("Unique player IDs:", player_team_mapping["player_id"].nunique())
print("Unique players:", player_team_mapping["player_name"].nunique())
print("Unique teams:", player_team_mapping["team_id"].nunique())

print("\nTOP TEAM ASSOCIATIONS")
display(
    player_team_matches.sort_values(
        "smat_matches", ascending=False
    ).head(20)
)

# Save mapping
player_team_mapping.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Player_Team_Mapping.csv",
    index=False
)

player_team_matches.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Player_Team_Matches.csv",
    index=False
)

SMAT deliveries: 158839

PLAYER MAPPING
Unique player IDs: 0
Unique players: 0
Unique teams: 0

TOP TEAM ASSOCIATIONS


,player_id,player_name,team_id,team_name,smat_matches


In [35]:
# STAGE 2B: SMAT IDENTIFIER DIAGNOSTICS

smat_df = master[
    master["comp_name"].str.contains(
        "Syed Mushtaq Ali Trophy", case=False, na=False
    )
].copy()

cols = [
    "striker_id", "striker_name",
    "non_striker_id", "non_striker_name",
    "bowler_id", "bowler_name",
    "batting_team_id", "batting_team",
    "bowling_team_id", "bowling_team"
]

print("SMAT rows:", len(smat_df))

print("\n1. NULL VALUE COUNTS")
display(smat_df[cols].isna().sum().to_frame("null_count"))

print("\n2. NON-NULL VALUE COUNTS")
display(smat_df[cols].notna().sum().to_frame("available_count"))

print("\n3. SAMPLE SMAT RECORDS")
display(smat_df[cols + ["match_id", "comp_name"]].head(10))

print("\n4. ORIGINAL SMAT DATASET COLUMNS")
print(data["smat"].columns.tolist())

print("\n5. ORIGINAL SMAT SAMPLE")
display(data["smat"].head(5))

SMAT rows: 158839

1. NULL VALUE COUNTS


,null_count
striker_id,158839
striker_name,158839
non_striker_id,158839
non_striker_name,158839
bowler_id,158839
bowler_name,158839
batting_team_id,158839
batting_team,0
bowling_team_id,158839
bowling_team,0



2. NON-NULL VALUE COUNTS


,available_count
striker_id,0
striker_name,0
non_striker_id,0
non_striker_name,0
bowler_id,0
bowler_name,0
batting_team_id,0
batting_team,158839
bowling_team_id,0
bowling_team,158839



3. SAMPLE SMAT RECORDS


,striker_id,striker_name,non_striker_id,non_striker_name,bowler_id,bowler_name,batting_team_id,batting_team,bowling_team_id,bowling_team,match_id,comp_name
200999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy
201000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy
201001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy
201002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy
201003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy
201004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy
201005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy
201006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy
201007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy
201008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Baroda,NaN,Mumbai,SMAT_1130661,Syed Mushtaq Ali Trophy



4. ORIGINAL SMAT DATASET COLUMNS
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3', 'competition']

5. ORIGINAL SMAT SAMPLE


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3,competition
0,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.10,0.10,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy
1,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.20,0.20,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy
2,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.30,0.30,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy
3,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.40,0.40,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy
4,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.50,0.50,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy


In [36]:
# STAGE 2C: BUILD PLAYER-TEAM MAPPING FROM ORIGINAL SMAT DATA

smat_raw = data["smat"].copy()

# Standardize player-team records
batter_records = smat_raw[
    ["match_id", "season", "striker", "batting_team"]
].rename(columns={
    "striker": "player_name",
    "batting_team": "team_name"
})

non_striker_records = smat_raw[
    ["match_id", "season", "non_striker", "batting_team"]
].rename(columns={
    "non_striker": "player_name",
    "batting_team": "team_name"
})

bowler_records = smat_raw[
    ["match_id", "season", "bowler", "bowling_team"]
].rename(columns={
    "bowler": "player_name",
    "bowling_team": "team_name"
})

# Combine records
player_team_records = pd.concat(
    [batter_records, non_striker_records, bowler_records],
    ignore_index=True
)

player_team_records = player_team_records.dropna(
    subset=["player_name", "team_name"]
)

# Normalize names for matching
player_team_records["player_key"] = (
    player_team_records["player_name"]
    .astype(str).str.strip().str.lower()
)

# Count appearances by player, team and season
player_team_matches = (
    player_team_records
    .drop_duplicates(["match_id", "player_key", "team_name"])
    .groupby(["player_key", "player_name", "team_name", "season"])
    .agg(smat_matches=("match_id", "nunique"))
    .reset_index()
)

# Prepare candidate names
batters["player_key"] = (
    batters["batter"].astype(str).str.strip().str.lower()
)

bowlers["player_key"] = (
    bowlers["bowler_name"].astype(str).str.strip().str.lower()
)

# Match batter candidates
batter_profile = batters.merge(
    player_team_matches,
    on="player_key",
    how="left"
)

# Match bowler candidates
bowler_profile = bowlers.merge(
    player_team_matches,
    on="player_key",
    how="left"
)

# Coverage report
print("=" * 65)
print("PLAYER-TEAM MAPPING RESULTS")
print("=" * 65)

print("Unique SMAT players:", player_team_records["player_key"].nunique())
print("Unique domestic teams:", player_team_records["team_name"].nunique())

print("\nBATTER MATCHING")
print("Candidates:", len(batters))
print("Matched:", batter_profile["team_name"].notna().sum())
print("Unmatched:", batter_profile["team_name"].isna().sum())

print("\nBOWLER MATCHING")
print("Candidates:", len(bowlers))
print("Matched:", bowler_profile["team_name"].notna().sum())
print("Unmatched:", bowler_profile["team_name"].isna().sum())

print("\nBATTER PROFILE SAMPLE")
display(
    batter_profile[
        ["batter", "team_name", "season", "smat_matches", "scouting_score"]
    ].head(15)
)

print("\nBOWLER PROFILE SAMPLE")
display(
    bowler_profile[
        ["bowler_name", "team_name", "season", "smat_matches", "scouting_score"]
    ].head(15)
)

# Save outputs
batter_profile.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Batter_Profile.csv", index=False
)

bowler_profile.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Bowler_Profile.csv", index=False
)

player_team_matches.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Player_Team_Season.csv", index=False
)

PLAYER-TEAM MAPPING RESULTS
Unique SMAT players: 1463
Unique domestic teams: 39

BATTER MATCHING
Candidates: 213
Matched: 1125
Unmatched: 0

BOWLER MATCHING
Candidates: 183
Matched: 927
Unmatched: 0

BATTER PROFILE SAMPLE


,batter,team_name,season,smat_matches,scouting_score
0,RK Singh,Uttar Pradesh,2017/18,8,93.57
1,RK Singh,Uttar Pradesh,2018/19,10,93.57
2,RK Singh,Uttar Pradesh,2019/20,5,93.57
3,RK Singh,Uttar Pradesh,2020/21,2,93.57
4,RK Singh,Uttar Pradesh,2021/22,3,93.57
5,RK Singh,Uttar Pradesh,2022/23,1,93.57
6,RK Singh,Uttar Pradesh,2023/24,5,93.57
7,RK Singh,Uttar Pradesh,2024/25,1,93.57
8,AA Barot,Saurashtra,2015/16,6,91.50
9,AA Barot,Saurashtra,2020/21,5,91.50



BOWLER PROFILE SAMPLE


,bowler_name,team_name,season,smat_matches,scouting_score
0,Ashutosh Aman,Bihar,2018/19,5,94.84
1,Ashutosh Aman,Bihar,2019/20,6,94.84
2,Ashutosh Aman,Bihar,2020/21,5,94.84
3,Ashutosh Aman,Bihar,2021/22,5,94.84
4,Ashutosh Aman,Bihar,2022/23,3,94.84
5,Shivank Vashisth,Delhi,2020/21,1,92.31
6,Shivank Vashisth,Delhi,2021/22,3,92.31
7,Shivank Vashisth,Delhi,2022/23,4,92.31
8,C Sakariya,Saurashtra,2018/19,4,91.68
9,C Sakariya,Saurashtra,2019/20,6,91.68


In [37]:
# STAGE 3: CONSOLIDATED PLAYER PROFILES

def create_player_profile(profile_df, name_col):
    x = profile_df.copy()

    # Extract ending year for chronological sorting
    x["season_end_year"] = (
        x["season"].astype(str)
        .str.extract(r"(\d{4})$")[0]
        .astype(float)
    )

    # Sort chronologically
    x = x.sort_values(
        ["player_key", "season_end_year"],
        ascending=[True, False]
    )

    # Latest team and season
    latest = (
        x.groupby("player_key")
        .first()
        [["player_name", "team_name", "season"]]
        .rename(columns={
            "player_name": "player_name",
            "team_name": "latest_team",
            "season": "latest_smat_season"
        })
    )

    # Career exposure
    exposure = (
        x.groupby("player_key")
        .agg(
            smat_seasons=("season", "nunique"),
            total_smat_matches=("smat_matches", "sum")
        )
    )

    # Recent appearances
    recent = (
        x[x["season"].isin(["2022/23", "2023/24", "2024/25"])]
        .groupby("player_key")
        .agg(
            recent_seasons=("season", "nunique"),
            recent_smat_matches=("smat_matches", "sum")
        )
    )

    # Combine
    result = latest.join(exposure).join(recent)

    result = result.reset_index()

    # Add original scouting statistics
    candidate_stats = profile_df[
        ["player_key", name_col, "scouting_score"]
    ].drop_duplicates("player_key")

    result = result.merge(
        candidate_stats,
        on="player_key",
        how="left"
    )

    return result


# Build consolidated profiles
batters_master = create_player_profile(
    batter_profile, "batter"
)

bowlers_master = create_player_profile(
    bowler_profile, "bowler_name"
)

# Missing recent appearances become zero
for table in [batters_master, bowlers_master]:
    table[["recent_seasons", "recent_smat_matches"]] = (
        table[["recent_seasons", "recent_smat_matches"]]
        .fillna(0)
    )

# Check duplicate names across different teams
print("=" * 65)
print("CONSOLIDATED PLAYER MASTER")
print("=" * 65)

print("Unique batters:", len(batters_master))
print("Unique bowlers:", len(bowlers_master))

print("\nBATTER PROFILE")
display(
    batters_master.sort_values(
        "scouting_score", ascending=False
    ).head(20)
)

print("\nBOWLER PROFILE")
display(
    bowlers_master.sort_values(
        "scouting_score", ascending=False
    ).head(20)
)

# Save
batters_master.to_csv(
    DATA_DIR / "PRESSURE_X_Batter_Master.csv", index=False
)

bowlers_master.to_csv(
    DATA_DIR / "PRESSURE_X_Bowler_Master.csv", index=False
)

print("\nSaved both consolidated player master files.")

CONSOLIDATED PLAYER MASTER
Unique batters: 213
Unique bowlers: 183

BATTER PROFILE


,player_key,player_name,latest_team,latest_smat_season,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,batter,scouting_score
146,rk singh,RK Singh,Uttar Pradesh,2017/18,8,35,3.00,7.00,RK Singh,93.57
3,aa barot,AA Barot,Saurashtra,2015/16,2,11,0.00,0.00,AA Barot,91.50
147,rm patidar,RM Patidar,Madhya Pradesh,2017/18,8,34,3.00,9.00,RM Patidar,91.10
72,jm sharma,JM Sharma,Vidarbha,2015/16,8,41,2.00,6.00,JM Sharma,90.49
130,r parag,R Parag,Assam,2018/19,6,28,2.00,6.00,R Parag,90.23
127,punit bisht,Punit Bisht,Meghalaya,2018/19,4,20,0.00,0.00,Punit Bisht,90.05
22,anmolpreet singh,Anmolpreet Singh,Punjab,2017/18,7,33,2.00,12.00,Anmolpreet Singh,89.91
109,n rana,N Rana,Delhi,2015/16,9,48,3.00,10.00,N Rana,89.46
135,rahul singh,Rahul Singh,Assam,2020/21,6,29,1.00,2.00,Rahul Singh,88.73
1,a manohar,A Manohar,Karnataka,2021/22,3,8,2.00,4.00,A Manohar,88.65



BOWLER PROFILE


,player_key,player_name,latest_team,latest_smat_season,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,bowler_name,scouting_score
27,ashutosh aman,Ashutosh Aman,Bihar,2018/19,5,24,1.00,3.00,Ashutosh Aman,94.84
153,shivank vashisth,Shivank Vashisth,Delhi,2020/21,3,8,1.00,4.00,Shivank Vashisth,92.31
43,c sakariya,C Sakariya,Saurashtra,2018/19,5,23,1.00,2.00,C Sakariya,91.68
35,b kumar,B Kumar,Uttar Pradesh,2019/20,4,10,2.00,6.00,B Kumar,91.16
131,ravi teja,Ravi Teja,Hyderabad (India),2017/18,7,20,2.00,5.00,Ravi Teja,90.18
44,cheepurapalli stephen,Cheepurapalli Stephen,Andhra,2015/16,5,22,1.00,2.00,Cheepurapalli Stephen,90.08
129,ravi bishnoi,Ravi Bishnoi,Gujarat,2023/24,6,19,2.00,6.00,Ravi Bishnoi,88.16
111,na rathva,NA Rathva,Baroda,2017/18,6,30,2.00,8.00,NA Rathva,87.61
79,khrievitso kense,Khrievitso Kense,Nagaland,2020/21,3,9,1.00,2.00,Khrievitso Kense,85.53
17,amit ali,Amit Ali,Tripura,2021/22,2,7,1.00,2.00,Amit Ali,83.96



Saved both consolidated player master files.


In [38]:
def create_player_profile(profile_df, name_col):

    x = profile_df.copy()

    # Correctly extract the starting year from season labels
    x["season_start_year"] = (
        x["season"].astype(str)
        .str.extract(r"(\d{4})")[0]
        .astype(float)
    )

    # Latest season for each player
    latest_year = x.groupby("player_key")["season_start_year"].max()

    latest_rows = x.merge(
        latest_year.rename("latest_year"),
        left_on="player_key",
        right_index=True
    )

    latest_rows = latest_rows[
        latest_rows["season_start_year"] == latest_rows["latest_year"]
    ]

    # Handle multiple teams in the latest season
    latest = (
        latest_rows.groupby("player_key")
        .agg(
            latest_smat_season=("season", "first"),
            latest_team=("team_name",
                         lambda s: ", ".join(sorted(s.dropna().unique()))),
            latest_team_count=("team_name", "nunique")
        )
    )

    latest["team_status"] = np.where(
        latest["latest_team_count"] > 1,
        "Multiple teams",
        "Single team"
    )

    # Overall SMAT exposure
    exposure = (
        x.groupby("player_key")
        .agg(
            smat_seasons=("season", "nunique"),
            total_smat_matches=("smat_matches", "sum")
        )
    )

    # Recent exposure
    recent = (
        x[x["season_start_year"].between(2022, 2024)]
        .groupby("player_key")
        .agg(
            recent_seasons=("season", "nunique"),
            recent_smat_matches=("smat_matches", "sum")
        )
    )

    # Candidate scoring information
    candidate_stats = (
        profile_df[
            ["player_key", name_col, "scouting_score"]
        ]
        .drop_duplicates("player_key")
        .set_index("player_key")
    )

    result = (
        latest.join(exposure)
        .join(recent)
        .join(candidate_stats)
        .reset_index()
    )

    result[["recent_seasons", "recent_smat_matches"]] = (
        result[["recent_seasons", "recent_smat_matches"]]
        .fillna(0)
    )

    return result.sort_values(
        "scouting_score", ascending=False
    ).reset_index(drop=True)


batters_master = create_player_profile(
    batter_profile, "batter"
)

bowlers_master = create_player_profile(
    bowler_profile, "bowler_name"
)

batters_master.to_csv(
    DATA_DIR / "PRESSURE_X_Batter_Master.csv", index=False
)

bowlers_master.to_csv(
    DATA_DIR / "PRESSURE_X_Bowler_Master.csv", index=False
)

print("BATTER MASTER")
display(batters_master.head(15))

print("\nBOWLER MASTER")
display(bowlers_master.head(15))

print("\nValidation:")
print("Batters:", batters_master["player_key"].nunique())
print("Bowlers:", bowlers_master["player_key"].nunique())
print("Batters with multiple latest-season teams:",
      (batters_master["team_status"] == "Multiple teams").sum())
print("Bowlers with multiple latest-season teams:",
      (bowlers_master["team_status"] == "Multiple teams").sum())

BATTER MASTER


,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,batter,scouting_score
0,rk singh,2024/25,Uttar Pradesh,1,Single team,8,35,3.00,7.00,RK Singh,93.57
1,aa barot,2020/21,Saurashtra,1,Single team,2,11,0.00,0.00,AA Barot,91.50
2,rm patidar,2024/25,Madhya Pradesh,1,Single team,8,34,3.00,9.00,RM Patidar,91.10
3,jm sharma,2024/25,Vidarbha,1,Single team,8,41,2.00,6.00,JM Sharma,90.49
4,r parag,2023/24,Assam,1,Single team,6,28,2.00,6.00,R Parag,90.23
5,punit bisht,2021/22,Meghalaya,1,Single team,4,20,0.00,0.00,Punit Bisht,90.05
6,anmolpreet singh,2023/24,Punjab,1,Single team,7,33,2.00,12.00,Anmolpreet Singh,89.91
7,n rana,2024/25,Uttar Pradesh,1,Single team,9,48,3.00,10.00,N Rana,89.46
8,rahul singh,2023/24,Hyderabad (India),1,Single team,6,29,1.00,2.00,Rahul Singh,88.73
9,a manohar,2023/24,Karnataka,1,Single team,3,8,2.00,4.00,A Manohar,88.65



BOWLER MASTER


,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,bowler_name,scouting_score
0,ashutosh aman,2022/23,Bihar,1,Single team,5,24,1.00,3.00,Ashutosh Aman,94.84
1,shivank vashisth,2022/23,Delhi,1,Single team,3,8,1.00,4.00,Shivank Vashisth,92.31
2,c sakariya,2022/23,Saurashtra,1,Single team,5,23,1.00,2.00,C Sakariya,91.68
3,b kumar,2024/25,Uttar Pradesh,1,Single team,4,10,2.00,6.00,B Kumar,91.16
4,ravi teja,2023/24,Hyderabad (India),1,Single team,7,20,2.00,5.00,Ravi Teja,90.18
5,cheepurapalli stephen,2023/24,Andhra,1,Single team,5,22,1.00,2.00,Cheepurapalli Stephen,90.08
6,ravi bishnoi,2023/24,Gujarat,1,Single team,6,19,2.00,6.00,Ravi Bishnoi,88.16
7,na rathva,2023/24,Baroda,1,Single team,6,30,2.00,8.00,NA Rathva,87.61
8,khrievitso kense,2023/24,Nagaland,1,Single team,3,9,1.00,2.00,Khrievitso Kense,85.53
9,amit ali,2023/24,Tripura,1,Single team,2,7,1.00,2.00,Amit Ali,83.96



Validation:
Batters: 213
Bowlers: 183
Batters with multiple latest-season teams: 0
Bowlers with multiple latest-season teams: 1


In [40]:
display(
    bowlers_master.loc[
        bowlers_master["team_status"] == "Multiple teams",
        ["bowler_name", "latest_team", "latest_smat_season",
         "latest_team_count", "total_smat_matches"]
    ]
)

,bowler_name,latest_team,latest_smat_season,latest_team_count,total_smat_matches
52,Sumit Kumar,"Haryana, Nagaland",2023/24,2,43


In [41]:
# ==========================================
# STAGE 4: IPL PLAYER EXPOSURE AUDIT
# ==========================================

import re
import pandas as pd
import numpy as np

# 1. Extract IPL records from existing master dataframe
ipl = df[
    df["competition"].astype(str).str.contains(
        "IPL|Indian Premier League", case=False, na=False
    )
].copy()

print("IPL records found:", len(ipl))
print("IPL matches:", ipl["match_id"].nunique())

# 2. Normalize player names for exact matching
def normalize_name(name):
    if pd.isna(name):
        return ""
    return re.sub(r"[^a-z]", "", str(name).lower())


# 3. Build IPL batting exposure
ipl_batting = (
    ipl[ipl["batter"].notna()]
    .groupby("batter")
    .agg(
        ipl_matches=("match_id", "nunique"),
        ipl_runs=("runs", "sum")
    )
    .reset_index()
)

ipl_batting["player_key"] = (
    ipl_batting["batter"].apply(normalize_name)
)

# 4. Build IPL bowling exposure
ipl_bowling = (
    ipl[ipl["bowler_name"].notna()]
    .groupby("bowler_name")
    .agg(
        ipl_matches=("match_id", "nunique")
    )
    .reset_index()
)

ipl_bowling["player_key"] = (
    ipl_bowling["bowler_name"].apply(normalize_name)
)

# 5. Consolidate IPL exposure for batter candidates
batters_ipl = batters_master.merge(
    ipl_batting[
        ["player_key", "ipl_matches", "ipl_runs"]
    ],
    on="player_key",
    how="left"
)

batters_ipl["ipl_matches"] = (
    batters_ipl["ipl_matches"].fillna(0).astype(int)
)

batters_ipl["ipl_runs"] = (
    batters_ipl["ipl_runs"].fillna(0).astype(int)
)

batters_ipl["ipl_exposure"] = np.where(
    batters_ipl["ipl_matches"] > 0,
    "Played IPL",
    "No recorded IPL appearance"
)

# 6. Consolidate IPL exposure for bowler candidates
bowlers_ipl = bowlers_master.merge(
    ipl_bowling[
        ["player_key", "ipl_matches"]
    ],
    on="player_key",
    how="left"
)

bowlers_ipl["ipl_matches"] = (
    bowlers_ipl["ipl_matches"].fillna(0).astype(int)
)

bowlers_ipl["ipl_exposure"] = np.where(
    bowlers_ipl["ipl_matches"] > 0,
    "Played IPL",
    "No recorded IPL appearance"
)

# 7. Save results
batters_ipl.to_csv(
    DATA_DIR / "PRESSURE_X_Batter_IPL_Exposure.csv",
    index=False
)

bowlers_ipl.to_csv(
    DATA_DIR / "PRESSURE_X_Bowler_IPL_Exposure.csv",
    index=False
)

# 8. Display audit summary
print("\n========== BATTER IPL EXPOSURE ==========")
print(batters_ipl["ipl_exposure"].value_counts())

print("\n========== BOWLER IPL EXPOSURE ==========")
print(bowlers_ipl["ipl_exposure"].value_counts())

print("\n========== BATTERS WITH IPL APPEARANCES ==========")
display(
    batters_ipl[batters_ipl["ipl_matches"] > 0][
        ["batter", "latest_team", "ipl_matches",
         "ipl_runs", "scouting_score"]
    ].sort_values("ipl_matches", ascending=False).head(20)
)

print("\n========== BOWLERS WITH IPL APPEARANCES ==========")
display(
    bowlers_ipl[bowlers_ipl["ipl_matches"] > 0][
        ["bowler_name", "latest_team", "ipl_matches",
         "scouting_score"]
    ].sort_values("ipl_matches", ascending=False).head(20)
)

# 9. Show unmatched candidates for alias verification
print("\n========== NAME MATCHING AUDIT ==========")

print(
    "Unmatched batters:",
    (batters_ipl["ipl_matches"] == 0).sum()
)

print(
    "Unmatched bowlers:",
    (bowlers_ipl["ipl_matches"] == 0).sum()
)

print("\nSaved both IPL exposure files.")

IPL records found: 87687
IPL matches: 367

========== BATTER IPL EXPOSURE ==========
ipl_exposure
No recorded IPL appearance    213
Name: count, dtype: int64

========== BOWLER IPL EXPOSURE ==========
ipl_exposure
No recorded IPL appearance    183
Name: count, dtype: int64

========== BATTERS WITH IPL APPEARANCES ==========


,batter,latest_team,ipl_matches,ipl_runs,scouting_score



========== BOWLERS WITH IPL APPEARANCES ==========


,bowler_name,latest_team,ipl_matches,scouting_score



========== NAME MATCHING AUDIT ==========
Unmatched batters: 213
Unmatched bowlers: 183

Saved both IPL exposure files.


In [42]:
# ==========================================
# STAGE 4 FIX: IPL EXPOSURE NAME MATCHING
# ==========================================

import re
import numpy as np
import pandas as pd

def normalize_name(name):
    if pd.isna(name):
        return ""
    return re.sub(r"[^a-z]", "", str(name).lower())


# 1. Extract IPL records
ipl = df[
    df["competition"].astype(str).str.contains(
        "IPL|Indian Premier League",
        case=False,
        na=False
    )
].copy()

# 2. Build IPL batting exposure
ipl_batting = (
    ipl[ipl["batter"].notna()]
    .groupby("batter")
    .agg(
        ipl_matches=("match_id", "nunique"),
        ipl_runs=("runs", "sum")
    )
    .reset_index()
)

ipl_batting["match_key"] = (
    ipl_batting["batter"].apply(normalize_name)
)

ipl_batting = (
    ipl_batting.groupby("match_key")
    .agg(
        ipl_matches=("ipl_matches", "max"),
        ipl_runs=("ipl_runs", "sum")
    )
    .reset_index()
)

# 3. Build IPL bowling exposure
ipl_bowling = (
    ipl[ipl["bowler_name"].notna()]
    .groupby("bowler_name")
    .agg(
        ipl_matches=("match_id", "nunique")
    )
    .reset_index()
)

ipl_bowling["match_key"] = (
    ipl_bowling["bowler_name"].apply(normalize_name)
)

ipl_bowling = (
    ipl_bowling.groupby("match_key")
    .agg(
        ipl_matches=("ipl_matches", "max")
    )
    .reset_index()
)

# 4. Normalize candidate names
batters_ipl = batters_master.copy()
bowlers_ipl = bowlers_master.copy()

batters_ipl["match_key"] = (
    batters_ipl["batter"].apply(normalize_name)
)

bowlers_ipl["match_key"] = (
    bowlers_ipl["bowler_name"].apply(normalize_name)
)

# 5. Merge exposure
batters_ipl = batters_ipl.merge(
    ipl_batting,
    on="match_key",
    how="left"
)

bowlers_ipl = bowlers_ipl.merge(
    ipl_bowling,
    on="match_key",
    how="left"
)

# 6. Fill missing exposure
for col in ["ipl_matches", "ipl_runs"]:
    batters_ipl[col] = batters_ipl[col].fillna(0).astype(int)

bowlers_ipl["ipl_matches"] = (
    bowlers_ipl["ipl_matches"].fillna(0).astype(int)
)

# 7. Classify exposure
batters_ipl["ipl_exposure"] = np.where(
    batters_ipl["ipl_matches"] > 0,
    "Played IPL",
    "No exact name match"
)

bowlers_ipl["ipl_exposure"] = np.where(
    bowlers_ipl["ipl_matches"] > 0,
    "Played IPL",
    "No exact name match"
)

# 8. Save
batters_ipl.to_csv(
    DATA_DIR / "PRESSURE_X_Batter_IPL_Exposure.csv",
    index=False
)

bowlers_ipl.to_csv(
    DATA_DIR / "PRESSURE_X_Bowler_IPL_Exposure.csv",
    index=False
)

# 9. Validation
print("BATTER EXPOSURE")
print(batters_ipl["ipl_exposure"].value_counts())

print("\nBOWLER EXPOSURE")
print(bowlers_ipl["ipl_exposure"].value_counts())

print("\nMATCHED BATTERS")
display(
    batters_ipl[batters_ipl["ipl_matches"] > 0][
        ["batter", "latest_team", "ipl_matches",
         "ipl_runs", "scouting_score"]
    ].sort_values("ipl_matches", ascending=False)
)

print("\nMATCHED BOWLERS")
display(
    bowlers_ipl[bowlers_ipl["ipl_matches"] > 0][
        ["bowler_name", "latest_team",
         "ipl_matches", "scouting_score"]
    ].sort_values("ipl_matches", ascending=False)
)

BATTER EXPOSURE
ipl_exposure
No exact name match    213
Name: count, dtype: int64

BOWLER EXPOSURE
ipl_exposure
No exact name match    183
Name: count, dtype: int64

MATCHED BATTERS


,batter,latest_team,ipl_matches,ipl_runs,scouting_score



MATCHED BOWLERS


,bowler_name,latest_team,ipl_matches,scouting_score


In [47]:
print("BATTERS MASTER COLUMNS")
print(batters_master.columns.tolist())

print("\nBOWLERS MASTER COLUMNS")
print(bowlers_master.columns.tolist())

print("\nBATTERS MASTER SAMPLE")
display(batters_master.head(5))

print("\nBOWLERS MASTER SAMPLE")
display(bowlers_master.head(5))

print("\nIPL BATTING COLUMNS")
print(ipl_batting.columns.tolist())

print("\nIPL BOWLING COLUMNS")
print(ipl_bowling.columns.tolist())

BATTERS MASTER COLUMNS
['player_key', 'latest_smat_season', 'latest_team', 'latest_team_count', 'team_status', 'smat_seasons', 'total_smat_matches', 'recent_seasons', 'recent_smat_matches', 'batter', 'scouting_score']

BOWLERS MASTER COLUMNS
['player_key', 'latest_smat_season', 'latest_team', 'latest_team_count', 'team_status', 'smat_seasons', 'total_smat_matches', 'recent_seasons', 'recent_smat_matches', 'bowler_name', 'scouting_score']

BATTERS MASTER SAMPLE


,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,batter,scouting_score
0,rk singh,2024/25,Uttar Pradesh,1,Single team,8,35,3.00,7.00,RK Singh,93.57
1,aa barot,2020/21,Saurashtra,1,Single team,2,11,0.00,0.00,AA Barot,91.50
2,rm patidar,2024/25,Madhya Pradesh,1,Single team,8,34,3.00,9.00,RM Patidar,91.10
3,jm sharma,2024/25,Vidarbha,1,Single team,8,41,2.00,6.00,JM Sharma,90.49
4,r parag,2023/24,Assam,1,Single team,6,28,2.00,6.00,R Parag,90.23



BOWLERS MASTER SAMPLE


,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,bowler_name,scouting_score
0,ashutosh aman,2022/23,Bihar,1,Single team,5,24,1.00,3.00,Ashutosh Aman,94.84
1,shivank vashisth,2022/23,Delhi,1,Single team,3,8,1.00,4.00,Shivank Vashisth,92.31
2,c sakariya,2022/23,Saurashtra,1,Single team,5,23,1.00,2.00,C Sakariya,91.68
3,b kumar,2024/25,Uttar Pradesh,1,Single team,4,10,2.00,6.00,B Kumar,91.16
4,ravi teja,2023/24,Hyderabad (India),1,Single team,7,20,2.00,5.00,Ravi Teja,90.18



IPL BATTING COLUMNS
['match_key', 'ipl_matches', 'ipl_runs']

IPL BOWLING COLUMNS
['match_key', 'ipl_matches']


In [48]:
# ==========================================
# IPL NAME MATCHING: RAW DATA DIAGNOSTIC
# ==========================================

import re
import pandas as pd

def normalize_name(name):
    if pd.isna(name):
        return ""
    return re.sub(r"[^a-z]", "", str(name).lower())


# Extract IPL records
ipl = df[
    df["competition"].astype(str).str.contains(
        "IPL|Indian Premier League", case=False, na=False
    )
].copy()

# Get original IPL player names
ipl_batter_names = (
    ipl["batter"].dropna().astype(str).unique()
)

ipl_bowler_names = (
    ipl["bowler_name"].dropna().astype(str).unique()
)

# Normalize IPL names
ipl_batter_keys = {
    normalize_name(name): name for name in ipl_batter_names
}

ipl_bowler_keys = {
    normalize_name(name): name for name in ipl_bowler_names
}

# Normalize candidate names
batters_master["match_key"] = (
    batters_master["batter"].apply(normalize_name)
)

bowlers_master["match_key"] = (
    bowlers_master["bowler_name"].apply(normalize_name)
)

# Exact overlap
batters_master["exact_ipl_match"] = (
    batters_master["match_key"].isin(ipl_batter_keys)
)

bowlers_master["exact_ipl_match"] = (
    bowlers_master["match_key"].isin(ipl_bowler_keys)
)

print("IPL unique batter names:", len(ipl_batter_names))
print("IPL unique bowler names:", len(ipl_bowler_names))

print("\nExact batter matches:",
      batters_master["exact_ipl_match"].sum())

print("Exact bowler matches:",
      bowlers_master["exact_ipl_match"].sum())

print("\nCANDIDATE BATTERS")
display(
    batters_master[
        ["batter", "match_key", "exact_ipl_match"]
    ].head(25)
)

print("\nCANDIDATE BOWLERS")
display(
    bowlers_master[
        ["bowler_name", "match_key", "exact_ipl_match"]
    ].head(25)
)

print("\nSAMPLE RAW IPL BATTER NAMES")
display(pd.DataFrame({"IPL batter": sorted(ipl_batter_names)[:30]}))

print("\nSAMPLE RAW IPL BOWLER NAMES")
display(pd.DataFrame({"IPL bowler": sorted(ipl_bowler_names)[:30]}))

IPL unique batter names: 0
IPL unique bowler names: 0

Exact batter matches: 0
Exact bowler matches: 0

CANDIDATE BATTERS


,batter,match_key,exact_ipl_match
0,RK Singh,rksingh,False
1,AA Barot,aabarot,False
2,RM Patidar,rmpatidar,False
3,JM Sharma,jmsharma,False
4,R Parag,rparag,False
5,Punit Bisht,punitbisht,False
6,Anmolpreet Singh,anmolpreetsingh,False
7,N Rana,nrana,False
8,Rahul Singh,rahulsingh,False
9,A Manohar,amanohar,False



CANDIDATE BOWLERS


,bowler_name,match_key,exact_ipl_match
0,Ashutosh Aman,ashutoshaman,False
1,Shivank Vashisth,shivankvashisth,False
2,C Sakariya,csakariya,False
3,B Kumar,bkumar,False
4,Ravi Teja,raviteja,False
5,Cheepurapalli Stephen,cheepurapallistephen,False
6,Ravi Bishnoi,ravibishnoi,False
7,NA Rathva,narathva,False
8,Khrievitso Kense,khrievitsokense,False
9,Amit Ali,amitali,False



SAMPLE RAW IPL BATTER NAMES


,IPL batter



SAMPLE RAW IPL BOWLER NAMES


,IPL bowler


In [49]:
# ==========================================
# IPL PLAYER COLUMN AUDIT
# ==========================================

ipl = df[
    df["competition"].astype(str).str.contains(
        "IPL|Indian Premier League",
        case=False,
        na=False
    )
].copy()

print("IPL shape:", ipl.shape)
print("\nAvailable columns:")
print(ipl.columns.tolist())

# Check which player-related columns contain data
player_cols = [
    col for col in ipl.columns
    if any(word in col.lower() for word in [
        "batter", "striker", "bowler", "player", "name"
    ])
]

print("\nPLAYER COLUMN NON-NULL COUNTS:")

for col in player_cols:
    print(
        f"{col}: {ipl[col].notna().sum()} non-null | "
        f"{ipl[col].nunique()} unique"
    )

print("\nSAMPLE IPL ROWS:")
display(ipl.head(10))

print("\nNON-NULL PLAYER IDENTIFIERS:")
id_cols = [
    col for col in ipl.columns
    if "id" in col.lower()
]

display(ipl[id_cols].head(10))

IPL shape: (87687, 72)

Available columns:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time', 'original_match_id', 'source_competition', 'season_x', 'start_date', 'venue', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3', 'competition', 'comp_name', 'match_format', 'season_y',

,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,non_striker_name,bowler_id,bowler_name,batting_type,bowling_type,batting_order,runs,is_four,is_six,is_wicket,dismissal_type_id,dismissal_type,dismissed_player_id,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time,original_match_id,source_competition,season_x,start_date,venue,ball,actual_delivery,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3,competition,comp_name,match_format,season_y,batter,season,innings_key,is_legal_ball,batter_ball,dismissed_player,bowler_wicket,bowler_runs_conceded
3191,IPL_T20_T20I_14094,1,1.00,1.00,1.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,<NA>,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,NaN,<NA>,NaN,Overstep,1.00,0.00,Pace,0.00,Backfoot defensive,Inswing,Over,Off-side,2022-03-26 19:30:05 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,<NA>,<NA>,IPL_T20_T20I_14094_1,True,True,<NA>,False,0.00
3192,IPL_T20_T20I_14094,1,1.00,2.00,2.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,<NA>,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,NaN,<NA>,NaN,NaN,0.00,0.00,Pace,137.70,Cover Drive,Out Swing,Over,Off-side,2022-03-26 19:30:49 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,<NA>,<NA>,IPL_T20_T20I_14094_1,True,True,<NA>,False,0.00
3193,IPL_T20_T20I_14094,1,1.00,3.00,3.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,<NA>,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,NaN,<NA>,NaN,wide,1.00,0.00,Pace,139.30,Leg Glance,Seamed In,Over,On-side,2022-03-26 19:31:20 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,<NA>,<NA>,IPL_T20_T20I_14094_1,True,True,<NA>,False,0.00
3194,IPL_T20_T20I_14094,1,1.00,4.00,4.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,<NA>,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,NaN,<NA>,NaN,NaN,0.00,0.00,Pace,0.00,Backfoot defensive,Inswing,Over,Off-side,2022-03-26 19:31:52 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,<NA>,<NA>,IPL_T20_T20I_14094_1,True,True,<NA>,False,0.00
3195,IPL_T20_T20I_14094,1,1.00,5.00,5.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,<NA>,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,2.00,<NA>,6059.00,NaN,0.00,0.00,Pace,0.00,Square Cut,Out Swing,Over,Off-side,2022-03-26 19:32:31 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,<NA>,<NA>,IPL_T20_T20I_14094_1,True,True,<NA>,False,0.00
3196,IPL_T20_T20I_14094,1,1.00,6.00,6.00,1.00,8.00,169.00,Robin Uthappa,1941.00,Devon Conway,403.00,<NA>,RHB,Right-arm fast-medium,3.00,0.00,False,False,False,NaN,<NA>,NaN,wide,1.00,0.00,Pace,136.80,Leg Glance,Inswing,Over,On-side,2022-03-26 19:35:02 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,<NA>,<NA>,IPL_T20_T20I_14094_1,True,True,<NA>,False,0.00
3197,IPL_T20_T20I_14094,1,1.00,7.00,7.00,1.00,8.00,169.00,Robin Uthappa,1941.00,Devon Conway,403.00,<NA>,RHB,Right-arm fast-medium,3.00,0.00,False,False,False,NaN,<NA>,NaN,NaN,0.00,0.00,Pace,135.70,Off Drive,Out Swing,Over,Off-side,2022-03-26 19:35:32 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N


NON-NULL PLAYER IDENTIFIERS:


,match_id,overid,batting_team_id,bowling_team_id,striker_id,non_striker_id,bowler_id,dismissal_type_id,dismissed_player_id,original_match_id,wides
3191,IPL_T20_T20I_14094,1.00,1.00,8.00,6059.00,1941.00,403.00,NaN,NaN,14094,0.00
3192,IPL_T20_T20I_14094,1.00,1.00,8.00,6059.00,1941.00,403.00,NaN,NaN,14094,0.00
3193,IPL_T20_T20I_14094,1.00,1.00,8.00,6059.00,1941.00,403.00,NaN,NaN,14094,0.00
3194,IPL_T20_T20I_14094,1.00,1.00,8.00,6059.00,1941.00,403.00,NaN,NaN,14094,0.00
3195,IPL_T20_T20I_14094,1.00,1.00,8.00,6059.00,1941.00,403.00,2.00,6059.00,14094,0.00
3196,IPL_T20_T20I_14094,1.00,1.00,8.00,169.00,1941.00,403.00,NaN,NaN,14094,0.00
3197,IPL_T20_T20I_14094,1.00,1.00,8.00,169.00,1941.00,403.00,NaN,NaN,14094,0.00
3198,IPL_T20_T20I_14094,1.00,1.00,8.00,169.00,1941.00,403.00,NaN,NaN,14094,0.00
3199,IPL_T20_T20I_14094,1.00,1.00,8.00,169.00,1941.00,403.00,NaN,NaN,14094,0.00
3200,IPL_T20_T20I_14094,2.00,1.00,8.00,1941.00,169.00,6286.00,NaN,NaN,14094,0.00


In [50]:
# ==========================================
# IPL PLAYER ID MAPPING AUDIT
# ==========================================

mapping = data["competition_mapping"].copy()

print("MAPPING SHAPE:", mapping.shape)

print("\nMAPPING COLUMNS:")
print(mapping.columns.tolist())

print("\nMAPPING SAMPLE:")
display(mapping.head(10))

print("\nTEAM DATA COLUMNS:")
print(data["teams"].columns.tolist())

print("\nIPL PLAYER IDs:")
ipl = df[
    df["competition"].astype(str).str.contains(
        "IPL", case=False, na=False
    )
].copy()

for col in [
    "striker_id",
    "non_striker_id",
    "bowler_id",
    "dismissed_player_id"
]:
    if col in ipl.columns:
        print(
            col,
            "| Unique IDs:", ipl[col].nunique(),
            "| Non-null:", ipl[col].notna().sum()
        )

print("\nMAPPING COMPETITION DISTRIBUTION:")
display(mapping.iloc[:, 0].value_counts().head(15))

MAPPING SHAPE: (6413, 6)

MAPPING COLUMNS:
['comp_id', 'comp_name', 'team_id', 'team_name', 'player_id', 'player_name']

MAPPING SAMPLE:


,comp_id,comp_name,team_id,team_name,player_id,player_name
0,1331,England in West Indies T20I Series 2022,22,England,478,Eoin Morgan
1,1331,England in West Indies T20I Series 2022,22,England,479,Adil Rashid
2,1331,England in West Indies T20I Series 2022,22,England,722,Liam Dawson
3,1331,England in West Indies T20I Series 2022,22,England,786,Chris Jordan
4,1331,England in West Indies T20I Series 2022,22,England,1187,Jason Roy
5,1331,England in West Indies T20I Series 2022,22,England,1210,James Vince
6,1331,England in West Indies T20I Series 2022,22,England,1556,David Payne
7,1331,England in West Indies T20I Series 2022,22,England,1596,Moeen Ali
8,1331,England in West Indies T20I Series 2022,22,England,1924,Sam Billings
9,1331,England in West Indies T20I Series 2022,22,England,2304,Tymal Mills



TEAM DATA COLUMNS:
['comp_id', 'team_id', 'team_name']

IPL PLAYER IDs:
striker_id | Unique IDs: 341 | Non-null: 87687
non_striker_id | Unique IDs: 329 | Non-null: 87687
bowler_id | Unique IDs: 260 | Non-null: 87687
dismissed_player_id | Unique IDs: 307 | Non-null: 4462

MAPPING COMPETITION DISTRIBUTION:


comp_id
3298    275
1715    264
2193    263
2725    255
1358    251
3133    160
3289    140
2670    137
1533    116
1551     92
2605     56
2269     55
3159     55
1608     50
1523     49
Name: count, dtype: int64

In [51]:
# ==========================================
# STAGE 4: RECOVER IPL PLAYER IDENTITIES
# ==========================================

# 1. Extract IPL deliveries
ipl = df[
    df["competition"].astype(str).str.contains(
        "IPL|Indian Premier League",
        case=False,
        na=False
    )
].copy()

# 2. Build global player ID -> name mapping
player_lookup = (
    data["competition_mapping"]
    [["player_id", "player_name"]]
    .dropna()
    .drop_duplicates()
)

# Check whether any player ID has conflicting names
conflicts = (
    player_lookup.groupby("player_id")["player_name"]
    .nunique()
)

conflicting_ids = conflicts[conflicts > 1]

print("Player IDs in mapping:", player_lookup["player_id"].nunique())
print("Conflicting player IDs:", len(conflicting_ids))

# Keep IDs with a single consistent name
player_lookup = player_lookup[
    ~player_lookup["player_id"].isin(conflicting_ids.index)
].drop_duplicates("player_id")

player_lookup = player_lookup.set_index("player_id")["player_name"]

# 3. Recover player names from delivery IDs
for id_col, name_col in [
    ("striker_id", "ipl_batter"),
    ("bowler_id", "ipl_bowler"),
    ("non_striker_id", "ipl_non_striker"),
    ("dismissed_player_id", "ipl_dismissed_player")
]:
    ipl[name_col] = ipl[id_col].map(player_lookup)

# 4. Check recovery coverage
print("\nPLAYER NAME RECOVERY")

for col in [
    "ipl_batter",
    "ipl_bowler",
    "ipl_non_striker",
    "ipl_dismissed_player"
]:
    print(
        col,
        "| Recovered:", ipl[col].notna().sum(),
        "| Missing:", ipl[col].isna().sum(),
        "| Unique players:", ipl[col].nunique()
    )

# 5. Display sample recovered names
print("\nRECOVERED IPL BATTERS")
display(ipl["ipl_batter"].value_counts().head(20).reset_index())

print("\nRECOVERED IPL BOWLERS")
display(ipl["ipl_bowler"].value_counts().head(20).reset_index())

# 6. Save recovered IPL identity data
ipl.to_csv(
    DATA_DIR / "PRESSURE_X_IPL_Player_Identity.csv",
    index=False
)

print("\nSaved: PRESSURE_X_IPL_Player_Identity.csv")

Player IDs in mapping: 1217
Conflicting player IDs: 0

PLAYER NAME RECOVERY
ipl_batter | Recovered: 87687 | Missing: 0 | Unique players: 341
ipl_bowler | Recovered: 87687 | Missing: 0 | Unique players: 260
ipl_non_striker | Recovered: 87687 | Missing: 0 | Unique players: 329
ipl_dismissed_player | Recovered: 4462 | Missing: 83225 | Unique players: 307

RECOVERED IPL BATTERS


,ipl_batter,count
0,Virat Kohli,2167
1,Shubman Gill,2158
2,Jos Buttler,1849
3,Lokesh Rahul,1840
4,Sai Sudharsan,1760
5,Yashasvi Jaiswal,1538
6,Ruturaj Gaikwad,1507
7,Ishan Kishan,1490
8,Sanju Samson,1428
9,Suryakumar Yadav,1391



RECOVERED IPL BOWLERS


,ipl_bowler,count
0,Mohammed Siraj,1757
1,Rashid Khan,1739
2,Bhuvneshwar Kumar,1715
3,Arshdeep Singh,1681
4,Yuzvendra Chahal,1656
5,Sunil Narine,1556
6,Varun Chakravarthy,1440
7,Kuldeep Yadav,1423
8,Trent Boult,1419
9,Harshal Patel,1327



Saved: PRESSURE_X_IPL_Player_Identity.csv


In [52]:
from difflib import SequenceMatcher
import pandas as pd
import re

# Load recovered IPL identities if not already available
ipl = pd.read_csv(DATA_DIR / "PRESSURE_X_IPL_Player_Identity.csv")

# Build IPL appearance tables
def build_ipl_exposure(name_col):
    x = ipl[ipl[name_col].notna()].copy()

    return (
        x.groupby(name_col)
        .agg(
            IPL_Deliveries=("match_id", "size"),
            IPL_Matches=("match_id", "nunique")
        )
        .reset_index()
        .rename(columns={name_col: "IPL_Player"})
    )

ipl_batter_exposure = build_ipl_exposure("ipl_batter")
ipl_bowler_exposure = build_ipl_exposure("ipl_bowler")


# Name normalization
def clean_name(name):
    return re.sub(r"[^a-z]", "", str(name).lower())


def get_initial(name):
    tokens = str(name).strip().split()
    return tokens[0][0].lower() if tokens else ""


def get_surname(name):
    tokens = str(name).strip().split()
    return tokens[-1].lower() if tokens else ""


# Candidate matching
def match_candidates(candidate_df, candidate_col, ipl_exposure, role):

    ipl_names = ipl_exposure["IPL_Player"].dropna().unique().tolist()

    exact_lookup = {
        clean_name(name): name for name in ipl_names
    }

    results = []

    for candidate in candidate_df[candidate_col].dropna().unique():

        candidate_clean = clean_name(candidate)

        # 1. Exact name match
        if candidate_clean in exact_lookup:
            matched_name = exact_lookup[candidate_clean]
            match_type = "Exact match"
            status = "IPL Played"

        else:
            # 2. Match using first initial + surname
            possible = [
                name for name in ipl_names
                if get_initial(name) == get_initial(candidate)
                and get_surname(name) == get_surname(candidate)
            ]

            if len(possible) == 1:
                matched_name = possible[0]
                match_type = "Initial + surname"
                status = "Manual verification"

            elif len(possible) > 1:
                matched_name = "; ".join(possible)
                match_type = "Ambiguous"
                status = "Manual verification"

            else:
                # 3. Generate fuzzy suggestions, not automatic matches
                scores = sorted(
                    [
                        (
                            name,
                            SequenceMatcher(
                                None,
                                candidate_clean,
                                clean_name(name)
                            ).ratio()
                        )
                        for name in ipl_names
                    ],
                    key=lambda x: x[1],
                    reverse=True
                )

                best_name, best_score = scores[0] if scores else ("", 0)

                if best_score >= 0.55:
                    matched_name = best_name
                    match_type = f"Fuzzy suggestion ({best_score:.2f})"
                    status = "Manual verification"
                else:
                    matched_name = ""
                    match_type = "No match"
                    status = "No IPL appearance found"

        # Get exposure only for a single identified IPL name
        exposure = ipl_exposure[
            ipl_exposure["IPL_Player"] == matched_name
        ] if status == "IPL Played" else pd.DataFrame()

        if not exposure.empty:
            deliveries = int(exposure.iloc[0]["IPL_Deliveries"])
            matches = int(exposure.iloc[0]["IPL_Matches"])
        else:
            deliveries = 0
            matches = 0

        results.append({
            "Candidate": candidate,
            "IPL_Player_Match": matched_name,
            "Match_Type": match_type,
            "IPL_Status": status,
            "IPL_Matches": matches,
            "IPL_Deliveries": deliveries,
            "Role": role
        })

    return pd.DataFrame(results)


# Match batters and bowlers
batter_ipl_audit = match_candidates(
    batters_master,
    "batter",
    ipl_batter_exposure,
    "Batter"
)

bowler_ipl_audit = match_candidates(
    bowlers_master,
    "bowler_name",
    ipl_bowler_exposure,
    "Bowler"
)


# Save outputs
batter_ipl_audit.to_csv(
    DATA_DIR / "PRESSURE_X_Batter_IPL_Exposure_Audit.csv",
    index=False
)

bowler_ipl_audit.to_csv(
    DATA_DIR / "PRESSURE_X_Bowler_IPL_Exposure_Audit.csv",
    index=False
)


# Summary
for role, audit in [
    ("BATTERS", batter_ipl_audit),
    ("BOWLERS", bowler_ipl_audit)
]:
    print(f"\n{'='*20} {role} {'='*20}")
    print(audit["IPL_Status"].value_counts(dropna=False))
    print("\nCandidates requiring manual verification:")
    display(
        audit[audit["IPL_Status"] == "Manual verification"]
        [["Candidate", "IPL_Player_Match", "Match_Type"]]
    )

print("\nSaved both IPL exposure audit files.")


==================== BATTERS ====================
IPL_Status
Manual verification        167
No IPL appearance found     27
IPL Played                  19
Name: count, dtype: int64

Candidates requiring manual verification:


,Candidate,IPL_Player_Match,Match_Type
0,RK Singh,Ramandeep Singh; Ravi Singh; Rinku Singh,Ambiguous
1,AA Barot,Sean Abbott,Fuzzy suggestion (0.59)
2,RM Patidar,Rajat Patidar,Initial + surname
3,JM Sharma,Jitesh Sharma,Initial + surname
4,R Parag,Riyan Parag,Initial + surname
...,...,...,...
205,Nazeeb Saiyed,Navdeep Saini,Fuzzy suggestion (0.58)
206,Kangabam Singh,Kumar Kartikeya Singh,Initial + surname
207,Joseph Lalthankhuma,Shardul Thakur,Fuzzy suggestion (0.58)
208,Karnajit Yumnam,Karn Sharma,Fuzzy suggestion (0.58)



==================== BOWLERS ====================
IPL_Status
Manual verification        125
IPL Played                  30
No IPL appearance found     28
Name: count, dtype: int64

Candidates requiring manual verification:


,Candidate,IPL_Player_Match,Match_Type
0,Ashutosh Aman,Ashok Sharma,Fuzzy suggestion (0.78)
1,Shivank Vashisth,Shivang Kumar,Fuzzy suggestion (0.59)
2,C Sakariya,Chetan Sakariya,Initial + surname
3,B Kumar,Bhuvneshwar Kumar,Initial + surname
4,Ravi Teja,Travis Head,Fuzzy suggestion (0.67)
...,...,...,...
175,Jaskaran Singh,Shashank Singh,Fuzzy suggestion (0.69)
177,Vidyadhar Patil,Axar Patel,Fuzzy suggestion (0.61)
178,Harsh Tyagi,Harshit Rana,Fuzzy suggestion (0.67)
180,Diwesh Pathania,Digvesh Rathi,Fuzzy suggestion (0.69)



Saved both IPL exposure audit files.


In [53]:
def classify_ipl_exposure(audit, ipl_exposure):

    ipl_names = ipl_exposure["IPL_Player"].dropna().unique().tolist()
    results = []

    for _, row in audit.iterrows():

        candidate = row["Candidate"]
        candidate_clean = clean_name(candidate)

        # Exact full-name match
        exact = [
            name for name in ipl_names
            if clean_name(name) == candidate_clean
        ]

        if exact:
            matched = exact[0]
            status = "IPL Played - Exact"
            match_type = "Exact"

        else:
            # Match first initial + surname
            possible = [
                name for name in ipl_names
                if get_initial(name) == get_initial(candidate)
                and get_surname(name) == get_surname(candidate)
            ]

            if len(possible) == 1:
                matched = possible[0]
                status = "Probable IPL Played"
                match_type = "Unique initial + surname"

            elif len(possible) > 1:
                matched = "; ".join(possible)
                status = "Manual Verification"
                match_type = "Multiple possible matches"

            else:
                matched = ""
                status = "No IPL Match Found"
                match_type = "No identity match"

        results.append({
            "Candidate": candidate,
            "IPL_Player_Match": matched,
            "IPL_Status": status,
            "Match_Type": match_type
        })

    return pd.DataFrame(results)


batter_ipl_audit = classify_ipl_exposure(
    batter_ipl_audit,
    ipl_batter_exposure
)

bowler_ipl_audit = classify_ipl_exposure(
    bowler_ipl_audit,
    ipl_bowler_exposure
)

# Save revised audits
batter_ipl_audit.to_csv(
    DATA_DIR / "PRESSURE_X_Batter_IPL_Exposure_Audit.csv",
    index=False
)

bowler_ipl_audit.to_csv(
    DATA_DIR / "PRESSURE_X_Bowler_IPL_Exposure_Audit.csv",
    index=False
)

# Display summaries
for role, audit in [
    ("BATTERS", batter_ipl_audit),
    ("BOWLERS", bowler_ipl_audit)
]:
    print(f"\n{'='*20} {role} {'='*20}")
    print(audit["IPL_Status"].value_counts())

    print("\nProbable IPL appearances:")
    display(
        audit[audit["IPL_Status"] == "Probable IPL Played"]
    )

    print("\nAmbiguous identities:")
    display(
        audit[audit["IPL_Status"] == "Manual Verification"]
    )


==================== BATTERS ====================
IPL_Status
No IPL Match Found     135
Probable IPL Played     54
IPL Played - Exact      19
Manual Verification      5
Name: count, dtype: int64

Probable IPL appearances:


,Candidate,IPL_Player_Match,IPL_Status,Match_Type
2,RM Patidar,Rajat Patidar,Probable IPL Played,Unique initial + surname
3,JM Sharma,Jitesh Sharma,Probable IPL Played,Unique initial + surname
4,R Parag,Riyan Parag,Probable IPL Played,Unique initial + surname
7,N Rana,Nitish Rana,Probable IPL Played,Unique initial + surname
9,A Manohar,Abhinav Manohar,Probable IPL Played,Unique initial + surname
11,P Simran Singh,Prabhsimran Singh,Probable IPL Played,Unique initial + surname
13,VR Iyer,Venkatesh Iyer,Probable IPL Played,Unique initial + surname
15,SP Jackson,Sheldon Jackson,Probable IPL Played,Unique initial + surname
17,AM Rahane,Ajinkya Rahane,Probable IPL Played,Unique initial + surname
19,PN Mankad,Prerak Mankad,Probable IPL Played,Unique initial + surname



Ambiguous identities:


,Candidate,IPL_Player_Match,IPL_Status,Match_Type
0,RK Singh,Ramandeep Singh; Ravi Singh; Rinku Singh,Manual Verification,Multiple possible matches
8,Rahul Singh,Ramandeep Singh; Ravi Singh; Rinku Singh,Manual Verification,Multiple possible matches
82,SB Dubey,Saurabh Dubey; Shubham Dubey,Manual Verification,Multiple possible matches
89,KS Sharma,Karan Sharma; Karn Sharma; Kartik Sharma,Manual Verification,Multiple possible matches
118,KV Sharma,Karan Sharma; Karn Sharma; Kartik Sharma,Manual Verification,Multiple possible matches



==================== BOWLERS ====================
IPL_Status
No IPL Match Found     113
Probable IPL Played     38
IPL Played - Exact      30
Manual Verification      2
Name: count, dtype: int64

Probable IPL appearances:


,Candidate,IPL_Player_Match,IPL_Status,Match_Type
2,C Sakariya,Chetan Sakariya,Probable IPL Played,Unique initial + surname
3,B Kumar,Bhuvneshwar Kumar,Probable IPL Played,Unique initial + surname
11,R Dhawan,Rishi Dhawan,Probable IPL Played,Unique initial + surname
17,K Gowtham,Krishnappa Gowtham,Probable IPL Played,Unique initial + surname
20,I Sharma,Ishant Sharma,Probable IPL Played,Unique initial + surname
24,S Kaul,Siddarth Kaul,Probable IPL Played,Unique initial + surname
25,Yuvraj Singh,Yudhvir Singh,Probable IPL Played,Unique initial + surname
26,JD Unadkat,Jaydev Unadkat,Probable IPL Played,Unique initial + surname
27,TU Deshpande,Tushar Deshpande,Probable IPL Played,Unique initial + surname
28,DG Nalkande,Darshan Nalkande,Probable IPL Played,Unique initial + surname



Ambiguous identities:


,Candidate,IPL_Player_Match,IPL_Status,Match_Type
82,Ankit Sharma,Abhishek Sharma; Ashok Sharma,Manual Verification,Multiple possible matches
166,Shiva Singh,Shashank Singh; Simarjeet Singh; Swapnil Singh,Manual Verification,Multiple possible matches


In [54]:
# Create a clean IPL identity verification register

def create_verification_register(audit, role):

    result = audit.copy()

    result["Verification_Status"] = np.where(
        result["IPL_Status"] == "IPL Played - Exact",
        "Confirmed",
        np.where(
            result["IPL_Status"] == "No IPL Match Found",
            "No match in available IPL data",
            "Pending manual verification"
        )
    )

    result["Final_IPL_Identity"] = np.where(
        result["Verification_Status"] == "Confirmed",
        result["IPL_Player_Match"],
        ""
    )

    result["Role"] = role

    return result[
        [
            "Candidate",
            "IPL_Player_Match",
            "Match_Type",
            "Verification_Status",
            "Final_IPL_Identity",
            "Role"
        ]
    ]


batter_review = create_verification_register(
    batter_ipl_audit, "Batter"
)

bowler_review = create_verification_register(
    bowler_ipl_audit, "Bowler"
)

# Combine both roles
ipl_verification_register = pd.concat(
    [batter_review, bowler_review],
    ignore_index=True
)

ipl_verification_register.to_csv(
    DATA_DIR / "PRESSURE_X_IPL_Identity_Verification.csv",
    index=False
)

print("IDENTITY VERIFICATION SUMMARY")

display(
    ipl_verification_register.groupby(
        ["Role", "Verification_Status"]
    ).size().unstack(fill_value=0)
)

print("\nSaved: PRESSURE_X_IPL_Identity_Verification.csv")

IDENTITY VERIFICATION SUMMARY


Verification_Status,Confirmed,No match in available IPL data,Pending manual verification
Role,,,
Batter,19,135,59
Bowler,30,113,40



Saved: PRESSURE_X_IPL_Identity_Verification.csv


In [55]:
# Integrate IPL exposure into batter and bowler master datasets

def integrate_ipl_exposure(master_df, audit_df, candidate_col):

    audit = audit_df[
        ["Candidate", "IPL_Player_Match", "IPL_Status", "Match_Type"]
    ].copy()

    audit = audit.rename(columns={
        "Candidate": candidate_col,
        "IPL_Player_Match": "IPL_Player_Match",
        "IPL_Status": "IPL_Match_Status",
        "Match_Type": "IPL_Match_Type"
    })

    result = master_df.merge(
        audit,
        on=candidate_col,
        how="left",
        validate="one_to_one"
    )

    result["IPL_Exposure"] = result["IPL_Match_Status"].map({
        "IPL Played - Exact": "Confirmed IPL appearance",
        "Probable IPL Played": "Pending identity verification",
        "Manual Verification": "Pending identity verification",
        "No IPL Match Found": "No appearance found in available IPL data"
    }).fillna("Not assessed")

    return result


batters_master = integrate_ipl_exposure(
    batters_master,
    batter_ipl_audit,
    "batter"
)

bowlers_master = integrate_ipl_exposure(
    bowlers_master,
    bowler_ipl_audit,
    "bowler_name"
)

# Save updated master files
batters_master.to_csv(
    DATA_DIR / "PRESSURE_X_Batter_Master.csv",
    index=False
)

bowlers_master.to_csv(
    DATA_DIR / "PRESSURE_X_Bowler_Master.csv",
    index=False
)

# Validate
print("BATTER MASTER:", batters_master.shape)
display(batters_master["IPL_Exposure"].value_counts())

print("\nBOWLER MASTER:", bowlers_master.shape)
display(bowlers_master["IPL_Exposure"].value_counts())

print("\nUpdated master datasets saved.")

BATTER MASTER: (213, 17)


IPL_Exposure
No appearance found in available IPL data    135
Pending identity verification                 59
Confirmed IPL appearance                      19
Name: count, dtype: int64


BOWLER MASTER: (183, 17)


IPL_Exposure
No appearance found in available IPL data    113
Pending identity verification                 40
Confirmed IPL appearance                      30
Name: count, dtype: int64


Updated master datasets saved.


In [56]:
# ==========================================
# PRESSURE-X: DOMESTIC PERFORMANCE CONSISTENCY
# ==========================================

smat_raw = data["smat"].copy()

# Standardize season labels
smat_raw["season"] = smat_raw["season"].astype(str).str.strip()

# Numeric conversions
numeric_cols = [
    "runs_off_bat", "wides", "noballs",
    "byes", "legbyes", "extras", "non_boundary"
]

for col in numeric_cols:
    smat_raw[col] = pd.to_numeric(
        smat_raw[col], errors="coerce"
    ).fillna(0)

# Valid delivery calculations
smat_raw["batter_ball"] = (
    smat_raw["wides"] == 0
).astype(int)

smat_raw["bowler_ball"] = (
    (smat_raw["wides"] == 0) &
    (smat_raw["noballs"] == 0)
).astype(int)

smat_raw["bowler_runs"] = (
    smat_raw["runs_off_bat"] +
    smat_raw["wides"] +
    smat_raw["noballs"]
)

smat_raw["is_four"] = (
    (smat_raw["runs_off_bat"] == 4) &
    (smat_raw["non_boundary"] == 0)
).astype(int)

smat_raw["is_six"] = (
    (smat_raw["runs_off_bat"] == 6) &
    (smat_raw["non_boundary"] == 0)
).astype(int)

# Dismissals credited to bowler
non_bowler_dismissals = {
    "run out",
    "retired hurt",
    "retired out",
    "obstructing the field"
}

smat_raw["dismissal_clean"] = (
    smat_raw["wicket_type"]
    .astype(str)
    .str.lower()
    .str.strip()
)

smat_raw["bowler_wicket"] = (
    smat_raw["wicket_type"].notna() &
    ~smat_raw["dismissal_clean"].isin(non_bowler_dismissals)
).astype(int)

# ==========================================
# BATTING SEASON STATS
# ==========================================

bat = smat_raw[smat_raw["striker"].notna()].copy()

batting_season_stats = (
    bat.groupby(["striker", "season"])
    .agg(
        Matches=("match_id", "nunique"),
        Innings=("innings", "nunique"),
        Runs=("runs_off_bat", "sum"),
        Balls=("batter_ball", "sum"),
        Fours=("is_four", "sum"),
        Sixes=("is_six", "sum"),
        Dismissals=("player_dismissed", "count")
    )
    .reset_index()
)

batting_season_stats["Average"] = (
    batting_season_stats["Runs"] /
    batting_season_stats["Dismissals"].replace(0, np.nan)
)

batting_season_stats["Strike_Rate"] = (
    batting_season_stats["Runs"] /
    batting_season_stats["Balls"].replace(0, np.nan) * 100
)

batting_season_stats["Boundary_Percentage"] = (
    (batting_season_stats["Fours"] * 4 +
     batting_season_stats["Sixes"] * 6) /
    batting_season_stats["Runs"].replace(0, np.nan) * 100
)

# ==========================================
# BOWLING SEASON STATS
# ==========================================

bowl = smat_raw[smat_raw["bowler"].notna()].copy()

bowling_season_stats = (
    bowl.groupby(["bowler", "season"])
    .agg(
        Matches=("match_id", "nunique"),
        Balls=("bowler_ball", "sum"),
        Runs_Conceded=("bowler_runs", "sum"),
        Wickets=("bowler_wicket", "sum")
    )
    .reset_index()
)

bowling_season_stats["Overs"] = (
    bowling_season_stats["Balls"] / 6
)

bowling_season_stats["Economy"] = (
    bowling_season_stats["Runs_Conceded"] /
    bowling_season_stats["Overs"].replace(0, np.nan)
)

bowling_season_stats["Average"] = (
    bowling_season_stats["Runs_Conceded"] /
    bowling_season_stats["Wickets"].replace(0, np.nan)
)

bowling_season_stats["Strike_Rate"] = (
    bowling_season_stats["Balls"] /
    bowling_season_stats["Wickets"].replace(0, np.nan)
)

# ==========================================
# CONSOLIDATED BATTER CONSISTENCY
# ==========================================

batting_consistency = (
    batting_season_stats.groupby("striker")
    .agg(
        Active_Seasons=("season", "nunique"),
        Total_Matches=("Matches", "sum"),
        Total_Runs=("Runs", "sum"),
        Avg_Season_Average=("Average", "mean"),
        Avg_Season_SR=("Strike_Rate", "mean"),
        Average_SR_SD=("Strike_Rate", "std"),
        Latest_Season=("season", "last")
    )
    .reset_index()
    .rename(columns={"striker": "batter"})
)

# ==========================================
# CONSOLIDATED BOWLER CONSISTENCY
# ==========================================

bowling_consistency = (
    bowling_season_stats.groupby("bowler")
    .agg(
        Active_Seasons=("season", "nunique"),
        Total_Matches=("Matches", "sum"),
        Total_Wickets=("Wickets", "sum"),
        Avg_Season_Economy=("Economy", "mean"),
        Avg_Season_Bowling_Average=("Average", "mean"),
        Avg_Season_Bowling_SR=("Strike_Rate", "mean"),
        Economy_SD=("Economy", "std"),
        Latest_Season=("season", "last")
    )
    .reset_index()
    .rename(columns={"bowler": "bowler_name"})
)

# ==========================================
# MERGE INTO PLAYER MASTERS
# ==========================================

batters_master = batters_master.drop(
    columns=[
        c for c in batting_consistency.columns
        if c != "batter" and c in batters_master.columns
    ],
    errors="ignore"
).merge(
    batting_consistency,
    on="batter",
    how="left",
    validate="one_to_one"
)

bowlers_master = bowlers_master.drop(
    columns=[
        c for c in bowling_consistency.columns
        if c != "bowler_name" and c in bowlers_master.columns
    ],
    errors="ignore"
).merge(
    bowling_consistency,
    on="bowler_name",
    how="left",
    validate="one_to_one"
)

# Save detailed seasonal and master outputs
batting_season_stats.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Batting_Season_Consistency.csv",
    index=False
)

bowling_season_stats.to_csv(
    DATA_DIR / "PRESSURE_X_SMAT_Bowling_Season_Consistency.csv",
    index=False
)

batters_master.to_csv(
    DATA_DIR / "PRESSURE_X_Batter_Master.csv",
    index=False
)

bowlers_master.to_csv(
    DATA_DIR / "PRESSURE_X_Bowler_Master.csv",
    index=False
)

# ==========================================
# VALIDATION
# ==========================================

print("BATTER CONSISTENCY")
print("Candidates:", len(batters_master))
print("Matched:", batters_master["Active_Seasons"].notna().sum())
display(
    batters_master.sort_values(
        ["Active_Seasons", "Total_Runs"],
        ascending=False
    )[[
        "batter", "Active_Seasons", "Total_Matches",
        "Total_Runs", "Avg_Season_Average",
        "Avg_Season_SR", "Average_SR_SD"
    ]].head(15)
)

print("\nBOWLER CONSISTENCY")
print("Candidates:", len(bowlers_master))
print("Matched:", bowlers_master["Active_Seasons"].notna().sum())
display(
    bowlers_master.sort_values(
        ["Active_Seasons", "Total_Wickets"],
        ascending=False
    )[[
        "bowler_name", "Active_Seasons", "Total_Matches",
        "Total_Wickets", "Avg_Season_Economy",
        "Avg_Season_Bowling_Average",
        "Economy_SD"
    ]].head(15)
)

print("\nSeasonal consistency datasets saved.")

BATTER CONSISTENCY
Candidates: 213
Matched: 213


,batter,Active_Seasons,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,Average_SR_SD
7,N Rana,9,46,1376,36.82,138.33,35.82
13,VR Iyer,9,33,901,44.52,133.66,18.91
36,S Dube,9,35,574,35.56,135.86,19.85
61,Harpreet Singh,8,34,1181,45.31,128.00,23.96
2,RM Patidar,8,33,1132,39.74,149.75,27.80
26,Mandeep Singh,8,37,1124,52.52,148.48,16.23
15,SP Jackson,8,37,1069,57.93,123.88,23.17
87,Vivek Singh,8,42,1056,26.14,126.32,30.53
3,JM Sharma,8,41,979,28.39,152.73,38.82
55,Priyank Panchal,8,35,968,28.55,131.46,14.66



BOWLER CONSISTENCY
Candidates: 183
Matched: 183


,bowler_name,Active_Seasons,Total_Matches,Total_Wickets,Avg_Season_Economy,Avg_Season_Bowling_Average,Economy_SD
26,JD Unadkat,9,41,65,6.60,21.16,1.42
40,A Sheth,9,44,59,8.25,20.17,1.55
129,S Dube,9,34,29,8.44,31.56,3.22
134,Chirag Jani,9,36,28,7.25,34.44,1.53
139,Dharmendrasinh Jadeja,9,38,26,7.25,42.00,1.14
86,VR Iyer,9,30,22,8.01,25.55,1.84
44,PP Chawla,8,43,67,6.38,18.50,1.21
22,LI Meriwala,8,40,62,7.52,18.95,1.05
73,KV Sharma,8,36,33,7.15,48.78,1.37
137,Manisankar Murasingh,8,37,27,7.39,31.90,1.15



Seasonal consistency datasets saved.


In [57]:
# ==========================================
# PRESSURE-X: PLAYER PROFILE ENRICHMENT
# ==========================================

# Batter profile
batter_profile = batters_master.copy()

batter_profile["Role"] = "Batter"
batter_profile["Date_of_Birth"] = ""
batter_profile["Age_as_of_2026_11_16"] = np.nan
batter_profile["DOB_Source"] = ""
batter_profile["Age_Verification"] = "Pending"

# Bowler profile
bowler_profile = bowlers_master.copy()

bowler_profile["Role"] = "Bowler"
bowler_profile["Date_of_Birth"] = ""
bowler_profile["Age_as_of_2026_11_16"] = np.nan
bowler_profile["DOB_Source"] = ""
bowler_profile["Age_Verification"] = "Pending"

# Standardize player names
batter_profile = batter_profile.rename(
    columns={"batter": "Player"}
)

bowler_profile = bowler_profile.rename(
    columns={"bowler_name": "Player"}
)

# Combine
player_profiles = pd.concat(
    [batter_profile, bowler_profile],
    ignore_index=True,
    sort=False
)

# Keep useful columns
profile_columns = [
    "Player", "Role",
    "latest_team", "latest_smat_season",
    "total_smat_matches", "smat_seasons",
    "recent_smat_matches", "recent_seasons",
    "IPL_Exposure", "IPL_Match_Status",
    "Date_of_Birth", "Age_as_of_2026_11_16",
    "DOB_Source", "Age_Verification"
]

player_profiles = player_profiles[
    [c for c in profile_columns if c in player_profiles.columns]
]

# Save enrichment sheet
player_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Player_Profile_Enrichment.csv",
    index=False
)

print("Total profiles:", len(player_profiles))
print("Batters:", (player_profiles["Role"] == "Batter").sum())
print("Bowlers:", (player_profiles["Role"] == "Bowler").sum())

print("\nDOB verification pending:",
      (player_profiles["Age_Verification"] == "Pending").sum())

display(player_profiles.head(10))

Total profiles: 396
Batters: 213
Bowlers: 183

DOB verification pending: 396


,Player,Role,latest_team,latest_smat_season,total_smat_matches,smat_seasons,recent_smat_matches,recent_seasons,IPL_Exposure,IPL_Match_Status,Date_of_Birth,Age_as_of_2026_11_16,DOB_Source,Age_Verification
0,RK Singh,Batter,Uttar Pradesh,2024/25,35,8,7.00,3.00,Pending identity verification,Manual Verification,,NaN,,Pending
1,AA Barot,Batter,Saurashtra,2020/21,11,2,0.00,0.00,No appearance found in available IPL data,No IPL Match Found,,NaN,,Pending
2,RM Patidar,Batter,Madhya Pradesh,2024/25,34,8,9.00,3.00,Pending identity verification,Probable IPL Played,,NaN,,Pending
3,JM Sharma,Batter,Vidarbha,2024/25,41,8,6.00,2.00,Pending identity verification,Probable IPL Played,,NaN,,Pending
4,R Parag,Batter,Assam,2023/24,28,6,6.00,2.00,Pending identity verification,Probable IPL Played,,NaN,,Pending
5,Punit Bisht,Batter,Meghalaya,2021/22,20,4,0.00,0.00,No appearance found in available IPL data,No IPL Match Found,,NaN,,Pending
6,Anmolpreet Singh,Batter,Punjab,2023/24,33,7,12.00,2.00,Confirmed IPL appearance,IPL Played - Exact,,NaN,,Pending
7,N Rana,Batter,Uttar Pradesh,2024/25,48,9,10.00,3.00,Pending identity verification,Probable IPL Played,,NaN,,Pending
8,Rahul Singh,Batter,Hyderabad (India),2023/24,29,6,2.00,1.00,Pending identity verification,Manual Verification,,NaN,,Pending
9,A Manohar,Batter,Karnataka,2023/24,8,3,4.00,2.00,Pending identity verification,Probable IPL Played,,NaN,,Pending


In [58]:
# ==========================================
# PRESSURE-X: DOB VERIFICATION QUEUE
# ==========================================

profiles = player_profiles.copy()

# Numeric fields
for col in [
    "recent_smat_matches",
    "total_smat_matches",
    "smat_seasons"
]:
    profiles[col] = pd.to_numeric(
        profiles[col], errors="coerce"
    )

# Prioritize candidates with no IPL match found,
# then pending identities, then confirmed IPL players.
priority_map = {
    "No appearance found in available IPL data": 1,
    "Pending identity verification": 2,
    "Confirmed IPL appearance": 3
}

profiles["Verification_Priority"] = (
    profiles["IPL_Exposure"].map(priority_map).fillna(4)
)

# Rank by recent participation and domestic experience
profiles = profiles.sort_values(
    [
        "Verification_Priority",
        "recent_smat_matches",
        "total_smat_matches"
    ],
    ascending=[True, False, False]
).reset_index(drop=True)

profiles["DOB_Verification_Order"] = range(1, len(profiles) + 1)

profiles["DOB_Source"] = ""
profiles["DOB_Verification"] = "Pending"
profiles["Date_of_Birth"] = ""
profiles["Age_as_of_2026_11_16"] = np.nan

# Save verification queue
profiles.to_csv(
    DATA_DIR / "PRESSURE_X_DOB_Verification_Queue.csv",
    index=False
)

print("DOB VERIFICATION QUEUE")
print("Total:", len(profiles))
print("\nPriority distribution:")
display(profiles["IPL_Exposure"].value_counts())

print("\nFirst 30 candidates to verify:")
display(
    profiles[[
        "DOB_Verification_Order",
        "Player",
        "Role",
        "latest_team",
        "total_smat_matches",
        "recent_smat_matches",
        "IPL_Exposure"
    ]].head(30)
)

DOB VERIFICATION QUEUE
Total: 396

Priority distribution:


IPL_Exposure
No appearance found in available IPL data    248
Pending identity verification                 99
Confirmed IPL appearance                      49
Name: count, dtype: int64


First 30 candidates to verify:


,DOB_Verification_Order,Player,Role,latest_team,total_smat_matches,recent_smat_matches,IPL_Exposure
0,1,Tanush Kotian,Bowler,Mumbai,16,12.00,No appearance found in available IPL data
1,2,A Sheth,Batter,Baroda,44,10.00,No appearance found in available IPL data
2,3,A Sheth,Bowler,Baroda,44,10.00,No appearance found in available IPL data
3,4,LI Meriwala,Bowler,Baroda,41,10.00,No appearance found in available IPL data
4,5,Vishnu Solanki,Batter,Baroda,40,10.00,No appearance found in available IPL data
5,6,Dharmendrasinh Jadeja,Bowler,Saurashtra,39,9.00,No appearance found in available IPL data
6,7,Chirag Jani,Batter,Saurashtra,37,9.00,No appearance found in available IPL data
7,8,Chirag Jani,Bowler,Saurashtra,37,9.00,No appearance found in available IPL data
8,9,Baltej Singh,Bowler,Punjab,27,9.00,No appearance found in available IPL data
9,10,Bhanu Pania,Batter,Baroda,18,9.00,No appearance found in available IPL data


In [59]:
# ==========================================
# UNIQUE PLAYER DOB VERIFICATION QUEUE
# ==========================================

profiles = player_profiles.copy()

# Normalize names and teams for grouping
profiles["Player_Key"] = (
    profiles["Player"].astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9]", "", regex=True)
)

# Group exact player-name and team matches
unique_profiles = (
    profiles.groupby(["Player_Key", "latest_team"], as_index=False)
    .agg(
        Player=("Player", "first"),
        Roles=("Role", lambda x: ", ".join(sorted(set(x)))),
        latest_smat_season=("latest_smat_season", "max"),
        total_smat_matches=("total_smat_matches", "max"),
        recent_smat_matches=("recent_smat_matches", "max"),
        IPL_Exposure=("IPL_Exposure", "first")
    )
)

# Verification priority
priority_map = {
    "No appearance found in available IPL data": 1,
    "Pending identity verification": 2,
    "Confirmed IPL appearance": 3
}

unique_profiles["Priority"] = (
    unique_profiles["IPL_Exposure"].map(priority_map).fillna(4)
)

unique_profiles = unique_profiles.sort_values(
    ["Priority", "recent_smat_matches", "total_smat_matches"],
    ascending=[True, False, False]
).reset_index(drop=True)

unique_profiles["DOB_Verification_Order"] = (
    range(1, len(unique_profiles) + 1)
)

unique_profiles["Date_of_Birth"] = ""
unique_profiles["Age_as_of_2026_11_16"] = np.nan
unique_profiles["DOB_Source"] = ""
unique_profiles["Age_Verification"] = "Pending"

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("Original profiles:", len(profiles))
print("Unique player-team records:", len(unique_profiles))
print("Duplicate records consolidated:", len(profiles) - len(unique_profiles))

display(unique_profiles.head(30))

Original profiles: 396
Unique player-team records: 364
Duplicate records consolidated: 32


,Player_Key,latest_team,Player,Roles,latest_smat_season,total_smat_matches,recent_smat_matches,IPL_Exposure,Priority,DOB_Verification_Order,Date_of_Birth,Age_as_of_2026_11_16,DOB_Source,Age_Verification
0,tanushkotian,Mumbai,Tanush Kotian,Bowler,2024/25,16,12.00,No appearance found in available IPL data,1,1,,NaN,,Pending
1,asheth,Baroda,A Sheth,"Batter, Bowler",2024/25,44,10.00,No appearance found in available IPL data,1,2,,NaN,,Pending
2,limeriwala,Baroda,LI Meriwala,Bowler,2024/25,41,10.00,No appearance found in available IPL data,1,3,,NaN,,Pending
3,vishnusolanki,Baroda,Vishnu Solanki,Batter,2024/25,40,10.00,No appearance found in available IPL data,1,4,,NaN,,Pending
4,dharmendrasinhjadeja,Saurashtra,Dharmendrasinh Jadeja,Bowler,2024/25,39,9.00,No appearance found in available IPL data,1,5,,NaN,,Pending
5,chiragjani,Saurashtra,Chirag Jani,"Batter, Bowler",2024/25,37,9.00,No appearance found in available IPL data,1,6,,NaN,,Pending
6,baltejsingh,Punjab,Baltej Singh,Bowler,2023/24,27,9.00,No appearance found in available IPL data,1,7,,NaN,,Pending
7,bhanupania,Baroda,Bhanu Pania,Batter,2024/25,18,9.00,No appearance found in available IPL data,1,8,,NaN,,Pending
8,mohitavasthi,Mumbai,Mohit Avasthi,Bowler,2024/25,13,9.00,No appearance found in available IPL data,1,9,,NaN,,Pending
9,narathva,Baroda,NA Rathva,"Batter, Bowler",2023/24,30,8.00,No appearance found in available IPL data,1,10,,NaN,,Pending


In [60]:
# Apply researched DOBs to the unique-player queue

verified_dobs = {
    "tanushkotian": ("1998-10-16", "Cricbuzz / Cricket Australia", "Cross-checked"),
    "rongsenjonathan": ("1986-10-04", "Cricket profile", "Secondary source"),
    "imliwatilemtur": ("1991-12-25", "Cricket profile", "Secondary source"),
    "tahmeedrahman": ("1998-09-26", "Cricket profile", "Secondary source"),
    "hokaitozhimomi": ("1986-09-24", "Cricket profile", "Secondary source"),
    "sedezhalierupero": ("1997-11-15", "Cricket profile", "Secondary source")
}

for idx, row in unique_profiles.iterrows():
    key = row["Player_Key"]

    if key in verified_dobs:
        dob, source, status = verified_dobs[key]
        unique_profiles.at[idx, "Date_of_Birth"] = dob
        unique_profiles.at[idx, "DOB_Source"] = source
        unique_profiles.at[idx, "Age_Verification"] = status

# Calculate age on 16 November 2026
reference_date = pd.Timestamp("2026-11-16")

dob_dates = pd.to_datetime(
    unique_profiles["Date_of_Birth"],
    errors="coerce"
)

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob_dates.dt.year
    - (
        (reference_date.month < dob_dates.dt.month)
        | (
            (reference_date.month == dob_dates.dt.month)
            & (reference_date.day < dob_dates.dt.day)
        )
    )
)

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob_dates.notna().sum())
print("DOB records pending:", dob_dates.isna().sum())


DOB records entered: 6
DOB records pending: 358


In [61]:
display(
    unique_profiles.loc[30:59, [
        "DOB_Verification_Order",
        "Player",
        "Roles",
        "latest_team",
        "IPL_Exposure",
        "Date_of_Birth",
        "Age_Verification"
    ]]
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth,Age_Verification
30,31,Joshua Ozukum,Batter,Nagaland,No appearance found in available IPL data,,Pending
31,32,Joseph Lalthankhuma,Batter,Mizoram,No appearance found in available IPL data,,Pending
32,33,Akshay Karnewar,Bowler,Vidarbha,No appearance found in available IPL data,,Pending
33,34,Manisankar Murasingh,"Batter, Bowler",Tripura,No appearance found in available IPL data,,Pending
34,35,Tanmay Agarwal,Batter,Hyderabad (India),No appearance found in available IPL data,,Pending
35,36,Ashwin Hebbar,Batter,Andhra,No appearance found in available IPL data,,Pending
36,37,Parth Sahani,Batter,Madhya Pradesh,No appearance found in available IPL data,,Pending
37,38,P Chopra,Batter,Himachal Pradesh,No appearance found in available IPL data,,Pending
38,39,CV Milind,Bowler,Hyderabad (India),No appearance found in available IPL data,,Pending
39,40,GH Vihari,Batter,Andhra,No appearance found in available IPL data,,Pending


In [62]:
batch2_dobs = {
    "joshuaozukum": ("2000-08-04", "Cricbuzz", "Verified"),
    "josephlalthankhuma": ("2000-09-26", "Cricbuzz", "Verified"),
    "akshaykarnewar": ("1992-10-12", "BCCI", "Verified"),
    "manisankarmurasingh": ("1993-01-01", "BCCI", "Verified"),
    "tanmayagarwal": ("1995-05-03", "Cricbuzz", "Verified"),
    "ashwinhebbar": ("1995-11-15", "Cricbuzz", "Verified"),
    "parthsahani": ("1993-03-09", "Cricbuzz", "Verified"),
    "cv milind": ("1994-09-04", "Cricbuzz", "Verified"),
    "hanumavihari": ("1993-10-13", "Cricbuzz", "Verified"),
    "ajaysarkar": ("1997-03-10", "Cricbuzz", "Verified"),
    "bishworjitkonthoujam": ("1996-02-03", "Cricbuzz", "Cross-checked"),
    "akshaysinghbains": ("1995-12-16", "Cricbuzz", "Cross-checked"),
    "lalhruairalte": ("1992-03-10", "Cricbuzz", "Verified"),
    "kanwarabhinay": ("1991-06-16", "ESPNcricinfo reference", "Cross-checked"),
    "rajatdey": ("1996-12-31", "BCCI", "Verified"),
    "sudipchatterjee": ("1991-11-11", "Cricbuzz", "Verified"),
    "akshaywadkar": ("1994-07-09", "BCCI", "Verified"),
    "tilakvarma": ("2002-11-08", "BCCI", "Verified"),
    "vishvarajjadeja": ("1998-07-19", "Cricbuzz", "Verified"),
    "rishavdas": ("1989-12-16", "BCCI", "Verified")
}

# Review the keys before applying because the queue's Player_Key
# may use different abbreviations for some names.
print("Available keys:")
display(unique_profiles.loc[30:59, ["Player", "Player_Key"]])

Available keys:


,Player,Player_Key
30,Joshua Ozukum,joshuaozukum
31,Joseph Lalthankhuma,josephlalthankhuma
32,Akshay Karnewar,akshaykarnewar
33,Manisankar Murasingh,manisankarmurasingh
34,Tanmay Agarwal,tanmayagarwal
35,Ashwin Hebbar,ashwinhebbar
36,Parth Sahani,parthsahani
37,P Chopra,pchopra
38,CV Milind,cvmilind
39,GH Vihari,ghvihari


In [63]:
# ==========================================
# PRESSURE-X: APPLY DOB BATCH 2
# ==========================================

batch2_dobs = {
    "joshuaozukum": ("2000-08-04", "https://www.cricbuzz.com/profiles/18672/joshua-ozukum", "Verified"),
    "josephlalthankhuma": ("2000-09-26", "https://www.cricbuzz.com/profiles/21708/joseph-lalthankhuma", "Verified"),
    "akshaykarnewar": ("1992-10-12", "https://www.bcci.tv/players/d4422087-56e9-484d-a1da-494c20b85d1a/akshay-karnewar/biography", "Verified"),
    "manisankarmurasingh": ("1993-01-01", "https://www.bcci.tv/players/3ccef381-507b-4f4f-bebe-869f33a08b64/manisankar-murasingh", "Verified"),
    "tanmayagarwal": ("1995-05-03", "https://www.cricbuzz.com/profiles/10258/tanmay-agarwal", "Verified"),
    "ashwinhebbar": ("1995-11-15", "https://www.cricbuzz.com/profiles/10254/ashwin-hebbar", "Verified"),
    "parthsahani": ("1993-03-09", "https://www.cricbuzz.com/profiles/10918/parth-sahani", "Verified"),
    "cv milind": ("1994-09-04", "https://www.cricbuzz.com/profiles/9281/chama-v-milind", "Verified"),
    "ghvihari": ("1993-10-13", "https://www.cricbuzz.com/profiles/8424/vihari-c", "Verified"),
    "ajaysarkar": ("1997-03-10", "https://www.cricbuzz.com/profiles/11015/ajay-sarkar", "Verified"),
    "bishworjitkonthoujam": ("1996-02-03", "https://www.cricbuzz.com/profiles/14136/bishworjit-konthoujam", "Cross-checked"),
    "aksbains": ("1995-12-16", "https://www.cricbuzz.com/profiles/9298/ankush-bains", "Cross-checked"),
    "lalhruairalte": ("1992-03-10", "https://www.cricbuzz.com/profiles/14065/hm-ralte", "Verified"),
    "kanwarabhinay": ("1991-06-16", "https://en.wikipedia.org/wiki/Kanwar_Abhinay", "Cross-checked"),
    "rajatdey": ("1996-12-31", "https://www.bcci.tv/players/afafa3dd-d77f-467c-b76a-8067bb5d0190/rajat-dey/biography", "Verified"),
    "sudipchatterjee": ("1991-11-11", "https://www.cricbuzz.com/profiles/8992/sudip-chatterjee", "Verified"),
    "akshaywadkar": ("1994-07-09", "https://www.bcci.tv/players/9f685200-4742-4d84-aa51-715f5580c5c6/akshay-wadkar", "Verified"),
    "tilakvarma": ("2002-11-08", "https://live-bccitv.epicon.in/players/993/tilak-varma", "Verified"),
    "vishvarajjadeja": ("1998-07-19", "https://www.cricbuzz.com/profiles/13493/vishvaraj-jadeja", "Verified"),
    "rishavdas": ("1989-12-16", "https://www.bcci.tv/players/6a960d65-229c-4965-9f8e-7c483c3d9e9e/rishav-das/biography", "Verified")
}

# Normalize key for CV Milind
batch2_dobs["cv milind"] = batch2_dobs.pop("cv milind") if "cv milind" in batch2_dobs else batch2_dobs.get("cvmilind")

# Apply records using exact Player_Key values
for key, (dob, source, status) in batch2_dobs.items():
    mask = unique_profiles["Player_Key"].eq(key)

    if mask.sum() == 1:
        unique_profiles.loc[mask, "Date_of_Birth"] = dob
        unique_profiles.loc[mask, "DOB_Source"] = source
        unique_profiles.loc[mask, "Age_Verification"] = status
    else:
        print(f"Check key: {key} | Matches: {mask.sum()}")

# Calculate age as of 16 November 2026
reference_date = pd.Timestamp("2026-11-16")
dob_dates = pd.to_datetime(
    unique_profiles["Date_of_Birth"],
    errors="coerce"
)

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob_dates.dt.year
    - (
        (reference_date.month < dob_dates.dt.month)
        | (
            (reference_date.month == dob_dates.dt.month)
            & (reference_date.day < dob_dates.dt.day)
        )
    )
)

# Save
unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:",
      unique_profiles["Date_of_Birth"].ne("").sum())

print("DOB records pending:",
      unique_profiles["Date_of_Birth"].eq("").sum())

display(
    unique_profiles.loc[30:59, [
        "Player",
        "Player_Key",
        "Date_of_Birth",
        "Age_as_of_2026_11_16",
        "DOB_Source",
        "Age_Verification"
    ]]
)

Check key: cv milind | Matches: 0
DOB records entered: 25
DOB records pending: 339


,Player,Player_Key,Date_of_Birth,Age_as_of_2026_11_16,DOB_Source,Age_Verification
30,Joshua Ozukum,joshuaozukum,2000-08-04,26.00,https://www.cricbuzz.com/profiles/18672/joshua...,Verified
31,Joseph Lalthankhuma,josephlalthankhuma,2000-09-26,26.00,https://www.cricbuzz.com/profiles/21708/joseph...,Verified
32,Akshay Karnewar,akshaykarnewar,1992-10-12,34.00,https://www.bcci.tv/players/d4422087-56e9-484d...,Verified
33,Manisankar Murasingh,manisankarmurasingh,1993-01-01,33.00,https://www.bcci.tv/players/3ccef381-507b-4f4f...,Verified
34,Tanmay Agarwal,tanmayagarwal,1995-05-03,31.00,https://www.cricbuzz.com/profiles/10258/tanmay...,Verified
35,Ashwin Hebbar,ashwinhebbar,1995-11-15,31.00,https://www.cricbuzz.com/profiles/10254/ashwin...,Verified
36,Parth Sahani,parthsahani,1993-03-09,33.00,https://www.cricbuzz.com/profiles/10918/parth-...,Verified
37,P Chopra,pchopra,,NaN,,Pending
38,CV Milind,cvmilind,,NaN,,Pending
39,GH Vihari,ghvihari,1993-10-13,33.00,https://www.cricbuzz.com/profiles/8424/vihari-c,Verified


In [64]:
# Correct CV Milind's DOB entry
mask = unique_profiles["Player_Key"].eq("cvmilind")

unique_profiles.loc[mask, "Date_of_Birth"] = "1994-09-04"
unique_profiles.loc[mask, "DOB_Source"] = (
    "https://www.cricbuzz.com/profiles/9281/chama-v-milind"
)
unique_profiles.loc[mask, "Age_Verification"] = "Verified"

# Recalculate ages
reference_date = pd.Timestamp("2026-11-16")
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (reference_date.month < dob.dt.month)
        | (
            (reference_date.month == dob.dt.month)
            & (reference_date.day < dob.dt.day)
        )
    )
)

# Save updated queue
unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", unique_profiles["Date_of_Birth"].ne("").sum())
print("DOB records pending:", unique_profiles["Date_of_Birth"].eq("").sum())

DOB records entered: 26
DOB records pending: 338


In [65]:
display(
    unique_profiles.loc[60:89, [
        "DOB_Verification_Order",
        "Player",
        "Roles",
        "latest_team",
        "IPL_Exposure",
        "Date_of_Birth",
        "Age_Verification"
    ]]
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth,Age_Verification
60,61,AV Rajput,"Batter, Bowler",Baroda,No appearance found in available IPL data,,Pending
61,62,Bikramkumar Das,Batter,Tripura,No appearance found in available IPL data,,Pending
62,63,Abhishek Goswami,Batter,Uttar Pradesh,No appearance found in available IPL data,,Pending
63,64,AA Sarwate,Bowler,Vidarbha,No appearance found in available IPL data,,Pending
64,65,AP Tare,Batter,Uttarakhand,No appearance found in available IPL data,,Pending
65,66,Lakshay Garg,Bowler,Goa,No appearance found in available IPL data,,Pending
66,67,Pankaj Jaswal,Bowler,Himachal Pradesh,No appearance found in available IPL data,,Pending
67,68,K V Sasikanth,Bowler,Andhra,No appearance found in available IPL data,,Pending
68,69,Amandeep Khare,Batter,Chhattisgarh,No appearance found in available IPL data,,Pending
69,70,Ajay Mandal,"Batter, Bowler",Chhattisgarh,No appearance found in available IPL data,,Pending


In [67]:
# Correct Siddhesh Lad's player key
lad_matches = unique_profiles[
    unique_profiles["Player"].str.contains(
        "Siddhesh Lad", case=False, na=False
    )
]

display(lad_matches[["Player", "Player_Key"]])

# Apply Siddhesh Lad DOB using the actual key
if len(lad_matches) == 1:
    idx = lad_matches.index[0]

    unique_profiles.loc[idx, "Date_of_Birth"] = "1992-05-23"
    unique_profiles.loc[idx, "DOB_Source"] = (
        "https://www.bcci.tv/players/dd9fc38a-6c4e-479b-906a-299c703b5511/siddhesh-lad/news"
    )
    unique_profiles.loc[idx, "Age_Verification"] = "Verified"

# Convert DOB column
unique_profiles["Date_of_Birth"] = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

# Calculate age as of 16 November 2026
reference_date = pd.Timestamp("2026-11-16")

dob = unique_profiles["Date_of_Birth"]

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year
    - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

# Save updated queue
unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())
print("Saved successfully.")

,Player,Player_Key


DOB records entered: 48
DOB records pending: 316
Saved successfully.


In [68]:
display(
    unique_profiles.loc[90:119, [
        "DOB_Verification_Order",
        "Player",
        "Roles",
        "latest_team",
        "IPL_Exposure",
        "Date_of_Birth",
        "Age_Verification"
    ]]
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth,Age_Verification
90,91,Darshan Misal,Batter,Goa,No appearance found in available IPL data,NaT,Pending
91,92,SP Goswami,Batter,Mizoram,No appearance found in available IPL data,NaT,Pending
92,93,Ganesh Satish,Batter,Tripura,No appearance found in available IPL data,NaT,Pending
93,94,Rajjakuddin Ahmed,"Batter, Bowler",Assam,No appearance found in available IPL data,NaT,Pending
94,95,Hari Nishaanth,Batter,Tamil Nadu,No appearance found in available IPL data,NaT,Pending
95,96,Shashank Chandrakar,Batter,Chhattisgarh,No appearance found in available IPL data,NaT,Pending
96,97,Damodaran Rohit,Batter,Puducherry,No appearance found in available IPL data,NaT,Pending
97,98,KC Cariappa,Bowler,Mizoram,No appearance found in available IPL data,NaT,Pending
98,99,T Kohli,Batter,Mizoram,No appearance found in available IPL data,NaT,Pending
99,100,Ashutosh Aman,Bowler,Bihar,No appearance found in available IPL data,NaT,Pending


In [69]:
dob_batch_4 = {
    "Darshan Misal": ("1993-09-11", "https://www.bcci.tv/players/f2863596-7976-49f4-b0ea-9511f72c5db8/darshan-misal/biography"),
    "Rajjakuddin Ahmed": ("1995-09-02", "https://www.cricbuzz.com/profiles/13178/rajjakuddin"),
    "Hari Nishaanth": ("1996-08-16", "https://www.cricbuzz.com/profiles/11623/hari-nishaanth"),
    "Shashank Chandrakar": ("1994-05-13", "https://www.cricbuzz.com/profiles/13490/shashank-chandraker"),
    "Damodaran Rohit": ("1992-05-28", "https://www.cricbuzz.com/profiles/14613/damodaran-rohit-c"),
    "KC Cariappa": ("1994-04-13", "https://www.cricx.com/player/kc-cariappa/"),
    "Ashutosh Aman": ("1986-05-19", "https://www.cricbuzz.com/profiles/13579/ashutosh-aman"),
    "Roosh Kalaria": ("1993-01-16", "https://www.cricbuzz.com/profiles/8353/roosh-kalaria"),
    "M Prasidh Krishna": ("1996-02-19", "https://www.bcci.tv/players/prasidh-krishna"),
    "Harishankar Reddy": ("1998-06-02", "https://www.cricbuzz.com/profiles/13521/harishankar-reddy"),
    "Subhransu Senapati": ("1996-12-30", "https://www.bcci.tv/players/593128d8-4235-45b9-a171-8c583c8ad5c0/subhranshu-senapati"),
    "Chetan Bist": ("1989-09-03", "https://www.cricbuzz.com/profiles/9666/chetan-bist-wk"),
    "Sagar Udeshi": ("1986-10-14", "https://www.cricbuzz.com/profiles/15484/sagar-udeshi"),
    "Nitesh Sedai": ("1998-02-08", "https://www.cricbuzz.com/profiles/15733/nitesh-sedai"),
    "Mayank Mishra": ("1990-10-09", "https://www.cricbuzz.com/profiles/14108/mayank-mishra"),
    "Nagaho Chishi": ("1997-11-12", "https://www.bcci.tv/players/68915fe7-d5bf-4b40-a5fe-48875d86b54c/nagaho-chishi"),
    "Nikhil Gangta": ("1992-09-01", "https://www.cricbuzz.com/profiles/8996/nikhil-gangta"),
    "Roshan Alam": ("1995-04-20", "https://www.cricbuzz.com/profiles/15483/roshan-alam"),
    "Abhijeet Saket": ("1995-08-03", "https://www.cricbuzz.com/profiles/14321/abhijeet-saket")
}

for player, (dob, source) in dob_batch_4.items():
    mask = unique_profiles["Player"].str.strip().str.casefold().eq(
        player.casefold()
    )

    if mask.sum() == 1:
        unique_profiles.loc[mask, "Date_of_Birth"] = dob
        unique_profiles.loc[mask, "DOB_Source"] = source
        unique_profiles.loc[mask, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

# Recalculate ages
unique_profiles["Date_of_Birth"] = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

reference_date = pd.Timestamp("2026-11-16")
dob = unique_profiles["Date_of_Birth"]

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 67
DOB records pending: 297


In [70]:
display(
    unique_profiles.loc[120:149, [
        "DOB_Verification_Order",
        "Player",
        "Roles",
        "latest_team",
        "IPL_Exposure",
        "Date_of_Birth",
        "Age_Verification"
    ]]
)


,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth,Age_Verification
120,121,Bipin Saurabh,Batter,Bihar,No appearance found in available IPL data,NaT,Pending
121,122,SS Agarwal,Bowler,Chhattisgarh,No appearance found in available IPL data,NaT,Pending
122,123,Anuj Raj,Bowler,Bihar,No appearance found in available IPL data,NaT,Pending
123,124,Ishaan Gadekar,Batter,Goa,No appearance found in available IPL data,NaT,Pending
124,125,Akshay Wakhare,Bowler,Vidarbha,No appearance found in available IPL data,NaT,Pending
125,126,Pallav Kumar Das,Batter,Assam,No appearance found in available IPL data,NaT,Pending
126,127,KB Arun Karthik,Batter,Puducherry,No appearance found in available IPL data,NaT,Pending
127,128,A Choudhary,Bowler,Rajasthan,No appearance found in available IPL data,NaT,Pending
128,129,Cheepurapalli Stephen,Bowler,Andhra,No appearance found in available IPL data,NaT,Pending
129,130,Snehal Kauthankar,Batter,Goa,No appearance found in available IPL data,NaT,Pending


In [71]:
dob_batch_5 = {
    "Bipin Saurabh": ("1999-11-20", "https://www.cricbuzz.com/profiles/21671/bipin-saurabh"),
    "Pallav Kumar Das": ("1990-01-17", "https://www.cricbuzz.com/profiles/8760/pallavkumar-das"),
    "KB Arun Karthik": ("1986-02-15", "https://www.cricbuzz.com/profiles/1672/arun-karthik-c"),
    "Cheepurapalli Stephen": ("1993-12-03", "https://www.cricbuzz.com/profiles/9620/cheepurapalli-stephen"),
    "Snehal Kauthankar": ("1995-10-19", "https://en.wikipedia.org/wiki/Snehal_Kauthankar"),
    "Sumit Ruikar": ("1990-06-06", "https://www.cricbuzz.com/profiles/8720/sumit-ruikar"),
    "Writtick Chatterjee": ("1992-09-28", "https://www.cricbuzz.com/profiles/9717/writtick-chatterjee"),
    "Akash Choudhary": ("1998-11-28", "https://cricketarchive.com/Archive/Players/2272/2272376/2272376.html"),
    "Mangal Mahrour": ("1992-04-01", "https://cricketarchive.com/Archive/Players/1135/1135191/1135191.html"),
    "Techi Doria": ("1994-03-02", "https://www.cricbuzz.com/profiles/14073/techi-doria-c"),
    "AL Menaria": ("1990-10-29", "https://cricketarchive.com/Archive/Players/256/256798/256798.html"),
    "Kanishk Seth": ("1997-11-04", "https://www.cricbuzz.com/profiles/10955/kanishk-seth"),
    "Unnikrishnan Manukrishnan": ("1988-10-04", "https://cricketarchive.com/Archive/Players/166/166137/166137.html"),
    "Kilco R Marak": ("1992-07-27", "https://www.cricbuzz.com/profiles/15735/kilco-marak"),
    "KR Kakade": ("1995-07-25", "https://cricketarchive.com/Archive/Players/923/923485/923485.html"),
    "Rohan Kunnummal": ("1998-05-10", "https://en.wikipedia.org/wiki/Rohan_Kunnummal"),
    "Sankar Paul": ("2000-11-15", "https://www.cricbuzz.com/profiles/18727/sankar-paul"),
    "Pradipta Pramanik": ("1998-10-08", "https://www.cricbuzz.com/profiles/10880/pradipta-pramanik"),
    "Khrievitso Kense": ("2004-03-06", "https://www.bcci.tv/players/235c152c-9fcb-484e-98fe-9d945da623aa/khrievitso-kense/news"),
    "Amit Ali": ("2002-10-26", "https://www.cricbuzz.com/profiles/19338/amit-ali"),
    "Ramachandran Ragupathy": ("1996-05-12", "https://www.cricbuzz.com/profiles/17224/ragupathy-wk"),
    "Amit Yadav": ("1989-10-10", "https://cricketarchive.com/Archive/Players/167/167615/167615.html")
}

for player, (dob, source) in dob_batch_5.items():
    mask = unique_profiles["Player"].str.strip().str.casefold().eq(
        player.casefold()
    )

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = source
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

# Recalculate ages
unique_profiles["Date_of_Birth"] = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

reference_date = pd.Timestamp("2026-11-16")
dob = unique_profiles["Date_of_Birth"]

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 89
DOB records pending: 275


In [72]:
display(
    unique_profiles.loc[150:179, [
        "DOB_Verification_Order",
        "Player",
        "Roles",
        "latest_team",
        "IPL_Exposure",
        "Date_of_Birth",
        "Age_Verification"
    ]]
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth,Age_Verification
150,151,Sachin Baby,Batter,Kerala,No appearance found in available IPL data,NaT,Pending
151,152,M Mohammed,Bowler,Tamil Nadu,No appearance found in available IPL data,NaT,Pending
152,153,Chaitanya Bishnoi,Batter,Haryana,No appearance found in available IPL data,NaT,Pending
153,154,P Dogra,Batter,Puducherry,No appearance found in available IPL data,NaT,Pending
154,155,B Aparajith,"Batter, Bowler",Tamil Nadu,No appearance found in available IPL data,NaT,Pending
155,156,Gurkeerat Singh,Batter,Punjab,No appearance found in available IPL data,NaT,Pending
156,157,Mohammed Azharuddeen,Batter,Kerala,No appearance found in available IPL data,NaT,Pending
157,158,R Sanjay Yadav,"Batter, Bowler",Tamil Nadu,No appearance found in available IPL data,NaT,Pending
158,159,Aditya Kaushik,Batter,Goa,No appearance found in available IPL data,NaT,Pending
159,160,S Midhun,Bowler,Kerala,No appearance found in available IPL data,NaT,Pending


In [73]:
dob_batch_6 = {
    "Sachin Baby": ("1988-12-18", "https://www.bcci.tv/players/6048d888-fff1-4fdf-9492-9773c4de15c7/sachin-baby"),
    "Chaitanya Bishnoi": ("1994-08-25", "https://www.cricbuzz.com/profiles/10672/chaitanya-bishnoi"),
    "P Dogra": ("1984-11-19", "https://www.bcci.tv/players/77bcef94-f955-402e-864f-f452877a635a/paras-dogra"),
    "B Aparajith": ("1994-07-08", "https://www.cricbuzz.com/profiles/8198/aparajith"),
    "Mohammed Azharuddeen": ("1994-03-22", "https://www.bcci.tv/players/b64f95eb-0fd1-4aa6-88bb-a364a30f9bf1/mohammed-azharuddeen"),
    "Aditya Kaushik": ("1991-09-10", "https://www.cricbuzz.com/profiles/10379/aditya-kaushik"),
    "Jalaj Saxena": ("1986-12-15", "https://www.cricbuzz.com/profiles/8107/JalajSaxena"),
    "IC Porel": ("1998-09-05", "https://www.cricbuzz.com/profiles/11404/ishan-porel"),
    "Ravi Kiran": ("1991-02-01", "https://www.cricbuzz.com/profiles/8936/ravi-kiran"),
    "Ishaan Gadekar": ("1997-08-31", "https://www.cricbuzz.com/profiles/19223/ishaan-gadekar"),
    "Akshay Wakhare": ("1985-10-03", "https://www.cricbuzz.com/profiles/8949/akshay-wakhare"),
    "Lamabam Singh": ("2002-03-10", "https://www.cricbuzz.com/profiles/1449368/lamabam-singh"),
    "Abhimanyu Rana": ("1991-09-21", "https://www.bcci.tv/players/551a1be9-6c00-44ad-bea6-c4fd8886ab54/abhimanyu-rana"),
    "Shivam Bhambri": ("1995-09-30", "https://www.cricbuzz.com/profiles/12359/shivam-bhambri"),
    "Dhrushant Soni": ("1995-10-09", "https://www.cricbuzz.com/profiles/15616/dhrushant-soni"),
    "Bhagmender Lather": ("1997-06-18", "https://m.cricbuzz.com/profiles/19003/bhagmender-lather"),
    "AS Rajpoot": ("1993-12-04", "https://www.cricbuzz.com/profiles/8924/rajpoot"),
    "SS Tiwary": ("1989-12-30", "https://www.mumbaiindians.com/players/4285-saurabh-tiwary-profile")
}

for player, (dob, source) in dob_batch_6.items():
    mask = unique_profiles["Player"].str.strip().str.casefold().eq(
        player.casefold()
    )

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = source
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

# Recalculate ages
unique_profiles["Date_of_Birth"] = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

reference_date = pd.Timestamp("2026-11-16")
dob = unique_profiles["Date_of_Birth"]

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

Jalaj Saxena: 0 matching records. Manual check required.
DOB records entered: 106
DOB records pending: 258


In [74]:
# Find the exact player record
mask = unique_profiles["Player"].str.contains(
    "Jalaj", case=False, na=False
)

display(
    unique_profiles.loc[mask, ["Player", "Player_Key"]]
)

# Apply DOB to the exact matching record
if mask.sum() == 1:
    idx = unique_profiles.index[mask][0]

    unique_profiles.loc[idx, "Date_of_Birth"] = "1986-12-15"
    unique_profiles.loc[idx, "DOB_Source"] = (
        "https://www.cricbuzz.com/profiles/8107/JalajSaxena"
    )
    unique_profiles.loc[idx, "Age_Verification"] = "Verified"
else:
    print("Multiple or no matches. Manual verification required.")

# Recalculate age
dob = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

,Player,Player_Key
160,Jalaj S Saxena,jalajssaxena


DOB records entered: 107
DOB records pending: 257


In [75]:
display(
    unique_profiles.loc[
        180:209,
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure",
            "Date_of_Birth"
        ]
    ].reset_index(drop=True)
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth
0,181,S Nadeem,Bowler,Jharkhand,No appearance found in available IPL data,NaT
1,182,Ankit Lamba,Batter,Rajasthan,No appearance found in available IPL data,NaT
2,183,Jay Bista,Batter,Uttarakhand,No appearance found in available IPL data,NaT
3,184,Parvez Rasool,Bowler,Jammu & Kashmir,No appearance found in available IPL data,NaT
4,185,Diwesh Pathania,Bowler,Services,No appearance found in available IPL data,NaT
5,186,Rohan Kadam,Batter,Karnataka,No appearance found in available IPL data,NaT
6,187,Shubham Ranjane,Batter,Goa,No appearance found in available IPL data,NaT
7,188,Arpit Vasavada,Batter,Saurashtra,No appearance found in available IPL data,NaT
8,189,DB Ravi Teja,Batter,Meghalaya,No appearance found in available IPL data,NaT
9,190,KD Karthik,Batter,Tamil Nadu,No appearance found in available IPL data,NaT


In [76]:
dob_batch_7 = {
    "S Nadeem": "1989-08-12",
    "Ankit Lamba": "1991-12-03",
    "Jay Bista": "1995-12-23",
    "Parvez Rasool": "1989-02-13",
    "Diwesh Pathania": "1989-06-22",
    "Rohan Kadam": "1994-07-05",
    "Shubham Ranjane": "1994-03-26",
    "Arpit Vasavada": "1988-10-28",
    "KD Karthik": "1985-06-01",
    "Ravi Chauhan": "1993-09-17",
    "Mihir Hirwani": "1994-05-14",
    "Punit Bisht": "1986-06-15",
    "Rajesh Dhuper": "1995-07-08"
}

for player, dob in dob_batch_7.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Verified cricket profile"
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

# Recalculate age
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")
reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 120
DOB records pending: 244


In [77]:
display(
    unique_profiles.loc[
        210:239,
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure",
            "Date_of_Birth"
        ]
    ].reset_index(drop=True)
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth
0,211,Yash Nahar,Batter,Maharashtra,No appearance found in available IPL data,NaT
1,212,AA Barot,Batter,Saurashtra,No appearance found in available IPL data,NaT
2,213,Kamsha Yangfo,Batter,Arunachal Pradesh,No appearance found in available IPL data,NaT
3,214,Mujtaba Yousuf,Bowler,Jammu & Kashmir,No appearance found in available IPL data,NaT
4,215,Saahil Jain,Batter,Assam,No appearance found in available IPL data,NaT
5,216,Samar Quadri,Bowler,Bihar,No appearance found in available IPL data,NaT
6,217,Varun Choudhary,Bowler,Services,No appearance found in available IPL data,NaT
7,218,Varun Sood,Batter,Sikkim,No appearance found in available IPL data,NaT
8,219,SK Patel,Batter,Baroda,No appearance found in available IPL data,NaT
9,220,Siddhesh Wath,Batter,Vidarbha,No appearance found in available IPL data,NaT


In [78]:
dob_batch_8 = {
    "Yash Nahar": "1994-10-10",
    "Kamsha Yangfo": "1992-11-16",
    "Mujtaba Yousuf": "2002-06-07",
    "Saahil Jain": "1998-10-22",
    "Samar Quadri": "1989-07-27",
    "Varun Choudhary": "1993-10-26",
    "Siddhesh Wath": "1997-07-03",
    "Chirag Khurana": "1992-11-03",
    "Nitin Tanwar": "1996-10-09",
    "Sarul Kanwar": "1987-11-08",
    "Pritam Das": "1988-10-16",
    "Divyang Hinganekar": "1993-10-14",
    "Himalay Agarwal": "1993-10-09",
    "Ankit Kaushik": "1991-09-05",
    "Arun Chaprana": "1994-01-05",
    "Aniruddha Joshi": "1987-11-07",
    "S Kaul": "1990-05-19"
}

for player, dob in dob_batch_8.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Verified cricket profile"
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")
reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 137
DOB records pending: 227


In [79]:
display(
    unique_profiles.loc[
        240:269,
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure",
            "Date_of_Birth"
        ]
    ].reset_index(drop=True)
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth
0,241,YV Dhull,Batter,Delhi,Pending identity verification,NaT
1,242,JD Unadkat,Bowler,Saurashtra,Pending identity verification,NaT
2,243,RM Patidar,Batter,Madhya Pradesh,Pending identity verification,NaT
3,244,SZ Mulani,Bowler,Mumbai,Pending identity verification,NaT
4,245,TU Deshpande,Bowler,Mumbai,Pending identity verification,NaT
5,246,PP Shaw,Batter,Mumbai,Pending identity verification,NaT
6,247,YBK Jaiswal,Batter,Mumbai,Pending identity verification,NaT
7,248,PP Chawla,"Batter, Bowler",Gujarat,Pending identity verification,NaT
8,249,KS Sharma,Batter,Uttar Pradesh,Pending identity verification,NaT
9,250,KV Sharma,"Batter, Bowler",Railways,Pending identity verification,NaT


In [80]:
dob_batch_9 = {
    "YV Dhull": "2002-11-11",
    "JD Unadkat": "1991-10-18",
    "RM Patidar": "1993-06-01",
    "SZ Mulani": "1997-03-13",
    "TU Deshpande": "1995-05-15",
    "PP Shaw": "1999-11-09",
    "YBK Jaiswal": "2001-12-28",
    "PP Chawla": "1988-12-24",
    "KS Sharma": "1998-10-31",
    "KV Sharma": "1987-10-23",
    "SP Jackson": "1986-09-27",
    "VR Iyer": "1994-12-25",
    "RK Singh": "1997-10-12",
    "Harpreet Singh": "1991-08-11",
    "SS Iyer": "1994-12-06",
    "Pratham Singh": "1992-08-31",
    "I Sharma": "1988-09-02",
    "SN Khan": "1997-10-22",
    "Shiva Singh": "1999-10-16",
    "JM Sharma": "1993-10-22",
    "PN Mankad": "1994-04-23",
    "R Parag": "2001-11-10",
    "DG Nalkande": "1998-10-04",
    "Atharva Taide": "2000-04-26",
    "Yuvraj Singh": "1998-08-22",
    "Upendra Yadav": "1996-10-08",
    "RV Patel": "1995-09-28",
    "B Kumar": "1990-02-05",
    "SB Dubey": "1994-08-27"
}

for player, dob in dob_batch_9.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket profile; identity matched"
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

# Recalculate age
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")
reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 166
DOB records pending: 198


In [81]:
display(
    unique_profiles.loc[
        270:299,
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure",
            "Date_of_Birth"
        ]
    ].reset_index(drop=True)
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth
0,271,R Dhawan,"Batter, Bowler",Himachal Pradesh,Pending identity verification,NaT
1,272,RK Bhui,Batter,Andhra,Pending identity verification,NaT
2,273,Kangabam Singh,Batter,Manipur,Pending identity verification,NaT
3,274,VG Arora,Bowler,Himachal Pradesh,Pending identity verification,NaT
4,275,UT Yadav,Bowler,Vidarbha,Pending identity verification,NaT
5,276,MA Agarwal,Batter,Karnataka,Pending identity verification,NaT
6,277,J Yadav,Bowler,Haryana,Pending identity verification,NaT
7,278,R Tewatia,Batter,Haryana,Pending identity verification,NaT
8,279,MK Pandey,Batter,Karnataka,Pending identity verification,NaT
9,280,Himanshu Rana,Batter,Haryana,Pending identity verification,NaT


In [82]:
dob_batch_10 = {
    "R Dhawan": "1990-02-19",
    "RK Bhui": "1996-09-29",
    "VG Arora": "1997-12-14",
    "UT Yadav": "1987-10-25",
    "MA Agarwal": "1991-02-16",
    "J Yadav": "1990-01-22",
    "R Tewatia": "1993-05-20",
    "MK Pandey": "1989-09-10",
    "Himanshu Rana": "1998-10-01",
    "K Gowtham": "1988-10-20",
    "AT Rayudu": "1985-09-23",
    "A Manohar": "1994-09-16",
    "Het Patel": "1998-10-13",
    "YS Chahal": "1990-07-23",
    "HV Patel": "1990-11-23",
    "D Padikkal": "2000-07-07",
    "WP Saha": "1984-10-24",
    "Ankit Sharma": "1991-04-20",
    "MM Sharma": "1988-09-18",
    "S Sandeep Warrier": "1991-04-04",
    "MK Lomror": "1999-11-16",
    "SV Samson": "1994-11-11"
}

for player, dob in dob_batch_10.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket profile; identity matched"
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

# Recalculate age
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")
reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 188
DOB records pending: 176


In [83]:
display(
    unique_profiles.loc[
        300:329,
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure",
            "Date_of_Birth"
        ]
    ].reset_index(drop=True)
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth
0,301,V Shankar,Batter,Tamil Nadu,Pending identity verification,NaT
1,302,Babul Kumar,Batter,Bihar,Pending identity verification,NaT
2,303,C Sakariya,Bowler,Saurashtra,Pending identity verification,NaT
3,304,RD Chahar,Bowler,Rajasthan,Pending identity verification,NaT
4,305,KL Nagarkoti,Bowler,Rajasthan,Pending identity verification,NaT
5,306,KK Nair,Batter,Vidarbha,Pending identity verification,NaT
6,307,J Suchith,Bowler,Karnataka,Pending identity verification,NaT
7,308,N Jagadeesan,Batter,Tamil Nadu,Pending identity verification,NaT
8,309,M Vohra,Batter,Chandigarh,Pending identity verification,NaT
9,310,RP Sharma,Batter,Haryana,Pending identity verification,NaT


In [84]:
dob_batch_11 = {
    "V Shankar": "1991-01-26",
    "Babul Kumar": "1993-01-12",
    "C Sakariya": "1998-02-28",
    "RD Chahar": "1999-08-04",
    "KL Nagarkoti": "1999-12-28",
    "KK Nair": "1991-12-06",
    "J Suchith": "1994-01-16",
    "N Jagadeesan": "1995-12-24",
    "M Vohra": "1993-07-18",
    "PJ Sangwan": "1990-11-05",
    "S Dhawan": "1985-12-05",
    "M Ashwin": "1990-09-08",
    "RD Gaikwad": "1997-01-31",
    "RV Uthappa": "1985-11-11",
    "KM Jadhav": "1985-03-26",
    "Sonu Yadav": "1999-11-11",
    "Abhishek Sharma": "2000-09-04",
    "Harpreet Brar": "1995-09-16",
    "Anmolpreet Singh": "1998-03-28",
    "Sanvir Singh": "1996-10-12",
    "Mandeep Singh": "1991-12-18",
    "Ramandeep Singh": "1997-04-13",
    "Himmat Singh": "1996-11-08",
    "Anuj Rawat": "1999-10-17",
    "Urvil Patel": "1998-10-17",
    "Navdeep Saini": "1992-11-23"
}

for player, dob in dob_batch_11.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket profile; identity matched"
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

# Recalculate age
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")
reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 214
DOB records pending: 150


In [85]:
display(
    unique_profiles.loc[
        330:359,
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure",
            "Date_of_Birth"
        ]
    ].reset_index(drop=True)
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth
0,331,Saurav Chauhan,Batter,Gujarat,Confirmed IPL appearance,NaT
1,332,Lalit Yadav,"Batter, Bowler",Delhi,Confirmed IPL appearance,NaT
2,333,Yash Dayal,Bowler,Uttar Pradesh,Confirmed IPL appearance,NaT
3,334,Arshdeep Singh,Bowler,Punjab,Confirmed IPL appearance,NaT
4,335,Sameer Rizvi,Batter,Uttar Pradesh,Confirmed IPL appearance,NaT
5,336,Sumit Kumar,Bowler,"Haryana, Nagaland",Confirmed IPL appearance,NaT
6,337,Yash Thakur,Bowler,Vidarbha,Confirmed IPL appearance,NaT
7,338,Avesh Khan,Bowler,Madhya Pradesh,Confirmed IPL appearance,NaT
8,339,Ravi Bishnoi,Bowler,Gujarat,Confirmed IPL appearance,NaT
9,340,Priyansh Arya,Batter,Delhi,Confirmed IPL appearance,NaT


In [86]:
dob_batch_12 = {
    "Saurav Chauhan": "2000-05-27",
    "Lalit Yadav": "1997-01-03",
    "Yash Dayal": "1997-12-13",
    "Arshdeep Singh": "1999-02-05",
    "Sameer Rizvi": "2003-12-06",
    "Sumit Kumar": "1995-12-12",
    "Yash Thakur": "1998-12-28",
    "Avesh Khan": "1996-12-13",
    "Ravi Bishnoi": "2000-09-05",
    "Priyansh Arya": "2001-09-18",
    "Suyash Sharma": "2003-05-15",
    "KS Bharat": "1993-10-03",
    "Mayank Dagar": "1996-11-11",
    "Shubman Gill": "1999-09-08",
    "Arjun Tendulkar": "1999-09-24",
    "Swapnil Singh": "1991-01-22",
    "Shashank Singh": "1991-11-21",
    "Akash Madhwal": "1993-11-25",
    "Yudhvir Singh": "1997-09-13",
    "Umran Malik": "1999-11-22",
    "Shahbaz Ahmed": "1994-12-11",
    "Mohsin Khan": "1998-07-15",
    "M Shahrukh Khan": "1995-05-27",
    "Shivam Mavi": "1998-11-26",
    "Vishnu Vinod": "1993-12-02",
    "KM Asif": "1993-07-24",
    "Abdul Samad": "2001-10-28",
    "Basil Thampi": "1993-09-11",
    "Sandeep Sharma": "1993-05-18",
    "Akash Deep": "1996-12-15"
}

for player, dob in dob_batch_12.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket player profile"
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

# Recalculate age
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")
reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 244
DOB records pending: 120


In [87]:
display(
    unique_profiles.loc[
        360:363,
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure",
            "Date_of_Birth"
        ]
    ].reset_index(drop=True)
)

,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure,Date_of_Birth
0,361,Mukesh Kumar,Bowler,Bengal,Confirmed IPL appearance,NaT
1,362,Simarjeet Singh,Bowler,Delhi,Confirmed IPL appearance,NaT
2,363,Auqib Nabi,Bowler,Jammu & Kashmir,Confirmed IPL appearance,NaT
3,364,Mukesh Choudhary,Bowler,Maharashtra,Confirmed IPL appearance,NaT


In [88]:
dob_batch_13 = {
    "Mukesh Kumar": "1993-10-12",
    "Simarjeet Singh": "1998-01-17",
    "Auqib Nabi": "1996-11-04",
    "Mukesh Choudhary": "1996-07-06"
}

for player, dob in dob_batch_13.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Verified player profile"
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"{player}: {mask.sum()} matching records. Manual check required.")

dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")
reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 248
DOB records pending: 116


In [89]:
pending_dobs = unique_profiles[
    unique_profiles["Date_of_Birth"].isna()
].copy()

pending_dobs = pending_dobs.sort_values(
    "DOB_Verification_Order"
)

pending_dobs.to_csv(
    DATA_DIR / "PRESSURE_X_Pending_DOB_Verification.csv",
    index=False
)

print("Pending players:", len(pending_dobs))

display(
    pending_dobs[
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure"
        ]
    ].reset_index(drop=True)
)

Pending players: 116


,DOB_Verification_Order,Player,Roles,latest_team,IPL_Exposure
0,2,A Sheth,"Batter, Bowler",Baroda,No appearance found in available IPL data
1,3,LI Meriwala,Bowler,Baroda,No appearance found in available IPL data
2,4,Vishnu Solanki,Batter,Baroda,No appearance found in available IPL data
3,5,Dharmendrasinh Jadeja,Bowler,Saurashtra,No appearance found in available IPL data
4,6,Chirag Jani,"Batter, Bowler",Saurashtra,No appearance found in available IPL data
...,...,...,...,...,...
111,300,RA Tripathi,Batter,Goa,Pending identity verification
112,310,RP Sharma,Batter,Haryana,Pending identity verification
113,316,Gurinder Singh,Bowler,Chandigarh,Pending identity verification
114,317,Milind Kumar,Batter,Tripura,Pending identity verification


In [90]:
batch = pending_dobs.head(30)

display(
    batch[
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure"
        ]
    ].to_string(index=False)
)

' DOB_Verification_Order                Player          Roles       latest_team                              IPL_Exposure\n                      2               A Sheth Batter, Bowler            Baroda No appearance found in available IPL data\n                      3           LI Meriwala         Bowler            Baroda No appearance found in available IPL data\n                      4        Vishnu Solanki         Batter            Baroda No appearance found in available IPL data\n                      5 Dharmendrasinh Jadeja         Bowler        Saurashtra No appearance found in available IPL data\n                      6           Chirag Jani Batter, Bowler        Saurashtra No appearance found in available IPL data\n                      7          Baltej Singh         Bowler            Punjab No appearance found in available IPL data\n                      8           Bhanu Pania         Batter            Baroda No appearance found in available IPL data\n                      9

In [91]:
dob_batch_1 = {
    "A Sheth": "1996-02-03",
    "LI Meriwala": "1991-12-11",
    "Vishnu Solanki": "1992-10-15",
    "Dharmendrasinh Jadeja": "1990-08-04",
    "Chirag Jani": "1989-11-09",
    "Baltej Singh": "1990-11-04",
    "Bhanu Pania": "1996-09-04",
    "Mohit Avasthi": "1992-11-18",
    "NA Rathva": "1999-03-10",
    "A Nagwaswalla": "1997-10-17",
    "Chintan Gaja": "1994-11-13",
    "K Kartikeya": "1997-12-26",
    "Jay Gohil": "2000-12-13",
    "Vivek Singh": "1993-11-01",
    "Chirag Gandhi": "1990-06-18",
    "Shivam Chaudhary": "1997-08-04",
    "Samarth Vyas": "1995-11-28",
    "Priyank Panchal": "1990-04-09",
    "Harsh Tyagi": "1999-12-23",
    "H Desai": "1999-10-04",
    "Mukhtar Hussain": "1999-01-11",
    "Mrinmoy Dutta": "1998-11-24",
    "P Chopra": "1992-10-07",
    "Ravi Teja": "1994-10-19",
    "CA Pujara": "1988-01-25",
    "Eknath Kerkar": "1993-09-10",
    "Rex Rajkumar": "2000-08-30",
    "Tanay Thyagarajan": "1995-11-15"
}

for player, dob in dob_batch_1.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket profile"
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"Manual match required: {player} ({mask.sum()} matches)")

unique_profiles["Date_of_Birth"] = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

reference_date = pd.Timestamp("2026-11-16")
dob = unique_profiles["Date_of_Birth"]

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 276
DOB records pending: 88


In [92]:
pending_dobs = unique_profiles[
    unique_profiles["Date_of_Birth"].isna()
].sort_values("DOB_Verification_Order")

display(
    pending_dobs.head(30)[
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure"
        ]
    ].to_string(index=False)
)

' DOB_Verification_Order                          Player          Roles       latest_team                              IPL_Exposure\n                     19                       K D Patel         Bowler        Saurashtra No appearance found in available IPL data\n                     21                     AV Wankhade         Batter          Vidarbha No appearance found in available IPL data\n                     57                      Denish Das         Batter             Assam No appearance found in available IPL data\n                     58                    Rahul Buddhi         Batter Hyderabad (India) No appearance found in available IPL data\n                     59                 L Kishan Singha         Bowler           Manipur No appearance found in available IPL data\n                     60                   Akash Vasisht         Batter  Himachal Pradesh No appearance found in available IPL data\n                     61                       AV Rajput Batter, Bowler     

In [93]:
dob_batch_2 = {
    "Denish Das": "2002-05-17",
    "Rahul Buddhi": "1997-09-20",
    "L Kishan Singha": "1996-12-23",
    "Akash Vasisht": "1994-12-17",
    "Shivank Vashisth": "1995-09-17",
    "Johnson Singh": "1999-10-10",
    "Qamran Iqbal": "2001-10-17",
    "SD Lad": "1992-05-23",
    "Ganesh Satish": "1988-03-15",
    "Lalhmangaiha": "1983-11-17",
    "Karnajit Yumnam": "1997-12-27",
    "Langlonyamba Meitan Keishangbam": "1997-12-06",
    "Manish Golamaru": "1996-09-13",
    "B Sai Sudharsan": "2001-10-15"
}

for player, dob in dob_batch_2.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket profile"
        unique_profiles.loc[idx, "Age_Verification"] = (
            "Conflict - verify" if player in
            ["Johnson Singh", "Lalhmangaiha"] else "Verified"
        )
    else:
        print(f"Manual match required: {player} ({mask.sum()} matches)")

dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")
reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 290
DOB records pending: 74


In [94]:
pending_dobs = unique_profiles[
    unique_profiles["Date_of_Birth"].isna()
].sort_values("DOB_Verification_Order")

display(
    pending_dobs.head(30)[
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure"
        ]
    ].to_string(index=False)
)

' DOB_Verification_Order            Player          Roles  latest_team                              IPL_Exposure\n                     19         K D Patel         Bowler   Saurashtra No appearance found in available IPL data\n                     21       AV Wankhade         Batter     Vidarbha No appearance found in available IPL data\n                     61         AV Rajput Batter, Bowler       Baroda No appearance found in available IPL data\n                     64        AA Sarwate         Bowler     Vidarbha No appearance found in available IPL data\n                     65           AP Tare         Batter  Uttarakhand No appearance found in available IPL data\n                     71         SS Mundhe         Batter     Nagaland No appearance found in available IPL data\n                     92        SP Goswami         Batter      Mizoram No appearance found in available IPL data\n                     99           T Kohli         Batter      Mizoram No appearance found in av

In [95]:
dob_batch_3 = {
    "AA Sarwate": "1989-12-10",
    "AP Tare": "1987-11-07",
    "SS Mundhe": "1988-10-27",
    "SP Goswami": "1989-05-18",
    "T Kohli": "1988-12-17",
    "Anuj Raj": "2000-11-01",
    "Arjit Gupta": "1989-09-12",
    "Agrim Tiwari": "1996-04-14",
    "A Choudhary": "1990-01-28",
    "Bobby Zothansanga": "1986-08-25",
    "M Mohammed": "1991-12-03",
    "Gurkeerat Singh": "1990-06-29",
    "R Sanjay Yadav": "1995-05-10",
    "Abhishek Thakuri": "1998-10-31",
    "Jagjit Singh": "1997-01-28",
    "Virat Singh": "1997-12-08",
    "Mrunal Devdhar": "1992-12-16",
    "Naushad Shaikh": "1991-10-15",
    "Satyajeet Bachhav": "1992-11-28",
    "Arkaprabha Sinha": "1998-07-29"
}

for player, dob in dob_batch_3.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket profile"
        unique_profiles.loc[idx, "Age_Verification"] = "Verified"
    else:
        print(f"Manual match required: {player} ({mask.sum()} matches)")

dob = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 310
DOB records pending: 54


In [96]:
pending_dobs = unique_profiles[
    unique_profiles["Date_of_Birth"].isna()
].sort_values("DOB_Verification_Order")

display(
    pending_dobs.head(30)[
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure"
        ]
    ].to_string(index=False)
)

' DOB_Verification_Order           Player          Roles       latest_team                              IPL_Exposure\n                     19        K D Patel         Bowler        Saurashtra No appearance found in available IPL data\n                     21      AV Wankhade         Batter          Vidarbha No appearance found in available IPL data\n                     61        AV Rajput Batter, Bowler            Baroda No appearance found in available IPL data\n                    114           S Gani         Batter             Bihar No appearance found in available IPL data\n                    122       SS Agarwal         Bowler      Chhattisgarh No appearance found in available IPL data\n                    150    RB Bishnoi Jr         Bowler         Meghalaya No appearance found in available IPL data\n                    160         S Midhun         Bowler            Kerala No appearance found in available IPL data\n                    164     AA Herwadkar         Batter      Ch

In [97]:
dob_batch_4 = {
    "S Midhun": "1994-10-07",
    "Azim Kazi": "1993-10-14",
    "Bavanaka Sandeep": "1992-04-25",
    "Lakhan Singh": "1998-08-21",
    "Shasheem Rathour": "1994-10-16",
    "Manvan Patel": "2000-01-23",
    "Dippu Sangma": "1997-05-20",
    "Amod Yadav": "2001-12-10"
}

for player, dob in dob_batch_4.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket player profile"
        unique_profiles.loc[idx, "Age_Verification"] = (
            "Verify team identity" if player == "Lakhan Singh"
            else "Profile matched"
        )
    else:
        print(f"Manual match required: {player} ({mask.sum()} matches)")

dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")
reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 318
DOB records pending: 46


In [98]:
pending_dobs = unique_profiles[
    unique_profiles["Date_of_Birth"].isna()
].sort_values("DOB_Verification_Order")

display(
    pending_dobs.head(30)[
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure"
        ]
    ].to_string(index=False)
)

' DOB_Verification_Order          Player          Roles       latest_team                              IPL_Exposure\n                     19       K D Patel         Bowler        Saurashtra No appearance found in available IPL data\n                     21     AV Wankhade         Batter          Vidarbha No appearance found in available IPL data\n                     61       AV Rajput Batter, Bowler            Baroda No appearance found in available IPL data\n                    114          S Gani         Batter             Bihar No appearance found in available IPL data\n                    122      SS Agarwal         Bowler      Chhattisgarh No appearance found in available IPL data\n                    150   RB Bishnoi Jr         Bowler         Meghalaya No appearance found in available IPL data\n                    164    AA Herwadkar         Batter      Chhattisgarh No appearance found in available IPL data\n                    172      KH Devdhar         Batter            Barod

In [99]:
dob_batch_5 = {
    "AA Herwadkar": "1994-10-31",
    "KH Devdhar": "1989-12-14",
    "DB Ravi Teja": "1987-09-05",
    "Iqbal Abdulla": "1989-12-02",
    "Jaskaran Singh": "1989-09-04",
    "AA Barot": "1992-06-25",
    "M Markande": "1997-11-11",
    "P Simran Singh": "2000-08-10",
    "S Dube": "1993-06-26",
    "AM Rahane": "1988-06-06"
}

for player, dob in dob_batch_5.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket player profile"
        unique_profiles.loc[idx, "Age_Verification"] = (
            "Identity check required"
            if player == "Jaskaran Singh"
            else "Profile matched"
        )
    else:
        print(f"Manual match required: {player} ({mask.sum()} matches)")

dob = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 328
DOB records pending: 36


In [100]:
dob_batch_5 = {
    "AA Herwadkar": "1994-10-31",
    "KH Devdhar": "1989-12-14",
    "DB Ravi Teja": "1987-09-05",
    "Iqbal Abdulla": "1989-12-02",
    "Jaskaran Singh": "1989-09-04",
    "AA Barot": "1992-06-25",
    "M Markande": "1997-11-11",
    "P Simran Singh": "2000-08-10",
    "S Dube": "1993-06-26",
    "AM Rahane": "1988-06-06"
}

for player, dob in dob_batch_5.items():
    mask = unique_profiles["Player"].eq(player)

    if mask.sum() == 1:
        idx = unique_profiles.index[mask][0]
        unique_profiles.loc[idx, "Date_of_Birth"] = dob
        unique_profiles.loc[idx, "DOB_Source"] = "Cricket player profile"
        unique_profiles.loc[idx, "Age_Verification"] = (
            "Identity check required"
            if player == "Jaskaran Singh"
            else "Profile matched"
        )
    else:
        print(f"Manual match required: {player} ({mask.sum()} matches)")

dob = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

reference_date = pd.Timestamp("2026-11-16")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOB records entered:", dob.notna().sum())
print("DOB records pending:", dob.isna().sum())

DOB records entered: 328
DOB records pending: 36


In [101]:
pending_dobs = unique_profiles[
    unique_profiles["Date_of_Birth"].isna()
].sort_values("DOB_Verification_Order")

display(
    pending_dobs[
        [
            "DOB_Verification_Order",
            "Player",
            "Roles",
            "latest_team",
            "IPL_Exposure"
        ]
    ].to_string(index=False)
)

' DOB_Verification_Order          Player          Roles       latest_team                              IPL_Exposure\n                     19       K D Patel         Bowler        Saurashtra No appearance found in available IPL data\n                     21     AV Wankhade         Batter          Vidarbha No appearance found in available IPL data\n                     61       AV Rajput Batter, Bowler            Baroda No appearance found in available IPL data\n                    114          S Gani         Batter             Bihar No appearance found in available IPL data\n                    122      SS Agarwal         Bowler      Chhattisgarh No appearance found in available IPL data\n                    150   RB Bishnoi Jr         Bowler         Meghalaya No appearance found in available IPL data\n                    192         SB Wagh         Bowler               Goa No appearance found in available IPL data\n                    196 Akhilesh Sahani Batter, Bowler Arunachal Prades

In [102]:
dob_verified = {
    "Akhilesh Sahani": "1994-03-16",
    "KB Pawan": "1987-12-19",
    "Varun Sood": "1990-10-12",
    "Kaif Ahmad": "1997-05-05",
    "Narsingh Yadav": "1992-03-03",
    "Vidyadhar Patil": "2000-09-05",
    "Kishan Lyngdoh": "1998-03-21",
    "Rohit Shah": "1997-12-27",
    "Nazeeb Saiyed": "1995-12-01"
}

for player, dob in dob_verified.items():
    mask = unique_profiles["Player"].eq(player)
    unique_profiles.loc[mask, "Date_of_Birth"] = pd.Timestamp(dob)
    unique_profiles.loc[mask, "DOB_Source"] = "Verified player profile"
    unique_profiles.loc[mask, "Age_Verification"] = "Profile matched"

reference_date = pd.Timestamp("2026-11-16")
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("DOBs entered:", unique_profiles["Date_of_Birth"].notna().sum())
print("DOBs pending:", unique_profiles["Date_of_Birth"].isna().sum())

DOBs entered: 337
DOBs pending: 27


In [103]:
dob_batch_6 = {
    "AV Wankhade": "1992-03-14",
    "S Gani": "1999-09-02",
    "Kangabam Singh": "1994-11-24",
    "Shivam Chauhan": "1997-10-14",
    "Milind Kumar": "1991-02-15"
}

for player, dob in dob_batch_6.items():
    mask = unique_profiles["Player"].eq(player)
    unique_profiles.loc[mask, "Date_of_Birth"] = pd.Timestamp(dob)
    unique_profiles.loc[mask, "DOB_Source"] = "Verified player profile"
    unique_profiles.loc[mask, "Age_Verification"] = "Profile matched"

reference_date = pd.Timestamp("2026-11-16")
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("Entered:", unique_profiles["Date_of_Birth"].notna().sum())
print("Pending:", unique_profiles["Date_of_Birth"].isna().sum())

Entered: 342
Pending: 22


In [104]:
pending_dobs = unique_profiles[
    unique_profiles["Date_of_Birth"].isna()
].sort_values("DOB_Verification_Order")

display(
    pending_dobs[
        ["DOB_Verification_Order", "Player", "Roles", "latest_team"]
    ].to_string(index=False)
)


' DOB_Verification_Order          Player          Roles       latest_team\n                     19       K D Patel         Bowler        Saurashtra\n                     61       AV Rajput Batter, Bowler            Baroda\n                    122      SS Agarwal         Bowler      Chhattisgarh\n                    150   RB Bishnoi Jr         Bowler         Meghalaya\n                    192         SB Wagh         Bowler               Goa\n                    207       BA Pathan         Bowler            Baroda\n                    208    Ashith Rajiv         Bowler        Puducherry\n                    219        SK Patel         Batter            Baroda\n                    229        A Palkar         Bowler       Maharashtra\n                    238          N Rana Batter, Bowler     Uttar Pradesh\n                    239       KH Pandya Batter, Bowler            Baroda\n                    240        A Badoni         Batter             Delhi\n                    270  Shubham Shar

In [105]:
dob_batch_7 = {
    "Shubham Sharma": "1993-12-24",
    "A Palkar": "1989-09-01",
    "KH Pandya": "1991-03-24",
    "A Badoni": "1999-12-03",
    "SS Prabhudessai": "1997-12-06",
    "Bharat Sharma": "1995-11-15",
    "Rahul Singh": "1995-09-18",
    "Gurinder Singh": "1992-07-09"
}

for player, dob in dob_batch_7.items():
    mask = unique_profiles["Player"].eq(player)
    unique_profiles.loc[mask, "Date_of_Birth"] = pd.Timestamp(dob)
    unique_profiles.loc[mask, "DOB_Source"] = "Verified player profile"
    unique_profiles.loc[mask, "Age_Verification"] = "Profile matched"

reference_date = pd.Timestamp("2026-11-16")
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("Entered:", unique_profiles["Date_of_Birth"].notna().sum())
print("Pending:", unique_profiles["Date_of_Birth"].isna().sum())

Entered: 350
Pending: 14


In [106]:
dob_batch_8 = {
    "AV Rajput": "1998-05-09",
    "Ashith Rajiv": "1993-03-26",
    "N Rana": "1993-12-27",
    "KR Sen": "1996-10-22",
    "Sachin Kumar": "1997-12-25"
}

for player, dob in dob_batch_8.items():
    mask = unique_profiles["Player"].eq(player)
    unique_profiles.loc[mask, "Date_of_Birth"] = pd.Timestamp(dob)
    unique_profiles.loc[mask, "DOB_Source"] = "Verified cricket profile"
    unique_profiles.loc[mask, "Age_Verification"] = "Profile matched"

reference_date = pd.Timestamp("2026-11-16")
dob = pd.to_datetime(unique_profiles["Date_of_Birth"], errors="coerce")

unique_profiles["Age_as_of_2026_11_16"] = (
    reference_date.year - dob.dt.year
    - (
        (dob.dt.month > reference_date.month)
        | (
            (dob.dt.month == reference_date.month)
            & (dob.dt.day > reference_date.day)
        )
    ).astype("Int64")
).where(dob.notna())

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("Entered:", unique_profiles["Date_of_Birth"].notna().sum())
print("Pending:", unique_profiles["Date_of_Birth"].isna().sum())

Entered: 355
Pending: 9


In [107]:
mask = unique_profiles["Player"].eq("RB Bishnoi Jr")

unique_profiles.loc[mask, "Date_of_Birth"] = pd.Timestamp("1990-01-25")
unique_profiles.loc[mask, "DOB_Source"] = "Cricbuzz player profile"
unique_profiles.loc[mask, "Age_Verification"] = "Profile matched"

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("Entered:", unique_profiles["Date_of_Birth"].notna().sum())
print("Pending:", unique_profiles["Date_of_Birth"].isna().sum())

Entered: 356
Pending: 8


In [108]:
pending_dobs = unique_profiles[
    unique_profiles["Date_of_Birth"].isna()
].sort_values("DOB_Verification_Order")

display(
    pending_dobs[
        ["DOB_Verification_Order", "Player", "Roles", "latest_team"]
    ].to_string(index=False)
)

' DOB_Verification_Order      Player  Roles  latest_team\n                     19   K D Patel Bowler   Saurashtra\n                    122  SS Agarwal Bowler Chhattisgarh\n                    192     SB Wagh Bowler          Goa\n                    207   BA Pathan Bowler       Baroda\n                    219    SK Patel Batter       Baroda\n                    288    HP Patel Bowler      Gujarat\n                    300 RA Tripathi Batter          Goa\n                    310   RP Sharma Batter      Haryana'

In [109]:
mask = unique_profiles["Player"].eq("SS Agarwal")

unique_profiles.loc[mask, "Date_of_Birth"] = pd.Timestamp("1993-11-21")
unique_profiles.loc[mask, "DOB_Source"] = "CricX player profile; full-name cross-check"
unique_profiles.loc[mask, "Age_Verification"] = "Profile matched"

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("Entered:", unique_profiles["Date_of_Birth"].notna().sum())
print("Pending:", unique_profiles["Date_of_Birth"].isna().sum())

Entered: 357
Pending: 7


In [110]:
dob_batch_9 = {
    "SB Wagh": "1988-10-09",
    "RA Tripathi": "1991-03-02"
}

for player, dob in dob_batch_9.items():
    mask = unique_profiles["Player"].eq(player)
    unique_profiles.loc[mask, "Date_of_Birth"] = pd.Timestamp(dob)
    unique_profiles.loc[mask, "DOB_Source"] = "Verified cricket profile"
    unique_profiles.loc[mask, "Age_Verification"] = "Profile matched"

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("Entered:", unique_profiles["Date_of_Birth"].notna().sum())
print("Pending:", unique_profiles["Date_of_Birth"].isna().sum())

Entered: 359
Pending: 5


In [111]:
dob_batch_10 = {
    "K D Patel": "1991-09-13",
    "BA Pathan": "1994-08-19",
    "RP Sharma": "1993-06-28"
}

for player, dob in dob_batch_10.items():
    mask = unique_profiles["Player"].eq(player)
    unique_profiles.loc[mask, "Date_of_Birth"] = pd.Timestamp(dob)
    unique_profiles.loc[mask, "DOB_Source"] = "Verified cricket profile"
    unique_profiles.loc[mask, "Age_Verification"] = "Profile matched"

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("Entered:", unique_profiles["Date_of_Birth"].notna().sum())
print("Pending:", unique_profiles["Date_of_Birth"].isna().sum())


Entered: 362
Pending: 2


In [112]:
mask = unique_profiles["Player"].eq("SK Patel")

unique_profiles.loc[mask, "Date_of_Birth"] = pd.Timestamp("1993-05-16")
unique_profiles.loc[mask, "DOB_Source"] = "CricketArchive: Smit Kamleshbhai Patel"
unique_profiles.loc[mask, "Age_Verification"] = "Profile and team matched"

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

print("Entered:", unique_profiles["Date_of_Birth"].notna().sum())
print("Pending:", unique_profiles["Date_of_Birth"].isna().sum())

Entered: 363
Pending: 1


In [114]:
# ==========================================
# PRESSURE-X: AGE PROFILING (FIXED)
# ==========================================

import pandas as pd
import numpy as np

REFERENCE_DATE = pd.Timestamp("2026-11-16")

# Convert DOB
unique_profiles["Date_of_Birth"] = pd.to_datetime(
    unique_profiles["Date_of_Birth"], errors="coerce"
)

dob = unique_profiles["Date_of_Birth"]

# Calculate age
unique_profiles["Age"] = (
    REFERENCE_DATE.year - dob.dt.year
    - (
        (dob.dt.month > REFERENCE_DATE.month)
        | (
            (dob.dt.month == REFERENCE_DATE.month)
            & (dob.dt.day > REFERENCE_DATE.day)
        )
    ).astype("Int64")
).where(dob.notna())

# Age groups
unique_profiles["Age_Group"] = pd.cut(
    unique_profiles["Age"].astype(float),
    bins=[-1, 22, 25, 29, 100],
    labels=["U23", "23-25", "26-29", "30+"]
)

# Emerging profile classification
age = unique_profiles["Age"].astype(float)

conditions = [
    age.isna().to_numpy(),
    age.le(23).fillna(False).to_numpy(),
    age.between(24, 25).fillna(False).to_numpy(),
    age.between(26, 29).fillna(False).to_numpy()
]

choices = [
    "Age Unverified",
    "Young Prospect",
    "Emerging Prospect",
    "Developing / Experienced"
]

unique_profiles["Emerging_Profile"] = np.select(
    conditions,
    choices,
    default="Experienced"
)

unique_profiles["Age_Status"] = np.where(
    age.isna(),
    "DOB Pending",
    "Age Calculated"
)

# Age distribution
print("AGE DISTRIBUTION")
display(
    unique_profiles["Age_Group"]
    .value_counts(dropna=False)
    .rename_axis("Age Group")
    .reset_index(name="Players")
)

print("\nEMERGING PLAYER DISTRIBUTION")
display(
    unique_profiles["Emerging_Profile"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Players")
)

# Player-wise profile
age_profile = unique_profiles.sort_values(
    ["Age", "Player"],
    na_position="last"
)

display(
    age_profile[
        [
            "Player", "latest_team", "Roles", "Age",
            "Age_Group", "Emerging_Profile", "IPL_Exposure"
        ]
    ]
)

# Export
age_profile.to_csv(
    DATA_DIR / "PRESSURE_X_Age_Profile.csv",
    index=False
)

print("\nSaved: PRESSURE_X_Age_Profile.csv")

AGE DISTRIBUTION


,Age Group,Players
0,30+,238
1,26-29,107
2,23-25,16
3,U23,2
4,NaN,1



EMERGING PLAYER DISTRIBUTION


,Category,Players
0,Experienced,238
1,Developing / Experienced,107
2,Emerging Prospect,15
3,Young Prospect,3
4,Age Unverified,1


,Player,latest_team,Roles,Age,Age_Group,Emerging_Profile,IPL_Exposure
145,Khrievitso Kense,Nagaland,Bowler,22.00,U23,Young Prospect,No appearance found in available IPL data
334,Sameer Rizvi,Uttar Pradesh,Batter,22.00,U23,Young Prospect,Confirmed IPL appearance
340,Suyash Sharma,Delhi,Bowler,23.00,23-25,Young Prospect,Confirmed IPL appearance
146,Amit Ali,Tripura,Bowler,24.00,23-25,Emerging Prospect,No appearance found in available IPL data
227,Amod Yadav,Bihar,Bowler,24.00,23-25,Emerging Prospect,No appearance found in available IPL data
...,...,...,...,...,...,...,...
153,P Dogra,Puducherry,Batter,41.00,30+,Experienced,No appearance found in available IPL data
314,RV Uthappa,Kerala,Batter,41.00,30+,Experienced,Pending identity verification
103,Lalhmangaiha,Mizoram,Batter,42.00,30+,Experienced,No appearance found in available IPL data
290,WP Saha,Tripura,Batter,42.00,30+,Experienced,Pending identity verification



Saved: PRESSURE_X_Age_Profile.csv


In [115]:
# ==========================================
# PRESSURE-X: IPL EXPOSURE RECHECK
# ==========================================

known_ipl_players = [
    "Tanush Kotian",
    "Lukman Meriwala",
    "Prashant Chopra",
    "Shrikant Mundhe",
    "Shreevats Goswami",
    "Aditya Tare",
    "Kedar Devdhar",
    "Dwaraka Ravi Teja"
]

review = unique_profiles[
    unique_profiles["Player"].str.strip().isin(known_ipl_players)
][
    [
        "Player",
        "Roles",
        "latest_team",
        "IPL_Exposure",
        "Age"
    ]
].copy()

print("Known IPL appearance corrections")
display(review.to_string(index=False))

print("\nCurrent IPL exposure distribution")
display(
    unique_profiles["IPL_Exposure"]
    .value_counts(dropna=False)
    .rename_axis("IPL Exposure")
    .reset_index(name="Players")
)


Known IPL appearance corrections


'       Player  Roles latest_team                              IPL_Exposure   Age\nTanush Kotian Bowler      Mumbai No appearance found in available IPL data 28.00'


Current IPL exposure distribution


,IPL Exposure,Players
0,No appearance found in available IPL data,232
1,Pending identity verification,88
2,Confirmed IPL appearance,44


In [116]:
# Correct verified IPL appearance
mask = unique_profiles["Player"].str.strip().eq("Tanush Kotian")

unique_profiles.loc[mask, "IPL_Exposure"] = "Confirmed IPL appearance"

# Save updated master
unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Unique_Player_DOB_Queue.csv",
    index=False
)

unique_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Player_Master_Updated.csv",
    index=False
)

print("Updated:", mask.sum(), "player")
print("\nIPL Exposure Summary:")

display(
    unique_profiles["IPL_Exposure"]
    .value_counts()
    .rename_axis("IPL Exposure")
    .reset_index(name="Players")
)

Updated: 1 player

IPL Exposure Summary:


,IPL Exposure,Players
0,No appearance found in available IPL data,231
1,Pending identity verification,88
2,Confirmed IPL appearance,45


In [117]:
print("BATTER MASTER")
print("Shape:", batters_master.shape)
print("Columns:", batters_master.columns.tolist())
display(batters_master.head(3))

print("\nBOWLER MASTER")
print("Shape:", bowlers_master.shape)
print("Columns:", bowlers_master.columns.tolist())
display(bowlers_master.head(3))

BATTER MASTER
Shape: (213, 24)
Columns: ['player_key', 'latest_smat_season', 'latest_team', 'latest_team_count', 'team_status', 'smat_seasons', 'total_smat_matches', 'recent_seasons', 'recent_smat_matches', 'batter', 'scouting_score', 'match_key', 'exact_ipl_match', 'IPL_Player_Match', 'IPL_Match_Status', 'IPL_Match_Type', 'IPL_Exposure', 'Active_Seasons', 'Total_Matches', 'Total_Runs', 'Avg_Season_Average', 'Avg_Season_SR', 'Average_SR_SD', 'Latest_Season']


,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,batter,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,Average_SR_SD,Latest_Season
0,rk singh,2024/25,Uttar Pradesh,1,Single team,8,35,3.00,7.00,RK Singh,93.57,rksingh,False,Ramandeep Singh; Ravi Singh; Rinku Singh,Manual Verification,Multiple possible matches,Pending identity verification,8,35,910,40.82,147.96,38.75,2024/25
1,aa barot,2020/21,Saurashtra,1,Single team,2,11,0.00,0.00,AA Barot,91.50,aabarot,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,2,11,348,33.72,141.73,61.15,2020/21
2,rm patidar,2024/25,Madhya Pradesh,1,Single team,8,34,3.00,9.00,RM Patidar,91.10,rmpatidar,False,Rajat Patidar,Probable IPL Played,Unique initial + surname,Pending identity verification,8,33,1132,39.74,149.75,27.80,2024/25



BOWLER MASTER
Shape: (183, 25)
Columns: ['player_key', 'latest_smat_season', 'latest_team', 'latest_team_count', 'team_status', 'smat_seasons', 'total_smat_matches', 'recent_seasons', 'recent_smat_matches', 'bowler_name', 'scouting_score', 'match_key', 'exact_ipl_match', 'IPL_Player_Match', 'IPL_Match_Status', 'IPL_Match_Type', 'IPL_Exposure', 'Active_Seasons', 'Total_Matches', 'Total_Wickets', 'Avg_Season_Economy', 'Avg_Season_Bowling_Average', 'Avg_Season_Bowling_SR', 'Economy_SD', 'Latest_Season']


,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,bowler_name,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Wickets,Avg_Season_Economy,Avg_Season_Bowling_Average,Avg_Season_Bowling_SR,Economy_SD,Latest_Season
0,ashutosh aman,2022/23,Bihar,1,Single team,5,24,1.00,3.00,Ashutosh Aman,94.84,ashutoshaman,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,23,27,6.34,11.85,13.76,1.89,2022/23
1,shivank vashisth,2022/23,Delhi,1,Single team,3,8,1.00,4.00,Shivank Vashisth,92.31,shivankvashisth,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,3,8,16,4.85,11.04,12.00,2.25,2022/23
2,c sakariya,2022/23,Saurashtra,1,Single team,5,23,1.00,2.00,C Sakariya,91.68,csakariya,False,Chetan Sakariya,Probable IPL Played,Unique initial + surname,Pending identity verification,5,23,40,7.02,16.78,14.01,1.24,2022/23


In [118]:
# =====================================================
# PRESSURE-X: FINAL SCOUTING RANKING MODEL
# =====================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ---------- Helper functions ----------

def percentile_score(series, higher_is_better=True):
    x = pd.to_numeric(series, errors="coerce")

    if not higher_is_better:
        x = -x

    return x.rank(pct=True).fillna(0) * 100


def prepare_age_data():
    age_col = next(
        (c for c in ["Age", "Age_as_of_2026_11_16"]
         if c in unique_profiles.columns),
        None
    )

    if age_col is None:
        return None

    age_data = unique_profiles[["Player", age_col]].copy()
    age_data["name_key"] = (
        age_data["Player"].astype(str)
        .str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
    )

    age_data = age_data.drop_duplicates("name_key")
    return age_data[["name_key", age_col]].rename(
        columns={age_col: "Age"}
    )


age_data = prepare_age_data()


# ---------- BATTER RANKING ----------

bat = batters_master.copy()

bat["name_key"] = (
    bat["batter"].astype(str)
    .str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
)

bat = bat[
    (bat["Total_Matches"] >= 8) &
    (bat["Total_Runs"] >= 250)
].copy()

bat["Score_Average"] = percentile_score(
    bat["Avg_Season_Average"]
)

bat["Score_SR"] = percentile_score(
    bat["Avg_Season_SR"]
)

bat["Score_Runs"] = percentile_score(
    bat["Total_Runs"]
)

bat["Score_Consistency"] = percentile_score(
    bat["Average_SR_SD"],
    higher_is_better=False
)

bat["Score_Recent"] = percentile_score(
    bat["recent_smat_matches"]
)

bat["Score_Experience"] = percentile_score(
    bat["Active_Seasons"]
)

bat["Final_Scouting_Score"] = (
    bat["Score_Average"] * 0.25 +
    bat["Score_SR"] * 0.25 +
    bat["Score_Runs"] * 0.20 +
    bat["Score_Consistency"] * 0.15 +
    bat["Score_Recent"] * 0.10 +
    bat["Score_Experience"] * 0.05
)

if age_data is not None:
    bat = bat.merge(age_data, on="name_key", how="left")

bat = bat.sort_values(
    "Final_Scouting_Score", ascending=False
).reset_index(drop=True)

bat.insert(0, "Rank", range(1, len(bat) + 1))


# ---------- BOWLER RANKING ----------

bowl = bowlers_master.copy()

bowl["name_key"] = (
    bowl["bowler_name"].astype(str)
    .str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
)

bowl = bowl[
    (bowl["Total_Matches"] >= 8) &
    (bowl["Total_Wickets"] >= 12)
].copy()

bowl["Score_Average"] = percentile_score(
    bowl["Avg_Season_Bowling_Average"],
    higher_is_better=False
)

bowl["Score_Economy"] = percentile_score(
    bowl["Avg_Season_Economy"],
    higher_is_better=False
)

bowl["Score_Wickets"] = percentile_score(
    bowl["Total_Wickets"]
)

bowl["Score_SR"] = percentile_score(
    bowl["Avg_Season_Bowling_SR"],
    higher_is_better=False
)

bowl["Score_Consistency"] = percentile_score(
    bowl["Economy_SD"],
    higher_is_better=False
)

bowl["Score_Recent"] = percentile_score(
    bowl["recent_smat_matches"]
)

bowl["Final_Scouting_Score"] = (
    bowl["Score_Average"] * 0.25 +
    bowl["Score_Economy"] * 0.25 +
    bowl["Score_Wickets"] * 0.20 +
    bowl["Score_SR"] * 0.15 +
    bowl["Score_Consistency"] * 0.10 +
    bowl["Score_Recent"] * 0.05
)

if age_data is not None:
    bowl = bowl.merge(age_data, on="name_key", how="left")

bowl = bowl.sort_values(
    "Final_Scouting_Score", ascending=False
).reset_index(drop=True)

bowl.insert(0, "Rank", range(1, len(bowl) + 1))


# ---------- FINAL SHORTLISTS ----------

top_10_batters = bat.head(10).copy()
top_10_bowlers = bowl.head(10).copy()

# Save complete rankings and final shortlist
outputs = {
    "PRESSURE_X_Final_Batter_Ranking.csv": bat,
    "PRESSURE_X_Final_Bowler_Ranking.csv": bowl,
    "PRESSURE_X_Final_Top_10_Batters.csv": top_10_batters,
    "PRESSURE_X_Final_Top_10_Bowlers.csv": top_10_bowlers
}

for filename, frame in outputs.items():
    frame.to_csv(DATA_DIR / filename, index=False)

# ---------- DISPLAY ----------

print("BATTER CANDIDATES:", len(bat))
print("BOWLER CANDIDATES:", len(bowl))

print("\nTOP 10 BATTERS")

display(top_10_batters[
    ["Rank", "batter", "latest_team", "Age",
     "Total_Matches", "Total_Runs",
     "Avg_Season_Average", "Avg_Season_SR",
     "IPL_Exposure", "Final_Scouting_Score"]
])

print("\nTOP 10 BOWLERS")

display(top_10_bowlers[
    ["Rank", "bowler_name", "latest_team", "Age",
     "Total_Matches", "Total_Wickets",
     "Avg_Season_Bowling_Average",
     "Avg_Season_Economy",
     "IPL_Exposure", "Final_Scouting_Score"]
])

print("\nAll four ranking files saved.")

BATTER CANDIDATES: 156
BOWLER CANDIDATES: 129

TOP 10 BATTERS


,Rank,batter,latest_team,Age,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,IPL_Exposure,Final_Scouting_Score
0,1,Mandeep Singh,Punjab,34.00,37,1124,52.52,148.48,Confirmed IPL appearance,92.79
1,2,SS Iyer,Mumbai,31.00,34,1213,45.07,146.40,Pending identity verification,88.99
2,3,RM Patidar,Madhya Pradesh,33.00,33,1132,39.74,149.75,Pending identity verification,84.71
3,4,P Simran Singh,Punjab,26.00,35,1002,42.72,146.53,Pending identity verification,82.64
4,5,VR Iyer,Madhya Pradesh,31.00,33,901,44.52,133.66,Pending identity verification,81.92
5,6,AV Wankhade,Vidarbha,34.00,32,835,52.03,140.22,No appearance found in available IPL data,81.62
6,7,R Parag,Assam,25.00,26,874,48.33,144.45,Pending identity verification,80.69
7,8,Harpreet Singh,Madhya Pradesh,35.00,34,1181,45.31,128.00,Pending identity verification,79.46
8,9,Vishnu Vinod,Kerala,32.00,27,768,50.91,149.82,Confirmed IPL appearance,79.28
9,10,N Rana,Uttar Pradesh,32.00,46,1376,36.82,138.33,Pending identity verification,79.07



TOP 10 BOWLERS


,Rank,bowler_name,latest_team,Age,Total_Matches,Total_Wickets,Avg_Season_Bowling_Average,Avg_Season_Economy,IPL_Exposure,Final_Scouting_Score
0,1,Ravi Bishnoi,Gujarat,26.00,19,29,14.21,5.83,Confirmed IPL appearance,89.59
1,2,Cheepurapalli Stephen,Andhra,32.00,22,42,16.44,6.25,No appearance found in available IPL data,89.46
2,3,PP Chawla,Gujarat,37.00,43,67,18.50,6.38,Pending identity verification,87.00
3,4,AS Rajpoot,Uttar Pradesh,32.00,36,55,18.87,6.82,No appearance found in available IPL data,81.53
4,5,RD Chahar,Rajasthan,27.00,19,26,15.98,6.67,Pending identity verification,80.27
5,6,Ashutosh Aman,Bihar,40.00,23,27,11.85,6.34,No appearance found in available IPL data,79.44
6,7,C Sakariya,Saurashtra,28.00,23,40,16.78,7.02,Pending identity verification,78.99
7,8,JD Unadkat,Saurashtra,35.00,41,65,21.16,6.60,Pending identity verification,78.57
8,9,LI Meriwala,Baroda,34.00,40,62,18.95,7.52,No appearance found in available IPL data,77.31
9,10,SB Wagh,Goa,38.00,23,32,16.69,6.89,No appearance found in available IPL data,75.91



All four ranking files saved.


In [119]:
# ==========================================
# PRESSURE-X: EMERGING PROSPECT SHORTLIST
# ==========================================

# Emerging batter pool
emerging_batters = bat[
    (bat["Age"] <= 29) &
    (bat["Total_Matches"] >= 8) &
    (bat["Total_Runs"] >= 250)
].copy()

# Emerging bowler pool
emerging_bowlers = bowl[
    (bowl["Age"] <= 29) &
    (bowl["Total_Matches"] >= 8) &
    (bowl["Total_Wickets"] >= 12)
].copy()

# Rank within the emerging pool
emerging_batters = emerging_batters.sort_values(
    "Final_Scouting_Score", ascending=False
).reset_index(drop=True)

emerging_bowlers = emerging_bowlers.sort_values(
    "Final_Scouting_Score", ascending=False
).reset_index(drop=True)

emerging_batters.insert(0, "Emerging_Rank", range(1, len(emerging_batters)+1))
emerging_bowlers.insert(0, "Emerging_Rank", range(1, len(emerging_bowlers)+1))

print("EMERGING BATTERS:", len(emerging_batters))
display(emerging_batters.head(15)[
    ["Emerging_Rank", "batter", "latest_team", "Age",
     "Total_Matches", "Total_Runs", "Avg_Season_Average",
     "Avg_Season_SR", "IPL_Exposure", "Final_Scouting_Score"]
])

print("\nEMERGING BOWLERS:", len(emerging_bowlers))
display(emerging_bowlers.head(15)[
    ["Emerging_Rank", "bowler_name", "latest_team", "Age",
     "Total_Matches", "Total_Wickets",
     "Avg_Season_Bowling_Average", "Avg_Season_Economy",
     "IPL_Exposure", "Final_Scouting_Score"]
])

# Save
emerging_batters.to_csv(
    DATA_DIR / "PRESSURE_X_Emerging_Batters.csv", index=False
)

emerging_bowlers.to_csv(
    DATA_DIR / "PRESSURE_X_Emerging_Bowlers.csv", index=False
)

print("\nEmerging candidate files saved.")

EMERGING BATTERS: 39


,Emerging_Rank,batter,latest_team,Age,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,IPL_Exposure,Final_Scouting_Score
0,1,P Simran Singh,Punjab,26.00,35,1002,42.72,146.53,Pending identity verification,82.64
1,2,R Parag,Assam,25.00,26,874,48.33,144.45,Pending identity verification,80.69
2,3,RK Singh,Uttar Pradesh,29.00,35,910,40.82,147.96,Pending identity verification,77.60
3,4,Virat Singh,Jharkhand,28.00,36,1236,46.94,129.46,No appearance found in available IPL data,73.96
4,5,Lalit Yadav,Delhi,29.00,27,529,39.12,154.73,Confirmed IPL appearance,71.59
5,6,RD Gaikwad,Maharashtra,29.00,34,1132,34.72,136.55,Pending identity verification,71.41
6,7,Shubman Gill,Punjab,27.00,18,635,39.69,137.23,Confirmed IPL appearance,70.66
7,8,MK Lomror,Rajasthan,27.00,31,814,42.86,122.96,Pending identity verification,70.02
8,9,Abdul Samad,Jammu & Kashmir,25.00,16,428,51.23,138.93,Confirmed IPL appearance,68.46
9,10,PP Shaw,Mumbai,27.00,25,718,28.85,157.86,Pending identity verification,68.08



EMERGING BOWLERS: 33


,Emerging_Rank,bowler_name,latest_team,Age,Total_Matches,Total_Wickets,Avg_Season_Bowling_Average,Avg_Season_Economy,IPL_Exposure,Final_Scouting_Score
0,1,Ravi Bishnoi,Gujarat,26.00,19,29,14.21,5.83,Confirmed IPL appearance,89.59
1,2,RD Chahar,Rajasthan,27.00,19,26,15.98,6.67,Pending identity verification,80.27
2,3,C Sakariya,Saurashtra,28.00,23,40,16.78,7.02,Pending identity verification,78.99
3,4,Khrievitso Kense,Nagaland,22.00,9,13,13.50,5.39,No appearance found in available IPL data,75.23
4,5,SZ Mulani,Mumbai,29.00,31,35,20.70,6.50,Pending identity verification,74.38
5,6,IC Porel,Bengal,28.00,21,26,16.78,6.97,No appearance found in available IPL data,73.31
6,7,Yash Thakur,Vidarbha,27.00,31,48,19.06,7.28,Confirmed IPL appearance,72.81
7,8,Tanush Kotian,Mumbai,28.00,15,18,17.89,6.07,No appearance found in available IPL data,71.92
8,9,Mohsin Khan,Uttar Pradesh,28.00,25,34,19.92,6.86,Confirmed IPL appearance,71.84
9,10,NA Rathva,Baroda,27.00,27,27,18.13,7.20,No appearance found in available IPL data,69.63



Emerging candidate files saved.


In [120]:
# Complete candidate review

print("ALL 39 EMERGING BATTERS")
display(emerging_batters[[
    "Emerging_Rank", "batter", "latest_team", "Age",
    "Total_Matches", "Total_Runs",
    "Avg_Season_Average", "Avg_Season_SR",
    "IPL_Exposure", "Final_Scouting_Score"
]].to_string(index=False))

print("\nALL 33 EMERGING BOWLERS")
display(emerging_bowlers[[
    "Emerging_Rank", "bowler_name", "latest_team", "Age",
    "Total_Matches", "Total_Wickets",
    "Avg_Season_Bowling_Average", "Avg_Season_Economy",
    "IPL_Exposure", "Final_Scouting_Score"
]].to_string(index=False))

ALL 39 EMERGING BATTERS


' Emerging_Rank             batter       latest_team   Age  Total_Matches  Total_Runs  Avg_Season_Average  Avg_Season_SR                              IPL_Exposure  Final_Scouting_Score\n             1     P Simran Singh            Punjab 26.00             35        1002               42.72         146.53             Pending identity verification                 82.64\n             2            R Parag             Assam 25.00             26         874               48.33         144.45             Pending identity verification                 80.69\n             3           RK Singh     Uttar Pradesh 29.00             35         910               40.82         147.96             Pending identity verification                 77.60\n             4        Virat Singh         Jharkhand 28.00             36        1236               46.94         129.46 No appearance found in available IPL data                 73.96\n             5        Lalit Yadav             Delhi 29.00             27  


ALL 33 EMERGING BOWLERS


' Emerging_Rank       bowler_name      latest_team   Age  Total_Matches  Total_Wickets  Avg_Season_Bowling_Average  Avg_Season_Economy                              IPL_Exposure  Final_Scouting_Score\n             1      Ravi Bishnoi          Gujarat 26.00             19             29                       14.21                5.83                  Confirmed IPL appearance                 89.59\n             2         RD Chahar        Rajasthan 27.00             19             26                       15.98                6.67             Pending identity verification                 80.27\n             3        C Sakariya       Saurashtra 28.00             23             40                       16.78                7.02             Pending identity verification                 78.99\n             4  Khrievitso Kense         Nagaland 22.00              9             13                       13.50                5.39 No appearance found in available IPL data                 75.23\n    

In [121]:
# ==========================================
# PRESSURE-X: PROVISIONAL FINAL SHORTLIST
# ==========================================

selected_batters = [
    "Virat Singh",
    "Atharva Taide",
    "Shubham Pundir",
    "Rajat Dey",
    "Amandeep Khare",
    "Shivam Chaudhary",
    "Rohan Kunnummal",
    "Vishvaraj Jadeja",
    "S Gani",
    "P Simran Singh"
]

selected_bowlers = [
    "Khrievitso Kense",
    "NA Rathva",
    "Akash Choudhary",
    "Pradipta Pramanik",
    "Mukhtar Hussain",
    "Kanishk Seth",
    "Mrinmoy Dutta",
    "Ajay Sarkar",
    "Harishankar Reddy",
    "Tahmeed Rahman"
]

final_batters = bat[
    bat["batter"].isin(selected_batters)
].copy()

final_bowlers = bowl[
    bowl["bowler_name"].isin(selected_bowlers)
].copy()

# Scouting priority
final_batters["Scouting_Priority"] = [
    "High" if x <= 5 else "Review"
    for x in range(len(final_batters))
]

final_bowlers["Scouting_Priority"] = [
    "High" if x <= 4 else "Review"
    for x in range(len(final_bowlers))
]

final_batters.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Batters.csv",
    index=False
)

final_bowlers.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Bowlers.csv",
    index=False
)

print("FINAL BATTER SHORTLIST:", len(final_batters))
display(final_batters[
    ["batter", "latest_team", "Age",
     "Total_Runs", "Avg_Season_Average",
     "Avg_Season_SR", "IPL_Exposure"]
])

print("\nFINAL BOWLER SHORTLIST:", len(final_bowlers))
display(final_bowlers[
    ["bowler_name", "latest_team", "Age",
     "Total_Wickets", "Avg_Season_Bowling_Average",
     "Avg_Season_Economy", "IPL_Exposure"]
])

print("\nSHORTLIST FILES SAVED.")

FINAL BATTER SHORTLIST: 10


,batter,latest_team,Age,Total_Runs,Avg_Season_Average,Avg_Season_SR,IPL_Exposure
3,P Simran Singh,Punjab,26.00,1002,42.72,146.53,Pending identity verification
15,Virat Singh,Jharkhand,28.00,1236,46.94,129.46,No appearance found in available IPL data
43,Atharva Taide,Vidarbha,26.00,631,38.51,124.99,Pending identity verification
61,Shivam Chaudhary,Railways,29.00,586,28.24,124.84,No appearance found in available IPL data
62,Amandeep Khare,Chhattisgarh,29.00,758,30.67,111.14,No appearance found in available IPL data
71,Rajat Dey,Tripura,29.00,488,28.22,130.99,No appearance found in available IPL data
76,Shubham Pundir,Jammu & Kashmir,28.00,452,35.65,111.03,No appearance found in available IPL data
92,Rohan Kunnummal,Kerala,28.00,361,36.73,113.38,No appearance found in available IPL data
104,Vishvaraj Jadeja,Saurashtra,28.00,315,28.38,116.43,No appearance found in available IPL data
107,S Gani,Bihar,27.00,252,30.60,115.86,No appearance found in available IPL data



FINAL BOWLER SHORTLIST: 10


,bowler_name,latest_team,Age,Total_Wickets,Avg_Season_Bowling_Average,Avg_Season_Economy,IPL_Exposure
10,Khrievitso Kense,Nagaland,22.00,13,13.50,5.39,No appearance found in available IPL data
24,NA Rathva,Baroda,27.00,27,18.13,7.20,No appearance found in available IPL data
36,Akash Choudhary,Meghalaya,27.00,19,21.15,6.42,No appearance found in available IPL data
47,Pradipta Pramanik,Bengal,28.00,12,11.40,7.22,No appearance found in available IPL data
98,Kanishk Seth,Bengal,29.00,16,27.08,8.56,No appearance found in available IPL data
99,Mukhtar Hussain,Assam,27.00,22,30.86,8.21,No appearance found in available IPL data
109,Mrinmoy Dutta,Assam,27.00,14,27.85,9.18,No appearance found in available IPL data
119,Ajay Sarkar,Tripura,29.00,16,38.13,7.74,No appearance found in available IPL data
120,Harishankar Reddy,Andhra,28.00,24,36.34,9.60,No appearance found in available IPL data
124,Tahmeed Rahman,Nagaland,28.00,12,38.68,7.97,No appearance found in available IPL data



SHORTLIST FILES SAVED.


In [122]:
# ==========================================
# PRESSURE-X: FINAL SHORTLIST QUALITY CHECK
# ==========================================

# Batting quality flags
final_batters["Review_Flag"] = np.select(
    [
        (final_batters["Avg_Season_SR"] < 120) |
        (final_batters["Total_Runs"] < 350),
        (final_batters["Avg_Season_Average"] < 25)
    ],
    [
        "Review: Strike rate / sample",
        "Review: Batting average"
    ],
    default="Statistically promising"
)

# Bowling quality flags
final_bowlers["Review_Flag"] = np.select(
    [
        (final_bowlers["Avg_Season_Economy"] > 8) |
        (final_bowlers["Avg_Season_Bowling_Average"] > 30),
        (final_bowlers["Total_Wickets"] < 15)
    ],
    [
        "Review: Economy / average",
        "Review: Wicket sample"
    ],
    default="Statistically promising"
)

# Sort by scoring model
final_batters = final_batters.sort_values(
    "Final_Scouting_Score", ascending=False
).reset_index(drop=True)

final_bowlers = final_bowlers.sort_values(
    "Final_Scouting_Score", ascending=False
).reset_index(drop=True)

final_batters.insert(0, "Final_Rank", range(1, 11))
final_bowlers.insert(0, "Final_Rank", range(1, 11))

# Save final versions
final_batters.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Batters.csv", index=False
)

final_bowlers.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Bowlers.csv", index=False
)

print("FINAL BATTER REVIEW")
display(final_batters[[
    "Final_Rank", "batter", "latest_team", "Age",
    "Total_Runs", "Avg_Season_Average", "Avg_Season_SR",
    "Final_Scouting_Score", "Review_Flag"
]])

print("\nFINAL BOWLER REVIEW")
display(final_bowlers[[
    "Final_Rank", "bowler_name", "latest_team", "Age",
    "Total_Wickets", "Avg_Season_Bowling_Average",
    "Avg_Season_Economy", "Final_Scouting_Score",
    "Review_Flag"
]])

print("\nFINAL SHORTLIST VALIDATED AND SAVED.")

FINAL BATTER REVIEW


,Final_Rank,batter,latest_team,Age,Total_Runs,Avg_Season_Average,Avg_Season_SR,Final_Scouting_Score,Review_Flag
0,1,P Simran Singh,Punjab,26.00,1002,42.72,146.53,82.64,Statistically promising
1,2,Virat Singh,Jharkhand,28.00,1236,46.94,129.46,73.96,Statistically promising
2,3,Atharva Taide,Vidarbha,26.00,631,38.51,124.99,62.72,Statistically promising
3,4,Shivam Chaudhary,Railways,29.00,586,28.24,124.84,54.78,Statistically promising
4,5,Amandeep Khare,Chhattisgarh,29.00,758,30.67,111.14,54.60,Review: Strike rate / sample
5,6,Rajat Dey,Tripura,29.00,488,28.22,130.99,53.03,Statistically promising
6,7,Shubham Pundir,Jammu & Kashmir,28.00,452,35.65,111.03,51.68,Review: Strike rate / sample
7,8,Rohan Kunnummal,Kerala,28.00,361,36.73,113.38,47.18,Review: Strike rate / sample
8,9,Vishvaraj Jadeja,Saurashtra,28.00,315,28.38,116.43,41.33,Review: Strike rate / sample
9,10,S Gani,Bihar,27.00,252,30.60,115.86,40.50,Review: Strike rate / sample



FINAL BOWLER REVIEW


,Final_Rank,bowler_name,latest_team,Age,Total_Wickets,Avg_Season_Bowling_Average,Avg_Season_Economy,Final_Scouting_Score,Review_Flag
0,1,Khrievitso Kense,Nagaland,22.00,13,13.50,5.39,75.23,Review: Wicket sample
1,2,NA Rathva,Baroda,27.00,27,18.13,7.20,69.63,Statistically promising
2,3,Akash Choudhary,Meghalaya,27.00,19,21.15,6.42,63.22,Statistically promising
3,4,Pradipta Pramanik,Bengal,28.00,12,11.40,7.22,56.43,Review: Wicket sample
4,5,Kanishk Seth,Bengal,29.00,16,27.08,8.56,38.18,Review: Economy / average
5,6,Mukhtar Hussain,Assam,27.00,22,30.86,8.21,37.93,Review: Economy / average
6,7,Mrinmoy Dutta,Assam,27.00,14,27.85,9.18,28.39,Review: Economy / average
7,8,Ajay Sarkar,Tripura,29.00,16,38.13,7.74,23.91,Review: Economy / average
8,9,Harishankar Reddy,Andhra,28.00,24,36.34,9.60,21.92,Review: Economy / average
9,10,Tahmeed Rahman,Nagaland,28.00,12,38.68,7.97,19.17,Review: Economy / average



FINAL SHORTLIST VALIDATED AND SAVED.


In [123]:
# Check bowling-related columns
bowling_columns = [
    col for col in df.columns
    if any(word in col.lower() for word in [
        "over", "ball", "bowler", "run", "wicket",
        "dismiss", "innings", "match", "competition"
    ])
]

print("Available bowling analysis columns:")
print(bowling_columns)

display(df[bowling_columns].head())

Available bowling analysis columns:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'bowler_id', 'bowler_name', 'runs', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras1_runs', 'extras_2_runs', 'over_or_around', 'original_match_id', 'source_competition', 'ball', 'bowler', 'runs_off_bat', 'noballs', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'competition', 'match_format', 'innings_key', 'is_legal_ball', 'batter_ball', 'dismissed_player', 'bowler_wicket', 'bowler_runs_conceded']


,match_id,innings,overid,over_ball_num,innings_ball_num,bowler_id,bowler_name,runs,is_wicket,dismissal_type_id,dismissal_type,dismissed_player_id,extras1_runs,extras_2_runs,over_or_around,original_match_id,source_competition,ball,bowler,runs_off_bat,noballs,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,competition,match_format,innings_key,is_legal_ball,batter_ball,dismissed_player,bowler_wicket,bowler_runs_conceded
0,IPL_T20_T20I_13511,1,1.00,1.00,1.00,1772.00,<NA>,0.00,False,NaN,<NA>,NaN,0.00,0.00,Over,13511,IPL_T20_T20I,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,England in West Indies T20I Series 2022,T20I,IPL_T20_T20I_13511_1,True,True,<NA>,False,0.00
1,IPL_T20_T20I_13511,1,1.00,2.00,2.00,1772.00,<NA>,0.00,False,NaN,<NA>,NaN,0.00,0.00,Over,13511,IPL_T20_T20I,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,England in West Indies T20I Series 2022,T20I,IPL_T20_T20I_13511_1,True,True,<NA>,False,0.00
2,IPL_T20_T20I_13511,1,1.00,3.00,3.00,1772.00,<NA>,0.00,False,NaN,<NA>,NaN,0.00,0.00,Over,13511,IPL_T20_T20I,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,England in West Indies T20I Series 2022,T20I,IPL_T20_T20I_13511_1,True,True,<NA>,False,0.00
3,IPL_T20_T20I_13511,1,1.00,4.00,4.00,1772.00,<NA>,0.00,False,NaN,<NA>,NaN,0.00,0.00,Over,13511,IPL_T20_T20I,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,England in West Indies T20I Series 2022,T20I,IPL_T20_T20I_13511_1,True,True,<NA>,False,0.00
4,IPL_T20_T20I_13511,1,1.00,5.00,5.00,1772.00,<NA>,0.00,False,NaN,<NA>,NaN,0.00,0.00,Over,13511,IPL_T20_T20I,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,England in West Indies T20I Series 2022,T20I,IPL_T20_T20I_13511_1,True,True,<NA>,False,0.00


In [124]:
# ==========================================
# PRESSURE-X: BOWLING PHASE ANALYSIS
# ==========================================

# Copy relevant data
phase_df = df.copy()

# Ensure numeric columns
numeric_cols = [
    "overid", "runs", "bowler_runs_conceded",
    "bowler_wicket", "is_legal_ball"
]

for col in numeric_cols:
    phase_df[col] = pd.to_numeric(
        phase_df[col], errors="coerce"
    )

# Classify match phases
phase_df["Bowling_Phase"] = np.select(
    [
        phase_df["overid"].between(1, 6),
        phase_df["overid"].between(7, 15),
        phase_df["overid"].between(16, 20)
    ],
    [
        "Powerplay",
        "Middle Overs",
        "Death Overs"
    ],
    default="Other"
)

# Keep T20 deliveries with valid bowler information
phase_df = phase_df[
    (phase_df["match_format"].str.upper() == "T20") &
    (phase_df["Bowling_Phase"] != "Other") &
    (phase_df["bowler_name"].notna())
].copy()

# Aggregate phase-wise bowling statistics
phase_stats = (
    phase_df.groupby(
        ["bowler_name", "Bowling_Phase"]
    )
    .agg(
        Matches=("match_id", "nunique"),
        Balls=("is_legal_ball", "sum"),
        Runs_Conceded=("bowler_runs_conceded", "sum"),
        Wickets=("bowler_wicket", "sum")
    )
    .reset_index()
)

# Calculate performance metrics
phase_stats["Overs"] = phase_stats["Balls"] / 6

phase_stats["Economy"] = np.where(
    phase_stats["Overs"] > 0,
    phase_stats["Runs_Conceded"] / phase_stats["Overs"],
    np.nan
)

phase_stats["Bowling_Average"] = np.where(
    phase_stats["Wickets"] > 0,
    phase_stats["Runs_Conceded"] / phase_stats["Wickets"],
    np.nan
)

phase_stats["Bowling_SR"] = np.where(
    phase_stats["Wickets"] > 0,
    phase_stats["Balls"] / phase_stats["Wickets"],
    np.nan
)

# Filter shortlisted bowlers
shortlisted_names = final_bowlers["bowler_name"].unique()

shortlisted_phase_stats = phase_stats[
    phase_stats["bowler_name"].isin(shortlisted_names)
].copy()

# Save results
shortlisted_phase_stats.to_csv(
    DATA_DIR / "PRESSURE_X_Bowling_Phase_Analysis.csv",
    index=False
)

display(
    shortlisted_phase_stats.sort_values(
        ["bowler_name", "Bowling_Phase"]
    ).round(2)
)

print("Bowling phase analysis saved.")

,bowler_name,Bowling_Phase,Matches,Balls,Runs_Conceded,Wickets,Overs,Economy,Bowling_Average,Bowling_SR


Bowling phase analysis saved.


In [125]:
# ==========================================
# PRESSURE-X: BOWLER ROLE IDENTIFICATION
# ==========================================

# Create a phase-wise summary
role_summary = shortlisted_phase_stats.pivot(
    index="bowler_name",
    columns="Bowling_Phase",
    values=["Overs", "Economy", "Wickets", "Bowling_Average"]
)

role_summary.columns = [
    f"{metric}_{phase.replace(' ', '_')}"
    for metric, phase in role_summary.columns
]

role_summary = role_summary.reset_index()

# Ensure all expected phase columns exist
expected_columns = [
    f"{metric}_{phase}"
    for phase in ["Powerplay", "Middle_Overs", "Death_Overs"]
    for metric in ["Overs", "Economy", "Wickets"]
]

for col in expected_columns:
    if col not in role_summary.columns:
        role_summary[col] = np.nan

# Identify strongest phase
def identify_role(row):
    phases = {
        "Powerplay Specialist": (
            row["Economy_Powerplay"],
            row["Overs_Powerplay"]
        ),
        "Middle Overs Specialist": (
            row["Economy_Middle_Overs"],
            row["Overs_Middle_Overs"]
        ),
        "Death Overs Specialist": (
            row["Economy_Death_Overs"],
            row["Overs_Death_Overs"]
        )
    }

    eligible = {
        role: economy
        for role, (economy, overs) in phases.items()
        if pd.notna(economy) and pd.notna(overs) and overs >= 5
    }

    if not eligible:
        return "Insufficient Phase Sample"

    return min(eligible, key=eligible.get)

role_summary["Suggested_Role"] = role_summary.apply(
    identify_role, axis=1
)

# Save role analysis
role_summary.to_csv(
    DATA_DIR / "PRESSURE_X_Bowler_Role_Analysis.csv",
    index=False
)

display(role_summary.round(2))

print("Bowler role analysis saved.")

,bowler_name,Overs_Powerplay,Economy_Powerplay,Wickets_Powerplay,Overs_Middle_Overs,Economy_Middle_Overs,Wickets_Middle_Overs,Overs_Death_Overs,Economy_Death_Overs,Wickets_Death_Overs,Suggested_Role


Bowler role analysis saved.


In [126]:
# ==========================================
# PRESSURE-X: RR BOWLING RECRUITMENT MATRIX
# ==========================================

# Merge overall bowling performance with role analysis
recruitment_matrix = final_bowlers.merge(
    role_summary[
        ["bowler_name", "Suggested_Role"]
    ],
    on="bowler_name",
    how="left"
)

# Assign scouting tiers
recruitment_matrix["Scouting_Tier"] = np.select(
    [
        recruitment_matrix["Final_Scouting_Score"] >= 60,
        recruitment_matrix["Final_Scouting_Score"] >= 40
    ],
    [
        "Priority Prospect",
        "Further Evaluation"
    ],
    default="Watchlist"
)

# Flag limited statistical samples
recruitment_matrix["Sample_Concern"] = np.where(
    recruitment_matrix["Total_Wickets"] < 15,
    "Limited wicket sample",
    "Adequate initial sample"
)

# Add recommendation
recruitment_matrix["RR_Recommendation"] = np.select(
    [
        (recruitment_matrix["Scouting_Tier"] == "Priority Prospect") &
        (recruitment_matrix["Sample_Concern"] == "Adequate initial sample"),

        recruitment_matrix["Scouting_Tier"] == "Priority Prospect",

        recruitment_matrix["Scouting_Tier"] == "Further Evaluation"
    ],
    [
        "Shortlist for detailed scouting",
        "Validate with additional matches",
        "Review role and match performance"
    ],
    default="Retain as backup candidate"
)

# Save recruitment matrix
recruitment_matrix.to_csv(
    DATA_DIR / "PRESSURE_X_RR_Bowling_Recruitment_Matrix.csv",
    index=False
)

display(
    recruitment_matrix[[
        "Final_Rank",
        "bowler_name",
        "latest_team",
        "Age",
        "Total_Wickets",
        "Avg_Season_Bowling_Average",
        "Avg_Season_Economy",
        "Final_Scouting_Score",
        "Suggested_Role",
        "Scouting_Tier",
        "Sample_Concern",
        "RR_Recommendation"
    ]]
)

print("RR Bowling Recruitment Matrix saved.")

,Final_Rank,bowler_name,latest_team,Age,Total_Wickets,Avg_Season_Bowling_Average,Avg_Season_Economy,Final_Scouting_Score,Suggested_Role,Scouting_Tier,Sample_Concern,RR_Recommendation
0,1,Khrievitso Kense,Nagaland,22.00,13,13.50,5.39,75.23,NaN,Priority Prospect,Limited wicket sample,Validate with additional matches
1,2,NA Rathva,Baroda,27.00,27,18.13,7.20,69.63,NaN,Priority Prospect,Adequate initial sample,Shortlist for detailed scouting
2,3,Akash Choudhary,Meghalaya,27.00,19,21.15,6.42,63.22,NaN,Priority Prospect,Adequate initial sample,Shortlist for detailed scouting
3,4,Pradipta Pramanik,Bengal,28.00,12,11.40,7.22,56.43,NaN,Further Evaluation,Limited wicket sample,Review role and match performance
4,5,Kanishk Seth,Bengal,29.00,16,27.08,8.56,38.18,NaN,Watchlist,Adequate initial sample,Retain as backup candidate
5,6,Mukhtar Hussain,Assam,27.00,22,30.86,8.21,37.93,NaN,Watchlist,Adequate initial sample,Retain as backup candidate
6,7,Mrinmoy Dutta,Assam,27.00,14,27.85,9.18,28.39,NaN,Watchlist,Limited wicket sample,Retain as backup candidate
7,8,Ajay Sarkar,Tripura,29.00,16,38.13,7.74,23.91,NaN,Watchlist,Adequate initial sample,Retain as backup candidate
8,9,Harishankar Reddy,Andhra,28.00,24,36.34,9.60,21.92,NaN,Watchlist,Adequate initial sample,Retain as backup candidate
9,10,Tahmeed Rahman,Nagaland,28.00,12,38.68,7.97,19.17,NaN,Watchlist,Limited wicket sample,Retain as backup candidate


RR Bowling Recruitment Matrix saved.


In [127]:
# ==========================================
# PRESSURE-X: BATTING ROLE ANALYSIS
# ==========================================

batting_df = df.copy()

# Convert numeric columns
numeric_cols = [
    "overid", "innings_ball_num",
    "runs_off_bat", "batter_ball"
]

for col in numeric_cols:
    batting_df[col] = pd.to_numeric(
        batting_df[col], errors="coerce"
    )

# Retain T20 data with valid batter information
batting_df = batting_df[
    (batting_df["match_format"].str.upper() == "T20") &
    (batting_df["batter"].notna())
].copy()

# ------------------------------------------
# 1. Determine batting position
# ------------------------------------------

first_ball = batting_df[
    batting_df["batter_ball"] == 1
].copy()

first_ball = (
    first_ball.groupby(
        ["innings_key", "batter"]
    )["innings_ball_num"]
    .min()
    .reset_index()
)

first_ball["Batting_Position"] = (
    first_ball.groupby("innings_key")["innings_ball_num"]
    .rank(method="first")
)

# Average batting position for each player
batting_positions = (
    first_ball.groupby("batter")
    .agg(
        Innings=("innings_key", "nunique"),
        Avg_Batting_Position=("Batting_Position", "mean")
    )
    .reset_index()
)

# ------------------------------------------
# 2. Classify batting roles
# ------------------------------------------

def classify_batting_role(position):
    if position <= 2.5:
        return "Opener"
    elif position <= 5.5:
        return "Middle Order"
    else:
        return "Finisher"

batting_positions["Suggested_Role"] = (
    batting_positions["Avg_Batting_Position"]
    .apply(classify_batting_role)
)

# ------------------------------------------
# 3. Calculate batting phase statistics
# ------------------------------------------

batting_df["Batting_Phase"] = np.select(
    [
        batting_df["overid"].between(1, 6),
        batting_df["overid"].between(7, 15),
        batting_df["overid"].between(16, 20)
    ],
    [
        "Powerplay",
        "Middle Overs",
        "Death Overs"
    ],
    default="Other"
)

batting_df = batting_df[
    batting_df["Batting_Phase"] != "Other"
].copy()

batting_df["Boundary"] = (
    batting_df["runs_off_bat"].isin([4, 6])
)

phase_batting = (
    batting_df.groupby(
        ["batter", "Batting_Phase"]
    )
    .agg(
        Runs=("runs_off_bat", "sum"),
        Balls=("batter_ball", "sum"),
        Boundaries=("Boundary", "sum")
    )
    .reset_index()
)

phase_batting["Strike_Rate"] = np.where(
    phase_batting["Balls"] > 0,
    phase_batting["Runs"] / phase_batting["Balls"] * 100,
    np.nan
)

phase_batting["Boundary_Percentage"] = np.where(
    phase_batting["Balls"] > 0,
    phase_batting["Boundaries"] /
    phase_batting["Balls"] * 100,
    np.nan
)

# ------------------------------------------
# 4. Filter shortlisted batters
# ------------------------------------------

shortlisted_batter_names = final_batters["batter"].unique()

shortlisted_positions = batting_positions[
    batting_positions["batter"].isin(shortlisted_batter_names)
].copy()

shortlisted_phase_batting = phase_batting[
    phase_batting["batter"].isin(shortlisted_batter_names)
].copy()

# ------------------------------------------
# 5. Save outputs
# ------------------------------------------

shortlisted_positions.to_csv(
    DATA_DIR / "PRESSURE_X_Batting_Role_Analysis.csv",
    index=False
)

shortlisted_phase_batting.to_csv(
    DATA_DIR / "PRESSURE_X_Batting_Phase_Analysis.csv",
    index=False
)

print("BATTING ROLE ANALYSIS")
display(shortlisted_positions.round(2))

print("\nBATTING PHASE PERFORMANCE")
display(
    shortlisted_phase_batting
    .sort_values(["batter", "Batting_Phase"])
    .round(2)
)

print("\nBoth analysis files saved.")

BATTING ROLE ANALYSIS


,batter,Innings,Avg_Batting_Position,Suggested_Role



BATTING PHASE PERFORMANCE


,batter,Batting_Phase,Runs,Balls,Boundaries,Strike_Rate,Boundary_Percentage



Both analysis files saved.


In [129]:
# ==========================================
# FIX: BOWLER SCOUTING PROFILE
# ==========================================

# Rebuild bowling phase pivot
bowling_phase_pivot = shortlisted_phase_stats.pivot_table(
    index="bowler_name",
    columns="Bowling_Phase",
    values=["Overs", "Wickets", "Economy",
            "Bowling_Average", "Bowling_SR"],
    aggfunc="first"
)

# Flatten column names
bowling_phase_pivot.columns = [
    f"{metric}_{phase.replace(' ', '_')}"
    for metric, phase in bowling_phase_pivot.columns
]

bowling_phase_pivot = bowling_phase_pivot.reset_index()

# Merge with recruitment matrix
bowler_profiles = recruitment_matrix.merge(
    bowling_phase_pivot,
    on="bowler_name",
    how="left"
)

# Ensure expected columns exist
phase_overs = [
    "Overs_Powerplay",
    "Overs_Middle_Overs",
    "Overs_Death_Overs"
]

for col in phase_overs:
    if col not in bowler_profiles.columns:
        bowler_profiles[col] = 0

# Calculate phase sample confidence
bowler_profiles["Phase_Sample_Confidence"] = np.where(
    bowler_profiles[phase_overs].fillna(0).sum(axis=1) >= 15,
    "Moderate",
    "Low"
)

# Save
bowler_profiles.to_csv(
    DATA_DIR / "PRESSURE_X_Final_Bowler_Scouting_Profiles.csv",
    index=False
)

print("Bowler profiles corrected and saved.")

display(bowler_profiles[[
    "bowler_name",
    "Suggested_Role",
    "Overs_Powerplay",
    "Overs_Middle_Overs",
    "Overs_Death_Overs",
    "Phase_Sample_Confidence"
]])

Bowler profiles corrected and saved.


,bowler_name,Suggested_Role,Overs_Powerplay,Overs_Middle_Overs,Overs_Death_Overs,Phase_Sample_Confidence
0,Khrievitso Kense,NaN,0,0,0,Low
1,NA Rathva,NaN,0,0,0,Low
2,Akash Choudhary,NaN,0,0,0,Low
3,Pradipta Pramanik,NaN,0,0,0,Low
4,Kanishk Seth,NaN,0,0,0,Low
5,Mukhtar Hussain,NaN,0,0,0,Low
6,Mrinmoy Dutta,NaN,0,0,0,Low
7,Ajay Sarkar,NaN,0,0,0,Low
8,Harishankar Reddy,NaN,0,0,0,Low
9,Tahmeed Rahman,NaN,0,0,0,Low


In [130]:
# ==========================================
# DIAGNOSE BOWLING PHASE DATA
# ==========================================

print("1. DATASET SIZE")
print("df:", df.shape)
print("phase_df:", phase_df.shape)
print("phase_stats:", phase_stats.shape)
print("shortlisted_phase_stats:", shortlisted_phase_stats.shape)

print("\n2. MATCH FORMAT VALUES")
print(df["match_format"].value_counts(dropna=False).head(15))

print("\n3. BOWLING PHASE VALUES")
print(phase_df["Bowling_Phase"].value_counts(dropna=False))

print("\n4. BOWLER NAME AVAILABILITY")
print("Non-null bowler_name:", df["bowler_name"].notna().sum())
print("Non-null bowler:", df["bowler"].notna().sum())

print("\n5. SAMPLE BOWLER NAMES")
print(df[["bowler_name", "bowler"]].dropna(how="all").head(15))

print("\n6. SHORTLIST NAME MATCHING")

for name in final_bowlers["bowler_name"]:
    matched = phase_stats[
        phase_stats["bowler_name"].astype(str).str.strip().str.lower()
        == str(name).strip().lower()
    ]

    print(f"{name}: {len(matched)} phase records")

print("\n7. AVAILABLE PHASE STATISTICS")
display(phase_stats.head(10))

1. DATASET SIZE
df: (542108, 72)
phase_df: (0, 73)
phase_stats: (0, 10)
shortlisted_phase_stats: (0, 10)

2. MATCH FORMAT VALUES
match_format
NaN     341109
T20I    106327
T20      94672
Name: count, dtype: int64

3. BOWLING PHASE VALUES
Series([], Name: count, dtype: int64)

4. BOWLER NAME AVAILABILITY
Non-null bowler_name: 341109
Non-null bowler: 341109

5. SAMPLE BOWLER NAMES
        bowler_name       bowler
200999  DS Kulkarni  DS Kulkarni
201000  DS Kulkarni  DS Kulkarni
201001  DS Kulkarni  DS Kulkarni
201002  DS Kulkarni  DS Kulkarni
201003  DS Kulkarni  DS Kulkarni
201004  DS Kulkarni  DS Kulkarni
201005       S Dube       S Dube
201006       S Dube       S Dube
201007       S Dube       S Dube
201008       S Dube       S Dube
201009       S Dube       S Dube
201010       S Dube       S Dube
201011       S Dube       S Dube
201012  DS Kulkarni  DS Kulkarni
201013  DS Kulkarni  DS Kulkarni

6. SHORTLIST NAME MATCHING
Khrievitso Kense: 0 phase records
NA Rathva: 0 phase records
A

,bowler_name,Bowling_Phase,Matches,Balls,Runs_Conceded,Wickets,Overs,Economy,Bowling_Average,Bowling_SR


In [131]:
# ==========================================
# PRESSURE-X: CORRECTED BOWLING PHASE ANALYSIS
# ==========================================

phase_df = df.copy()

# Convert over number
phase_df["overid"] = pd.to_numeric(
    phase_df["overid"], errors="coerce"
)

# Retain shortlisted bowlers
shortlisted_names = final_bowlers["bowler_name"].dropna().unique()

phase_df = phase_df[
    phase_df["bowler_name"].isin(shortlisted_names)
].copy()

# Classify phases
phase_df["Bowling_Phase"] = np.select(
    [
        phase_df["overid"].between(1, 6),
        phase_df["overid"].between(7, 15),
        phase_df["overid"].between(16, 20)
    ],
    [
        "Powerplay",
        "Middle Overs",
        "Death Overs"
    ],
    default="Other"
)

phase_df = phase_df[
    phase_df["Bowling_Phase"] != "Other"
].copy()

# Aggregate statistics
shortlisted_phase_stats = (
    phase_df.groupby(
        ["bowler_name", "Bowling_Phase"]
    )
    .agg(
        Matches=("match_id", "nunique"),
        Balls=("is_legal_ball", "sum"),
        Runs_Conceded=("bowler_runs_conceded", "sum"),
        Wickets=("bowler_wicket", "sum")
    )
    .reset_index()
)

# Calculate metrics
shortlisted_phase_stats["Overs"] = (
    shortlisted_phase_stats["Balls"] / 6
)

shortlisted_phase_stats["Economy"] = np.where(
    shortlisted_phase_stats["Overs"] > 0,
    shortlisted_phase_stats["Runs_Conceded"] /
    shortlisted_phase_stats["Overs"],
    np.nan
)

shortlisted_phase_stats["Bowling_Average"] = np.where(
    shortlisted_phase_stats["Wickets"] > 0,
    shortlisted_phase_stats["Runs_Conceded"] /
    shortlisted_phase_stats["Wickets"],
    np.nan
)

shortlisted_phase_stats["Bowling_SR"] = np.where(
    shortlisted_phase_stats["Wickets"] > 0,
    shortlisted_phase_stats["Balls"] /
    shortlisted_phase_stats["Wickets"],
    np.nan
)

# Save
shortlisted_phase_stats.to_csv(
    DATA_DIR / "PRESSURE_X_Bowling_Phase_Analysis.csv",
    index=False
)

display(
    shortlisted_phase_stats.sort_values(
        ["bowler_name", "Bowling_Phase"]
    ).round(2)
)

print("Phase analysis rebuilt.")
print("Records:", len(phase_df))

,bowler_name,Bowling_Phase,Matches,Balls,Runs_Conceded,Wickets,Overs,Economy,Bowling_Average,Bowling_SR


Phase analysis rebuilt.
Records: 0


In [132]:
# ==========================================
# TRACE SHORTLISTED BOWLERS TO SOURCE DATA
# ==========================================

smat_df = data["smat"]

print("MASTER DATASET")
print("Shape:", df.shape)
print("Bowler column:", df["bowler_name"].notna().sum())

print("\nSMAT DATASET")
print("Shape:", smat_df.shape)
print("Columns:")
print(smat_df.columns.tolist())

print("\nSMAT SAMPLE")
display(smat_df.head(5))

# Find possible bowler-name columns
name_columns = [
    col for col in smat_df.columns
    if any(x in col.lower() for x in ["bowler", "player", "name"])
]

print("\nPossible player columns:", name_columns)

# Check shortlisted names against both datasets
shortlist = final_bowlers["bowler_name"].dropna().unique()

print("\nNAME MATCHING")

for col in name_columns:
    available_names = (
        smat_df[col].dropna().astype(str).str.strip().str.lower()
    )

    matches = [
        name for name in shortlist
        if name.strip().lower() in set(available_names)
    ]

    print(f"{col}: {len(matches)}/{len(shortlist)} matches")
    print("Matched:", matches)

MASTER DATASET
Shape: (542108, 72)
Bowler column: 341109

SMAT DATASET
Shape: (158839, 28)
Columns:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3', 'competition']

SMAT SAMPLE


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3,competition
0,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.10,0.10,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy
1,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.20,0.20,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy
2,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.30,0.30,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy
3,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.40,0.40,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy
4,1130661,2017/18,2018-01-07,Saurashtra Cricket Association Stadium,1,0.50,0.50,Baroda,Mumbai,KH Devdhar,Urvil Patel,DS Kulkarni,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Syed Mushtaq Ali Trophy



Possible player columns: ['bowler', 'player_dismissed', 'other_player_dismissed']

NAME MATCHING
bowler: 10/10 matches
Matched: ['Khrievitso Kense', 'NA Rathva', 'Akash Choudhary', 'Pradipta Pramanik', 'Kanishk Seth', 'Mukhtar Hussain', 'Mrinmoy Dutta', 'Ajay Sarkar', 'Harishankar Reddy', 'Tahmeed Rahman']
player_dismissed: 10/10 matches
Matched: ['Khrievitso Kense', 'NA Rathva', 'Akash Choudhary', 'Pradipta Pramanik', 'Kanishk Seth', 'Mukhtar Hussain', 'Mrinmoy Dutta', 'Ajay Sarkar', 'Harishankar Reddy', 'Tahmeed Rahman']
other_player_dismissed: 0/10 matches
Matched: []


In [133]:
# ==========================================
# SMAT BOWLING PHASE ANALYSIS
# ==========================================

phase_df = data["smat"].copy()

# Numeric conversions
for col in ["ball", "runs_off_bat", "wides", "noballs"]:
    phase_df[col] = pd.to_numeric(phase_df[col], errors="coerce").fillna(0)

# SMAT ball notation uses 0-based over numbering:
# 0-5 = Powerplay, 6-14 = Middle, 15-19 = Death
phase_df["Over_Number"] = np.floor(phase_df["ball"]).astype(int)

phase_df["Bowling_Phase"] = np.select(
    [
        phase_df["Over_Number"].between(0, 5),
        phase_df["Over_Number"].between(6, 14),
        phase_df["Over_Number"].between(15, 19)
    ],
    ["Powerplay", "Middle Overs", "Death Overs"],
    default="Other"
)

# Keep shortlisted bowlers and valid T20 overs
shortlisted_names = final_bowlers["bowler_name"].dropna().unique()

phase_df = phase_df[
    phase_df["bowler"].isin(shortlisted_names) &
    (phase_df["Bowling_Phase"] != "Other")
].copy()

# Legal deliveries: wides and no-balls are not legal
phase_df["Legal_Ball"] = (
    phase_df["wides"].eq(0) &
    phase_df["noballs"].eq(0)
).astype(int)

# Runs charged to the bowler
phase_df["Runs_Conceded"] = (
    phase_df["runs_off_bat"] +
    phase_df["wides"] +
    phase_df["noballs"]
)

# Wickets credited to the bowler
bowler_wicket_types = [
    "bowled", "caught", "caught and bowled",
    "lbw", "stumped", "hit wicket"
]

phase_df["Bowler_Wicket"] = (
    phase_df["wicket_type"]
    .astype(str).str.lower().str.strip()
    .isin(bowler_wicket_types)
).astype(int)

# Aggregate
shortlisted_phase_stats = (
    phase_df.groupby(["bowler", "Bowling_Phase"])
    .agg(
        Matches=("match_id", "nunique"),
        Balls=("Legal_Ball", "sum"),
        Runs_Conceded=("Runs_Conceded", "sum"),
        Wickets=("Bowler_Wicket", "sum")
    )
    .reset_index()
    .rename(columns={"bowler": "bowler_name"})
)

shortlisted_phase_stats["Overs"] = (
    shortlisted_phase_stats["Balls"] / 6
)

shortlisted_phase_stats["Economy"] = np.where(
    shortlisted_phase_stats["Balls"] > 0,
    shortlisted_phase_stats["Runs_Conceded"] * 6 /
    shortlisted_phase_stats["Balls"],
    np.nan
)

shortlisted_phase_stats["Bowling_Average"] = np.where(
    shortlisted_phase_stats["Wickets"] > 0,
    shortlisted_phase_stats["Runs_Conceded"] /
    shortlisted_phase_stats["Wickets"],
    np.nan
)

shortlisted_phase_stats["Bowling_SR"] = np.where(
    shortlisted_phase_stats["Wickets"] > 0,
    shortlisted_phase_stats["Balls"] /
    shortlisted_phase_stats["Wickets"],
    np.nan
)

shortlisted_phase_stats = shortlisted_phase_stats.round(2)

print("SMAT phase records:", len(phase_df))
print("Phase summary rows:", len(shortlisted_phase_stats))

display(
    shortlisted_phase_stats.sort_values(
        ["bowler_name", "Bowling_Phase"]
    )
)

SMAT phase records: 3402
Phase summary rows: 30


,bowler_name,Bowling_Phase,Matches,Balls,Runs_Conceded,Wickets,Overs,Economy,Bowling_Average,Bowling_SR
0,Ajay Sarkar,Death Overs,16,141,197.00,6,23.50,8.38,32.83,23.50
1,Ajay Sarkar,Middle Overs,14,132,157.00,1,22.00,7.14,157.00,132.00
2,Ajay Sarkar,Powerplay,20,204,244.00,9,34.00,7.18,27.11,22.67
3,Akash Choudhary,Death Overs,12,96,129.00,6,16.00,8.06,21.50,16.00
4,Akash Choudhary,Middle Overs,7,43,54.00,4,7.17,7.53,13.50,10.75
5,Akash Choudhary,Powerplay,18,210,229.00,9,35.00,6.54,25.44,23.33
6,Harishankar Reddy,Death Overs,17,131,231.00,9,21.83,10.58,25.67,14.56
7,Harishankar Reddy,Middle Overs,13,120,168.00,7,20.00,8.40,24.00,17.14
8,Harishankar Reddy,Powerplay,18,168,220.00,8,28.00,7.86,27.50,21.00
9,Kanishk Seth,Death Overs,6,48,91.00,6,8.00,11.38,15.17,8.00


In [134]:
# ==========================================
# PHASE PERFORMANCE VALIDATION
# ==========================================

phase_review = shortlisted_phase_stats.copy()

# Require at least 60 legal balls (10 overs)
phase_review["Sample_Status"] = np.where(
    phase_review["Balls"] >= 60,
    "Sufficient for comparison",
    "Limited sample"
)

# Rank bowlers within each phase
phase_review["Economy_Rank"] = (
    phase_review.groupby("Bowling_Phase")["Economy"]
    .rank(method="min", ascending=True)
)

phase_review["Wicket_Rank"] = (
    phase_review.groupby("Bowling_Phase")["Wickets"]
    .rank(method="min", ascending=False)
)

phase_review["Phase_Score"] = (
    (1 - phase_review.groupby("Bowling_Phase")["Economy"]
     .rank(pct=True, ascending=True)) * 50
    +
    phase_review.groupby("Bowling_Phase")["Wickets"]
    .rank(pct=True, ascending=False) * 50
)

# Exclude insufficient samples from role comparisons
phase_review["Phase_Score"] = phase_review["Phase_Score"].where(
    phase_review["Balls"] >= 60
)

display(
    phase_review.sort_values(
        ["Bowling_Phase", "Phase_Score"],
        ascending=[True, False]
    ).round(2)
)

print("\nWICKET TYPE AUDIT")
display(
    phase_df["wicket_type"]
    .fillna("No wicket")
    .value_counts()
    .to_frame("Deliveries")
)

,bowler_name,Bowling_Phase,Matches,Balls,Runs_Conceded,Wickets,Overs,Economy,Bowling_Average,Bowling_SR,Sample_Status,Economy_Rank,Wicket_Rank,Phase_Score
3,Akash Choudhary,Death Overs,12,96,129.00,6,16.00,8.06,21.50,16.00,Sufficient for comparison,4.00,4.00,55.00
0,Ajay Sarkar,Death Overs,16,141,197.00,6,23.50,8.38,32.83,23.50,Sufficient for comparison,5.00,4.00,50.00
18,Mukhtar Hussain,Death Overs,12,96,145.00,8,16.00,9.06,18.12,12.00,Sufficient for comparison,6.00,3.00,35.00
6,Harishankar Reddy,Death Overs,17,131,231.00,9,21.83,10.58,25.67,14.56,Sufficient for comparison,7.00,2.00,25.00
9,Kanishk Seth,Death Overs,6,48,91.00,6,8.00,11.38,15.17,8.00,Limited sample,8.00,4.00,NaN
12,Khrievitso Kense,Death Overs,1,6,5.00,1,1.00,5.00,5.00,6.00,Limited sample,1.00,9.00,NaN
15,Mrinmoy Dutta,Death Overs,8,52,106.00,3,8.67,12.23,35.33,17.33,Limited sample,9.00,7.00,NaN
21,NA Rathva,Death Overs,7,42,51.00,10,7.00,7.29,5.10,4.20,Limited sample,2.00,1.00,NaN
24,Pradipta Pramanik,Death Overs,2,12,15.00,3,2.00,7.50,5.00,4.00,Limited sample,3.00,7.00,NaN
27,Tahmeed Rahman,Death Overs,5,30,62.00,1,5.00,12.40,62.00,30.00,Limited sample,10.00,9.00,NaN



WICKET TYPE AUDIT


,Deliveries
wicket_type,
No wicket,3205
caught,116
bowled,31
run out,22
lbw,14
stumped,8
caught and bowled,6


In [135]:
# ==========================================
# FINAL BOWLING ROLE SUMMARY
# ==========================================

eligible_phases = phase_review[
    (phase_review["Balls"] >= 60)
].copy()

# Select each bowler's highest-scoring eligible phase
role_summary = (
    eligible_phases.sort_values(
        ["bowler_name", "Phase_Score"],
        ascending=[True, False]
    )
    .drop_duplicates("bowler_name")
    [[
        "bowler_name",
        "Bowling_Phase",
        "Matches",
        "Balls",
        "Wickets",
        "Economy",
        "Bowling_Average",
        "Bowling_SR",
        "Phase_Score"
    ]]
    .rename(columns={
        "Bowling_Phase": "Suggested_Role",
        "Matches": "Phase_Matches",
        "Balls": "Phase_Balls",
        "Wickets": "Phase_Wickets",
        "Economy": "Phase_Economy"
    })
)

role_summary = role_summary.sort_values(
    "Phase_Score", ascending=False
).reset_index(drop=True)

display(role_summary.round(2))

print("Bowlers with sufficient phase samples:",
      role_summary["bowler_name"].nunique())

print("\nRole distribution:")
display(role_summary["Suggested_Role"].value_counts())

,bowler_name,Suggested_Role,Phase_Matches,Phase_Balls,Phase_Wickets,Phase_Economy,Bowling_Average,Bowling_SR,Phase_Score
0,Mrinmoy Dutta,Middle Overs,9,60,3,4.40,14.67,20.00,87.50
1,NA Rathva,Powerplay,10,60,2,5.80,29.00,30.00,75.00
2,Ajay Sarkar,Middle Overs,14,132,1,7.14,157.00,132.00,65.00
3,Mukhtar Hussain,Middle Overs,10,60,5,6.80,13.60,12.00,60.00
4,Akash Choudhary,Death Overs,12,96,6,8.06,21.50,16.00,55.00
5,Pradipta Pramanik,Middle Overs,10,132,8,6.32,17.38,16.50,55.00
6,Khrievitso Kense,Middle Overs,9,167,12,5.60,13.00,13.92,50.00
7,Harishankar Reddy,Powerplay,18,168,8,7.86,27.50,21.00,37.50
8,Kanishk Seth,Powerplay,14,192,7,8.00,36.57,27.43,30.00
9,Tahmeed Rahman,Middle Overs,21,279,11,8.28,35.00,25.36,20.00


Bowlers with sufficient phase samples: 10

Role distribution:


Suggested_Role
Middle Overs    6
Powerplay       3
Death Overs     1
Name: count, dtype: int64

In [136]:
# ==========================================
# BOWLING RECRUITMENT MATRIX
# ==========================================

recruitment_matrix = final_bowlers.merge(
    role_summary,
    on="bowler_name",
    how="left"
)

recruitment_matrix = recruitment_matrix.sort_values(
    "Rank"
).reset_index(drop=True)

display(recruitment_matrix)

print("Final candidates:", recruitment_matrix["bowler_name"].nunique())
print("Missing phase roles:",
      recruitment_matrix["Suggested_Role"].isna().sum())


,Final_Rank,Rank,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,bowler_name,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Wickets,Avg_Season_Economy,Avg_Season_Bowling_Average,Avg_Season_Bowling_SR,Economy_SD,Latest_Season,name_key,Score_Average,Score_Economy,Score_Wickets,Score_SR,Score_Consistency,Score_Recent,Final_Scouting_Score,Age,Scouting_Priority,Review_Flag,Suggested_Role,Phase_Matches,Phase_Balls,Phase_Wickets,Phase_Economy,Bowling_Average,Bowling_SR,Phase_Score
0,1,11,khrievitso kense,2023/24,Nagaland,1,Single team,3,9,1.00,2.00,Khrievitso Kense,85.53,khrievitsokense,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,3,9,13,5.39,13.50,15.13,0.35,2023/24,khrievitsokense,97.67,99.22,5.43,89.15,100.00,31.01,75.23,22.00,High,Review: Wicket sample,Middle Overs,9,167,12,5.60,13.00,13.92,50.00
1,2,25,na rathva,2023/24,Baroda,1,Single team,6,30,2.00,8.00,NA Rathva,87.61,narathva,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,27,27,7.20,18.13,15.63,1.99,2023/24,narathva,89.92,59.69,62.02,86.82,25.58,84.88,69.63,27.00,High,Statistically promising,Powerplay,10,60,2,5.80,29.00,30.00,75.00
2,3,37,akash choudhary,2023/24,Meghalaya,1,Single team,4,18,1.00,2.00,Akash Choudhary,75.61,akashchoudhary,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,18,19,6.42,21.15,17.29,1.91,2023/24,akashchoudhary,75.19,88.37,31.78,77.52,27.91,31.01,63.22,27.00,High,Statistically promising,Death Overs,12,96,6,8.06,21.50,16.00,55.00
3,4,48,pradipta pramanik,2024/25,Bengal,1,Single team,4,10,1.00,2.00,Pradipta Pramanik,66.15,pradiptapramanik,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,10,12,7.22,11.40,11.80,3.39,2024/25,pradiptapramanik,99.22,58.14,1.94,100.00,1.55,31.01,56.43,28.00,High,Review: Wicket sample,Middle Overs,10,132,8,6.32,17.38,16.50,55.00
4,5,99,kanishk seth,2024/25,Bengal,1,Single team,4,14,1.00,2.00,Kanishk Seth,28.09,kanishkseth,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,14,16,8.56,27.08,19.00,0.93,2024/25,kanishkseth,50.39,13.18,15.50,64.34,79.84,31.01,38.18,29.00,High,Review: Economy / average,Powerplay,14,192,7,8.00,36.57,27.43,30.00
5,6,100,mukhtar hussain,2023/24,Assam,1,Single team,5,19,2.00,6.00,Mukhtar Hussain,47.55,mukhtarhussain,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,19,22,8.21,30.86,21.71,1.57,2023/24,mukhtarhussain,35.66,20.93,41.86,50.39,42.64,71.71,37.93,27.00,Review,Review: Economy / average,Middle Overs,10,60,5,6.80,13.60,12.00,60.00
6,7,110,mrinmoy dutta,2023/24,Assam,1,Single team,5,14,2.00,6.00,Mrinmoy Dutta,6.64,mrinmoydutta,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,14,14,9.18,27.85,19.37,2.78,2023/24,mrinmoydutta,47.29,3.10,8.53,62.79,10.85,71.71,28.39,27.00,Review,Review: Economy / average,Middle Overs,9,60,3,4.40,14.67,20.00,87.50
7,8,120,ajay sarkar,2023/24,Tripura,1,Single team,7,24,2.00,5.00,Ajay Sarkar,28.76,ajaysarkar,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,7,24,16,7.74,38.13,33.15,1.98,2023/24,ajaysarkar,17.83,36.43,15.50,10.08,26.36,62.02,23.91,29.00,Review,Review: Economy / average,Middle Overs,14,132,1,7.14,157.00,132.00,65.00
8,9,121,harishankar reddy,2023/24,Andhra,1,Single team,7,20,2.00,3.00,Harishankar Reddy,35.25,harishankarreddy,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,7,20,24,9.60,36.34,25.12,3.19,2023/24,harishankarreddy,18.60,1.55,48.06,31.01,4.65,43.02,21.92,28.00,Review,Review: Economy / average,Powerplay,18,168,8,7.86,27.50,21.00,37.50
9,10,125,tahmeed rahman,2023/24,Nagaland,1,Single team,5,21,2.00,6.00,Tahmeed Rahman,28.33,ta

Final candidates: 10
Missing phase roles: 0


In [137]:
# ==========================================
# CORRECT PHASE SCORING
# ==========================================

phase_review = shortlisted_phase_stats.copy()

phase_review["Sample_Status"] = np.where(
    phase_review["Balls"] >= 60,
    "Sufficient for comparison",
    "Limited sample"
)

# Percentile ranks within each phase
phase_review["Economy_Percentile"] = (
    phase_review.groupby("Bowling_Phase")["Economy"]
    .rank(pct=True, ascending=True)
)

phase_review["Wicket_Percentile"] = (
    phase_review.groupby("Bowling_Phase")["Wickets"]
    .rank(pct=True, ascending=False)
)

# Higher score = better performance
phase_review["Phase_Score"] = (
    (1 - phase_review["Economy_Percentile"]) * 50
    +
    (1 - phase_review["Wicket_Percentile"]) * 50
)

# Exclude small samples
phase_review.loc[
    phase_review["Balls"] < 60, "Phase_Score"
] = np.nan

display(
    phase_review.sort_values(
        ["Bowling_Phase", "Phase_Score"],
        ascending=[True, False]
    ).round(2)
)

,bowler_name,Bowling_Phase,Matches,Balls,Runs_Conceded,Wickets,Overs,Economy,Bowling_Average,Bowling_SR,Sample_Status,Economy_Percentile,Wicket_Percentile,Phase_Score
3,Akash Choudhary,Death Overs,12,96,129.00,6,16.00,8.06,21.50,16.00,Sufficient for comparison,0.40,0.50,55.00
6,Harishankar Reddy,Death Overs,17,131,231.00,9,21.83,10.58,25.67,14.56,Sufficient for comparison,0.70,0.20,55.00
18,Mukhtar Hussain,Death Overs,12,96,145.00,8,16.00,9.06,18.12,12.00,Sufficient for comparison,0.60,0.30,55.00
0,Ajay Sarkar,Death Overs,16,141,197.00,6,23.50,8.38,32.83,23.50,Sufficient for comparison,0.50,0.50,50.00
9,Kanishk Seth,Death Overs,6,48,91.00,6,8.00,11.38,15.17,8.00,Limited sample,0.80,0.50,NaN
12,Khrievitso Kense,Death Overs,1,6,5.00,1,1.00,5.00,5.00,6.00,Limited sample,0.10,0.95,NaN
15,Mrinmoy Dutta,Death Overs,8,52,106.00,3,8.67,12.23,35.33,17.33,Limited sample,0.90,0.75,NaN
21,NA Rathva,Death Overs,7,42,51.00,10,7.00,7.29,5.10,4.20,Limited sample,0.20,0.10,NaN
24,Pradipta Pramanik,Death Overs,2,12,15.00,3,2.00,7.50,5.00,4.00,Limited sample,0.30,0.75,NaN
27,Tahmeed Rahman,Death Overs,5,30,62.00,1,5.00,12.40,62.00,30.00,Limited sample,1.00,0.95,NaN


In [138]:
# ==========================================
# CORRECT PHASE SCORING
# ==========================================

phase_review = shortlisted_phase_stats.copy()

phase_review["Sample_Status"] = np.where(
    phase_review["Balls"] >= 60,
    "Sufficient for comparison",
    "Limited sample"
)

# Percentile ranks within each phase
phase_review["Economy_Percentile"] = (
    phase_review.groupby("Bowling_Phase")["Economy"]
    .rank(pct=True, ascending=True)
)

phase_review["Wicket_Percentile"] = (
    phase_review.groupby("Bowling_Phase")["Wickets"]
    .rank(pct=True, ascending=False)
)

# Higher score = better performance
phase_review["Phase_Score"] = (
    (1 - phase_review["Economy_Percentile"]) * 50
    +
    (1 - phase_review["Wicket_Percentile"]) * 50
)

# Exclude small samples
phase_review.loc[
    phase_review["Balls"] < 60, "Phase_Score"
] = np.nan

display(
    phase_review.sort_values(
        ["Bowling_Phase", "Phase_Score"],
        ascending=[True, False]
    ).round(2)
)

,bowler_name,Bowling_Phase,Matches,Balls,Runs_Conceded,Wickets,Overs,Economy,Bowling_Average,Bowling_SR,Sample_Status,Economy_Percentile,Wicket_Percentile,Phase_Score
3,Akash Choudhary,Death Overs,12,96,129.00,6,16.00,8.06,21.50,16.00,Sufficient for comparison,0.40,0.50,55.00
6,Harishankar Reddy,Death Overs,17,131,231.00,9,21.83,10.58,25.67,14.56,Sufficient for comparison,0.70,0.20,55.00
18,Mukhtar Hussain,Death Overs,12,96,145.00,8,16.00,9.06,18.12,12.00,Sufficient for comparison,0.60,0.30,55.00
0,Ajay Sarkar,Death Overs,16,141,197.00,6,23.50,8.38,32.83,23.50,Sufficient for comparison,0.50,0.50,50.00
9,Kanishk Seth,Death Overs,6,48,91.00,6,8.00,11.38,15.17,8.00,Limited sample,0.80,0.50,NaN
12,Khrievitso Kense,Death Overs,1,6,5.00,1,1.00,5.00,5.00,6.00,Limited sample,0.10,0.95,NaN
15,Mrinmoy Dutta,Death Overs,8,52,106.00,3,8.67,12.23,35.33,17.33,Limited sample,0.90,0.75,NaN
21,NA Rathva,Death Overs,7,42,51.00,10,7.00,7.29,5.10,4.20,Limited sample,0.20,0.10,NaN
24,Pradipta Pramanik,Death Overs,2,12,15.00,3,2.00,7.50,5.00,4.00,Limited sample,0.30,0.75,NaN
27,Tahmeed Rahman,Death Overs,5,30,62.00,1,5.00,12.40,62.00,30.00,Limited sample,1.00,0.95,NaN


In [139]:
# ==========================================
# UPDATE FINAL BOWLING RECRUITMENT MATRIX
# ==========================================

best_phase = (
    phase_review[
        phase_review["Balls"] >= 60
    ]
    .sort_values(
        ["bowler_name", "Phase_Score"],
        ascending=[True, False]
    )
    .drop_duplicates("bowler_name")
    [[
        "bowler_name",
        "Bowling_Phase",
        "Balls",
        "Wickets",
        "Economy",
        "Phase_Score"
    ]]
    .rename(columns={
        "Bowling_Phase": "Best_Phase",
        "Balls": "Best_Phase_Balls",
        "Wickets": "Best_Phase_Wickets",
        "Economy": "Best_Phase_Economy"
    })
)

recruitment_matrix = final_bowlers.merge(
    best_phase,
    on="bowler_name",
    how="left"
)

recruitment_matrix = recruitment_matrix.sort_values(
    "Final_Rank"
).reset_index(drop=True)

display(
    recruitment_matrix[[
        "Final_Rank",
        "bowler_name",
        "Age",
        "Total_Matches",
        "Total_Wickets",
        "Avg_Season_Economy",
        "Avg_Season_Bowling_Average",
        "Final_Scouting_Score",
        "Scouting_Priority",
        "IPL_Exposure",
        "Best_Phase",
        "Best_Phase_Balls",
        "Best_Phase_Wickets",
        "Best_Phase_Economy",
        "Phase_Score"
    ]].round(2)
)

# Save final matrix
recruitment_matrix.to_csv(
    DATA_DIR / "PRESSURE_X_Final_Bowling_Recruitment_Matrix.csv",
    index=False
)

print("Saved final bowling recruitment matrix.")

,Final_Rank,bowler_name,Age,Total_Matches,Total_Wickets,Avg_Season_Economy,Avg_Season_Bowling_Average,Final_Scouting_Score,Scouting_Priority,IPL_Exposure,Best_Phase,Best_Phase_Balls,Best_Phase_Wickets,Best_Phase_Economy,Phase_Score
0,1,Khrievitso Kense,22.00,9,13,5.39,13.50,75.23,High,No appearance found in available IPL data,Middle Overs,167,12,5.60,80.00
1,2,NA Rathva,27.00,27,27,7.20,18.13,69.63,High,No appearance found in available IPL data,Middle Overs,318,15,6.87,70.00
2,3,Akash Choudhary,27.00,18,19,6.42,21.15,63.22,High,No appearance found in available IPL data,Powerplay,210,9,6.54,75.00
3,4,Pradipta Pramanik,28.00,10,12,7.22,11.40,56.43,High,No appearance found in available IPL data,Middle Overs,132,8,6.32,65.00
4,5,Kanishk Seth,29.00,14,16,8.56,27.08,38.18,High,No appearance found in available IPL data,Powerplay,192,7,8.00,20.00
5,6,Mukhtar Hussain,27.00,19,22,8.21,30.86,37.93,Review,No appearance found in available IPL data,Death Overs,96,8,9.06,55.00
6,7,Mrinmoy Dutta,27.00,14,14,9.18,27.85,28.39,Review,No appearance found in available IPL data,Middle Overs,60,3,4.40,52.50
7,8,Ajay Sarkar,29.00,24,16,7.74,38.13,23.91,Review,No appearance found in available IPL data,Powerplay,204,9,7.18,65.00
8,9,Harishankar Reddy,28.00,20,24,9.60,36.34,21.92,Review,No appearance found in available IPL data,Death Overs,131,9,10.58,55.00
9,10,Tahmeed Rahman,28.00,21,12,7.97,38.68,19.17,Review,No appearance found in available IPL data,Middle Overs,279,11,8.28,40.00


Saved final bowling recruitment matrix.


In [141]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [142]:
from pathlib import Path
import pandas as pd

output_path = DATA_DIR / "PRESSURE_X_Final_Scouting_Report.xlsx"

# Executive summary
summary = pd.DataFrame({
    "Metric": [
        "Target",
        "Shortlisted batters",
        "Shortlisted bowlers",
        "Primary bowling prospects",
        "Secondary bowling prospects",
        "Scoring approach"
    ],
    "Details": [
        "Identify 10 emerging batters and 10 emerging bowlers for RR scouting ahead of IPL 2027.",
        len(final_batters),
        len(final_bowlers),
        "Khrievitso Kense, NA Rathva, Akash Choudhary",
        "Pradipta Pramanik and the remaining shortlisted bowlers",
        "Weighted percentile-based performance scoring"
    ]
})

# Methodology and limitations
methodology = pd.DataFrame({
    "Section": [
        "Batter scoring",
        "Bowler scoring",
        "Candidate filtering",
        "Emerging-player filter",
        "Bowling phase analysis",
        "Data coverage",
        "IPL exposure audit",
        "Interpretation"
    ],
    "Details": [
        "Average 25%, strike rate 25%, runs 20%, consistency 15%, recent matches 10%, active seasons 5%.",
        "Average 25%, economy 25%, wickets 20%, bowling strike rate 15%, economy consistency 10%, recent matches 5%.",
        "Batters: at least 8 matches and 250 runs. Bowlers: at least 8 matches and 12 wickets.",
        "Age 29 or below was used as a supporting filter, not a strict recruitment eligibility rule.",
        "SMAT deliveries were grouped into Powerplay (overs 0-5), Middle Overs (6-14) and Death Overs (15-19).",
        "SMAT delivery data available through 2024/25. The analysis does not establish 2025/26 form.",
        "Name matching can miss appearances or create uncertain identity matches. 'No appearance found' is not proof of no IPL exposure.",
        "Scores indicate historical statistical performance, not guaranteed IPL suitability or recruitment success."
    ]
})

# Export workbook
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Executive Summary", index=False)
    final_batters.to_excel(writer, sheet_name="Batters", index=False)
    final_bowlers.to_excel(writer, sheet_name="Bowlers", index=False)
    shortlisted_phase_stats.to_excel(writer, sheet_name="Bowling Phases", index=False)
    recruitment_matrix.to_excel(writer, sheet_name="Bowling Recruitment", index=False)
    methodology.to_excel(writer, sheet_name="Methodology", index=False)

    # Basic formatting
    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        for column_cells in ws.columns:
            max_length = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in column_cells
            )
            ws.column_dimensions[column_cells[0].column_letter].width = min(
                max(max_length + 2, 12), 55
            )

print("Final report created:", output_path)
print("Workbook sheets:", [
    "Executive Summary",
    "Batters",
    "Bowlers",
    "Bowling Phases",
    "Bowling Recruitment",
    "Methodology"
])

Final report created: C:\Users\pavan\PRESSURE_X_Final_Scouting_Report.xlsx
Workbook sheets: ['Executive Summary', 'Batters', 'Bowlers', 'Bowling Phases', 'Bowling Recruitment', 'Methodology']


In [144]:
print("BATTER COLUMNS:")
print(final_batters.columns.tolist())

print("\nBOWLER COLUMNS:")
print(final_bowlers.columns.tolist())

print("\nBATTER SAMPLE:")
display(final_batters.head(3))

print("\nBOWLER SAMPLE:")
display(final_bowlers.head(3))

BATTER COLUMNS:
['Final_Rank', 'Rank', 'player_key', 'latest_smat_season', 'latest_team', 'latest_team_count', 'team_status', 'smat_seasons', 'total_smat_matches', 'recent_seasons', 'recent_smat_matches', 'batter', 'scouting_score', 'match_key', 'exact_ipl_match', 'IPL_Player_Match', 'IPL_Match_Status', 'IPL_Match_Type', 'IPL_Exposure', 'Active_Seasons', 'Total_Matches', 'Total_Runs', 'Avg_Season_Average', 'Avg_Season_SR', 'Average_SR_SD', 'Latest_Season', 'name_key', 'Score_Average', 'Score_SR', 'Score_Runs', 'Score_Consistency', 'Score_Recent', 'Score_Experience', 'Final_Scouting_Score', 'Age', 'Scouting_Priority', 'Review_Flag']

BOWLER COLUMNS:
['Final_Rank', 'Rank', 'player_key', 'latest_smat_season', 'latest_team', 'latest_team_count', 'team_status', 'smat_seasons', 'total_smat_matches', 'recent_seasons', 'recent_smat_matches', 'bowler_name', 'scouting_score', 'match_key', 'exact_ipl_match', 'IPL_Player_Match', 'IPL_Match_Status', 'IPL_Match_Type', 'IPL_Exposure', 'Active_Seasons

,Final_Rank,Rank,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,batter,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,Average_SR_SD,Latest_Season,name_key,Score_Average,Score_SR,Score_Runs,Score_Consistency,Score_Recent,Score_Experience,Final_Scouting_Score,Age,Scouting_Priority,Review_Flag
0,1,4,p simran singh,2023/24,Punjab,1,Single team,6,35,2.00,13.00,P Simran Singh,87.70,psimransingh,False,Prabhsimran Singh,Probable IPL Played,Unique initial + surname,Pending identity verification,6,35,1002,42.72,146.53,27.45,2023/24,psimransingh,85.26,92.31,89.74,51.28,99.36,53.53,82.64,26.00,High,Statistically promising
1,2,16,virat singh,2021/22,Jharkhand,1,Single team,6,36,0.00,0.00,Virat Singh,76.22,viratsingh,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,6,36,1236,46.94,129.46,20.37,2021/22,viratsingh,91.03,67.95,98.72,71.79,10.26,53.53,73.96,28.00,High,Statistically promising
2,3,44,atharva taide,2024/25,Vidarbha,1,Single team,5,25,2.00,6.00,Atharva Taide,74.06,atharvataide,False,Atharwa Taide,Probable IPL Played,Unique initial + surname,Pending identity verification,5,25,631,38.51,124.99,23.80,2024/25,atharvataide,78.21,55.13,58.33,58.33,74.68,30.13,62.72,26.00,High,Statistically promising



BOWLER SAMPLE:


,Final_Rank,Rank,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,bowler_name,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Wickets,Avg_Season_Economy,Avg_Season_Bowling_Average,Avg_Season_Bowling_SR,Economy_SD,Latest_Season,name_key,Score_Average,Score_Economy,Score_Wickets,Score_SR,Score_Consistency,Score_Recent,Final_Scouting_Score,Age,Scouting_Priority,Review_Flag
0,1,11,khrievitso kense,2023/24,Nagaland,1,Single team,3,9,1.00,2.00,Khrievitso Kense,85.53,khrievitsokense,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,3,9,13,5.39,13.50,15.13,0.35,2023/24,khrievitsokense,97.67,99.22,5.43,89.15,100.00,31.01,75.23,22.00,High,Review: Wicket sample
1,2,25,na rathva,2023/24,Baroda,1,Single team,6,30,2.00,8.00,NA Rathva,87.61,narathva,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,27,27,7.20,18.13,15.63,1.99,2023/24,narathva,89.92,59.69,62.02,86.82,25.58,84.88,69.63,27.00,High,Statistically promising
2,3,37,akash choudhary,2023/24,Meghalaya,1,Single team,4,18,1.00,2.00,Akash Choudhary,75.61,akashchoudhary,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,18,19,6.42,21.15,17.29,1.91,2023/24,akashchoudhary,75.19,88.37,31.78,77.52,27.91,31.01,63.22,27.00,High,Statistically promising


In [146]:
# Reload original emerging batter pool
emerging_batters = pd.read_csv(
    DATA_DIR / "PRESSURE_X_Emerging_Batters.csv"
)

# Inspect available candidates and their IPL exposure
candidate_review = emerging_batters[
    (emerging_batters["Age"] <= 29) &
    (emerging_batters["IPL_Exposure"] ==
     "No appearance found in available IPL data")
].copy()

candidate_review = candidate_review.sort_values(
    "Final_Scouting_Score", ascending=False
)

display(candidate_review[
    [
        "batter", "latest_team", "Age",
        "Total_Matches", "Total_Runs",
        "Avg_Season_Average", "Avg_Season_SR",
        "Final_Scouting_Score", "IPL_Exposure"
    ]
].head(15))

,batter,latest_team,Age,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,Final_Scouting_Score,IPL_Exposure
3,Virat Singh,Jharkhand,28.00,36,1236,46.94,129.46,73.96,No appearance found in available IPL data
12,Shivam Chaudhary,Railways,29.00,23,586,28.24,124.84,54.78,No appearance found in available IPL data
13,Amandeep Khare,Chhattisgarh,29.00,28,758,30.67,111.14,54.60,No appearance found in available IPL data
16,Subhransu Senapati,Madhya Pradesh,29.00,20,492,26.54,127.11,53.80,No appearance found in available IPL data
17,Rajat Dey,Tripura,29.00,21,488,28.22,130.99,53.03,No appearance found in available IPL data
18,Tilak Varma,Hyderabad (India),24.00,18,520,39.11,126.29,51.84,No appearance found in available IPL data
19,Shubham Pundir,Jammu & Kashmir,28.00,21,452,35.65,111.03,51.68,No appearance found in available IPL data
27,Rohan Kunnummal,Kerala,28.00,13,361,36.73,113.38,47.18,No appearance found in available IPL data
30,Vishvaraj Jadeja,Saurashtra,28.00,18,315,28.38,116.43,41.33,No appearance found in available IPL data
31,S Gani,Bihar,27.00,11,252,30.60,115.86,40.50,No appearance found in available IPL data


In [148]:
# Replacement candidates

emerging_batters = pd.read_csv(
    DATA_DIR / "PRESSURE_X_Emerging_Batters.csv"
)

selected_names = set(
    final_batters["batter"].astype(str).str.strip().str.lower()
)

excluded_names = {"p simran singh", "tilak varma"}

replacement_pool = emerging_batters.copy()

replacement_pool["name_clean"] = (
    replacement_pool["batter"].astype(str).str.strip().str.lower()
)

replacement_pool = replacement_pool[
    ~replacement_pool["name_clean"].isin(selected_names | excluded_names)
    & (replacement_pool["Age"] <= 29)
    & (replacement_pool["Total_Matches"] >= 8)
    & (replacement_pool["Total_Runs"] >= 250)
].copy()

replacement_pool = replacement_pool.sort_values(
    "Final_Scouting_Score", ascending=False
)

display(replacement_pool[[
    col for col in [
        "batter", "latest_team", "Age",
        "Total_Matches", "Total_Runs",
        "Avg_Season_Average", "Avg_Season_SR",
        "Final_Scouting_Score", "IPL_Exposure"
    ]
    if col in replacement_pool.columns
]].head(20))

,batter,latest_team,Age,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,Final_Scouting_Score,IPL_Exposure
1,R Parag,Assam,25.00,26,874,48.33,144.45,80.69,Pending identity verification
2,RK Singh,Uttar Pradesh,29.00,35,910,40.82,147.96,77.60,Pending identity verification
4,Lalit Yadav,Delhi,29.00,27,529,39.12,154.73,71.59,Confirmed IPL appearance
5,RD Gaikwad,Maharashtra,29.00,34,1132,34.72,136.55,71.41,Pending identity verification
6,Shubman Gill,Punjab,27.00,18,635,39.69,137.23,70.66,Confirmed IPL appearance
7,MK Lomror,Rajasthan,27.00,31,814,42.86,122.96,70.02,Pending identity verification
8,Abdul Samad,Jammu & Kashmir,25.00,16,428,51.23,138.93,68.46,Confirmed IPL appearance
9,PP Shaw,Mumbai,27.00,25,718,28.85,157.86,68.08,Pending identity verification
11,Shivam Chauhan,Haryana,29.00,22,627,31.84,122.29,55.34,Pending identity verification
14,Urvil Patel,Gujarat,28.00,31,611,19.27,146.02,54.44,Confirmed IPL appearance


In [149]:
ipl_identity = pd.read_csv(
    DATA_DIR / "PRESSURE_X_IPL_Player_Identity.csv"
)

print("IPL identity columns:")
print([
    col for col in ipl_identity.columns
    if "ipl" in col.lower() or "match" in col.lower()
])

display(ipl_identity.head())

IPL identity columns:
['match_id', 'original_match_id', 'match_format', 'ipl_batter', 'ipl_bowler', 'ipl_non_striker', 'ipl_dismissed_player']


,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,non_striker_name,bowler_id,bowler_name,batting_type,bowling_type,batting_order,runs,is_four,is_six,is_wicket,dismissal_type_id,dismissal_type,dismissed_player_id,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time,original_match_id,source_competition,season_x,start_date,venue,ball,actual_delivery,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3,competition,comp_name,match_format,season_y,batter,season,innings_key,is_legal_ball,batter_ball,dismissed_player,bowler_wicket,bowler_runs_conceded,ipl_batter,ipl_bowler,ipl_non_striker,ipl_dismissed_player
0,IPL_T20_T20I_14094,1,1.00,1.00,1.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,NaN,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,NaN,NaN,NaN,Overstep,1.00,0.00,Pace,0.00,Backfoot defensive,Inswing,Over,Off-side,2022-03-26 19:30:05 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,NaN,NaN,IPL_T20_T20I_14094_1,True,True,NaN,False,0.00,Ruturaj Gaikwad,Umesh Yadav,Devon Conway,NaN
1,IPL_T20_T20I_14094,1,1.00,2.00,2.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,NaN,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,NaN,NaN,NaN,NaN,0.00,0.00,Pace,137.70,Cover Drive,Out Swing,Over,Off-side,2022-03-26 19:30:49 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,NaN,NaN,IPL_T20_T20I_14094_1,True,True,NaN,False,0.00,Ruturaj Gaikwad,Umesh Yadav,Devon Conway,NaN
2,IPL_T20_T20I_14094,1,1.00,3.00,3.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,NaN,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,NaN,NaN,NaN,wide,1.00,0.00,Pace,139.30,Leg Glance,Seamed In,Over,On-side,2022-03-26 19:31:20 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,NaN,NaN,IPL_T20_T20I_14094_1,True,True,NaN,False,0.00,Ruturaj Gaikwad,Umesh Yadav,Devon Conway,NaN
3,IPL_T20_T20I_14094,1,1.00,4.00,4.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,NaN,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,NaN,NaN,NaN,NaN,0.00,0.00,Pace,0.00,Backfoot defensive,Inswing,Over,Off-side,2022-03-26 19:31:52 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,NaN,NaN,IPL_T20_T20I_14094_1,True,True,NaN,False,0.00,Ruturaj Gaikwad,Umesh Yadav,Devon Conway,NaN
4,IPL_T20_T20I_14094,1,1.00,5.00,5.00,1.00,8.00,6059.00,Ruturaj Gaikwad,1941.00,Devon Conway,403.00,NaN,RHB,Right-arm fast-medium,1.00,0.00,False,False,False,2.00,NaN,6059.00,NaN,0.00,0.00,Pace,0.00,Square Cut,Out Swing,Over,Off-side,2022-03-26 19:32:31 UTC,14094,IPL_T20_T20I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IPL 2022,IPL 2022,T20,2022,NaN,NaN,IPL_T20_T20I_14094_1,True,True,NaN,False,0.00,Ruturaj Gaikwad,Umesh Yadav,Devon Conway,Ruturaj Gaikwad


In [150]:
# Count unique IPL matches for each batter using all batting roles
ipl_batting = pd.concat([
    ipl_identity[["match_id", "ipl_batter"]].rename(
        columns={"ipl_batter": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_non_striker"]].rename(
        columns={"ipl_non_striker": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_dismissed_player"]].rename(
        columns={"ipl_dismissed_player": "IPL_Player"}
    )
], ignore_index=True)

ipl_batting["IPL_Player"] = ipl_batting["IPL_Player"].astype("string").str.strip()
ipl_batting = ipl_batting.dropna(subset=["IPL_Player", "match_id"])

ipl_match_counts = (
    ipl_batting.groupby("IPL_Player")["match_id"]
    .nunique()
    .reset_index(name="IPL_Matches")
)

# Normalize names for matching
def normalize_name(name):
    return re.sub(r"[^a-z]", "", str(name).lower())

ipl_match_counts["name_key"] = ipl_match_counts["IPL_Player"].apply(normalize_name)

final_batters["name_key"] = final_batters["batter"].apply(normalize_name)

# Merge IPL appearance counts
final_batters = final_batters.drop(
    columns=["IPL_Matches", "IPL_Player"],
    errors="ignore"
).merge(
    ipl_match_counts.rename(columns={"IPL_Player": "IPL_Player"}),
    on="name_key",
    how="left"
)

final_batters["IPL_Matches"] = final_batters["IPL_Matches"].fillna(0).astype(int)

# Review before removing anyone
display(
    final_batters[
        ["batter", "IPL_Player", "IPL_Matches", "Final_Scouting_Score"]
    ].sort_values("IPL_Matches", ascending=False)
)

,batter,IPL_Player,IPL_Matches,Final_Scouting_Score
0,Virat Singh,<NA>,0,73.96
1,Atharva Taide,<NA>,0,62.72
2,Shivam Chaudhary,<NA>,0,54.78
3,Amandeep Khare,<NA>,0,54.60
4,Rajat Dey,<NA>,0,53.03
5,Shubham Pundir,<NA>,0,51.68
6,Rohan Kunnummal,<NA>,0,47.18
7,Vishvaraj Jadeja,<NA>,0,41.33
8,S Gani,<NA>,0,40.50


In [151]:
# Check IPL dataset coverage
print("IPL dataset shape:", ipl_identity.shape)

print("\nMatch formats:")
display(ipl_identity["match_format"].value_counts(dropna=False))

print("\nIPL seasons:")
if "season_y" in ipl_identity.columns:
    display(ipl_identity["season_y"].value_counts(dropna=False).sort_index())
elif "season_x" in ipl_identity.columns:
    display(ipl_identity["season_x"].value_counts(dropna=False).sort_index())

# Unique names appearing in IPL batting records
ipl_names = pd.concat([
    ipl_identity["ipl_batter"],
    ipl_identity["ipl_non_striker"],
    ipl_identity["ipl_dismissed_player"]
]).dropna().astype(str).str.strip().unique()

print("\nUnique IPL player names:", len(ipl_names))
print("\nSample IPL names:")
display(pd.DataFrame({"IPL_Player": sorted(ipl_names)[:40]}))

# Check likely name matches
from difflib import get_close_matches

for player in final_batters["batter"]:
    matches = get_close_matches(player, ipl_names, n=3, cutoff=0.65)
    print(f"{player}: {matches}")

IPL dataset shape: (87687, 76)

Match formats:


match_format
T20    87687
Name: count, dtype: int64


IPL seasons:


season_y
2022    17909
2023    17863
2024    17103
2025    17285
2026    17527
Name: count, dtype: int64


Unique IPL player names: 342

Sample IPL names:


,IPL_Player
0,Aaron Finch
1,Abdul Basith
2,Abdul Samad
3,Abhijeet Tomar
4,Abhinav Manohar
5,Abhishek Sharma
6,Abishek Porel
7,Adam Milne
8,Adam Zampa
9,Adil Rashid


Virat Singh: ['Himmat Singh', 'Shivam Singh', 'Sanvir Singh']
Atharva Taide: ['Atharwa Taide']
Shivam Chaudhary: []
Amandeep Khare: ['Sandeep Sharma', 'Aman Khan']
Rajat Dey: []
Shubham Pundir: ['Shubham Dubey']
Rohan Kunnummal: []
Vishvaraj Jadeja: []
S Gani: []


In [153]:
# Remove columns left by previous merge attempts
final_batters = final_batters.drop(
    columns=[
        "IPL_Player", "IPL_Matches",
        "IPL_Matches_x", "IPL_Matches_y"
    ],
    errors="ignore"
)

# Rebuild IPL participation data
ipl_participation = pd.concat([
    ipl_identity[["match_id", "ipl_batter"]].rename(
        columns={"ipl_batter": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_non_striker"]].rename(
        columns={"ipl_non_striker": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_bowler"]].rename(
        columns={"ipl_bowler": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_dismissed_player"]].rename(
        columns={"ipl_dismissed_player": "IPL_Player"}
    )
], ignore_index=True).dropna()

ipl_participation["IPL_Player"] = (
    ipl_participation["IPL_Player"].astype(str).str.strip()
)

ipl_counts = (
    ipl_participation.groupby("IPL_Player")["match_id"]
    .nunique()
    .reset_index(name="IPL_Matches")
)

# Handle known spelling variation
name_aliases = {"Atharva Taide": "Atharwa Taide"}

final_batters["IPL_Player"] = (
    final_batters["batter"].replace(name_aliases)
)

# Merge appearance counts
final_batters = final_batters.merge(
    ipl_counts,
    on="IPL_Player",
    how="left",
    validate="many_to_one"
)

final_batters["IPL_Matches"] = (
    final_batters["IPL_Matches"].fillna(0).astype(int)
)

# Retain players with 12 or fewer recorded appearances
final_batters = (
    final_batters[final_batters["IPL_Matches"] <= 12]
    .sort_values("Final_Scouting_Score", ascending=False)
    .reset_index(drop=True)
)

final_batters["Final_Rank"] = range(1, len(final_batters) + 1)

display(final_batters[
    ["Final_Rank", "batter", "IPL_Player",
     "IPL_Matches", "Final_Scouting_Score"]
])

,Final_Rank,batter,IPL_Player,IPL_Matches,Final_Scouting_Score
0,1,Virat Singh,Virat Singh,0,73.96
1,2,Atharva Taide,Atharwa Taide,10,62.72
2,3,Shivam Chaudhary,Shivam Chaudhary,0,54.78
3,4,Amandeep Khare,Amandeep Khare,0,54.60
4,5,Rajat Dey,Rajat Dey,0,53.03
5,6,Shubham Pundir,Shubham Pundir,0,51.68
6,7,Rohan Kunnummal,Rohan Kunnummal,0,47.18
7,8,Vishvaraj Jadeja,Vishvaraj Jadeja,0,41.33
8,9,S Gani,S Gani,0,40.50


In [154]:
Rank	Batter	Recorded IPL matches	Scouting score
1	Virat Singh	0*	73.96
2	Atharva Taide	10	62.72
3	Shivam Chaudhary	0	54.78
4	Amandeep Khare	0	54.60
5	Rajat Dey	0	53.03
6	Shubham Pundir	0	51.68
7	Rohan Kunnummal	0	47.18
8	Vishvaraj Jadeja	0	41.33
9	S Gani	0	40.50

SyntaxError: invalid syntax (594650294.py, line 1)

In [155]:
display(
    final_batters[
        ["Final_Rank", "batter", "IPL_Player",
         "IPL_Matches", "Final_Scouting_Score"]
    ]
)

,Final_Rank,batter,IPL_Player,IPL_Matches,Final_Scouting_Score
0,1,Virat Singh,Virat Singh,0,73.96
1,2,Atharva Taide,Atharwa Taide,10,62.72
2,3,Shivam Chaudhary,Shivam Chaudhary,0,54.78
3,4,Amandeep Khare,Amandeep Khare,0,54.60
4,5,Rajat Dey,Rajat Dey,0,53.03
5,6,Shubham Pundir,Shubham Pundir,0,51.68
6,7,Rohan Kunnummal,Rohan Kunnummal,0,47.18
7,8,Vishvaraj Jadeja,Vishvaraj Jadeja,0,41.33
8,9,S Gani,S Gani,0,40.50


In [156]:
final_batters.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Batters.csv",
    index=False
)

In [157]:
# Remove columns from any previous attempts
final_bowlers = final_bowlers.drop(
    columns=["IPL_Player", "IPL_Matches"],
    errors="ignore"
)

# Collect player participation from IPL ball-by-ball data
ipl_participation = pd.concat([
    ipl_identity[["match_id", "ipl_batter"]].rename(
        columns={"ipl_batter": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_non_striker"]].rename(
        columns={"ipl_non_striker": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_bowler"]].rename(
        columns={"ipl_bowler": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_dismissed_player"]].rename(
        columns={"ipl_dismissed_player": "IPL_Player"}
    )
], ignore_index=True).dropna()

ipl_participation["IPL_Player"] = (
    ipl_participation["IPL_Player"].astype(str).str.strip()
)

# Count unique IPL matches per player
ipl_counts = (
    ipl_participation.groupby("IPL_Player")["match_id"]
    .nunique()
    .reset_index(name="IPL_Matches")
)

# Match bowler names
final_bowlers["IPL_Player"] = (
    final_bowlers["bowler_name"].astype(str).str.strip()
)

final_bowlers = final_bowlers.merge(
    ipl_counts,
    on="IPL_Player",
    how="left",
    validate="many_to_one"
)

final_bowlers["IPL_Matches"] = (
    final_bowlers["IPL_Matches"].fillna(0).astype(int)
)

# Retain players with 12 or fewer recorded IPL appearances
final_bowlers = (
    final_bowlers[final_bowlers["IPL_Matches"] <= 12]
    .sort_values("Final_Scouting_Score", ascending=False)
    .reset_index(drop=True)
)

final_bowlers["Final_Rank"] = range(1, len(final_bowlers) + 1)

display(final_bowlers[
    ["Final_Rank", "bowler_name", "IPL_Player",
     "IPL_Matches", "Final_Scouting_Score"]
])

,Final_Rank,bowler_name,IPL_Player,IPL_Matches,Final_Scouting_Score
0,1,Khrievitso Kense,Khrievitso Kense,0,75.23
1,2,NA Rathva,NA Rathva,0,69.63
2,3,Akash Choudhary,Akash Choudhary,0,63.22
3,4,Pradipta Pramanik,Pradipta Pramanik,0,56.43
4,5,Kanishk Seth,Kanishk Seth,0,38.18
5,6,Mukhtar Hussain,Mukhtar Hussain,0,37.93
6,7,Mrinmoy Dutta,Mrinmoy Dutta,0,28.39
7,8,Ajay Sarkar,Ajay Sarkar,0,23.91
8,9,Harishankar Reddy,Harishankar Reddy,0,21.92
9,10,Tahmeed Rahman,Tahmeed Rahman,0,19.17


In [158]:
# Save final batter and bowler shortlists
final_batters.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Batters.csv",
    index=False
)

final_bowlers.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Bowlers.csv",
    index=False
)

print("Saved successfully:")
print("1. PRESSURE_X_Final_10_Batters.csv")
print("2. PRESSURE_X_Final_10_Bowlers.csv")

print("\nFinal shortlist:")
print(f"Batters: {len(final_batters)}")
print(f"Bowlers: {len(final_bowlers)}")

Saved successfully:
1. PRESSURE_X_Final_10_Batters.csv
2. PRESSURE_X_Final_10_Bowlers.csv

Final shortlist:
Batters: 9
Bowlers: 10


In [159]:
print("Emerging batter columns:")
print(emerging_batters.columns.tolist())

display(emerging_batters.head(5))

Emerging batter columns:
['Emerging_Rank', 'Rank', 'player_key', 'latest_smat_season', 'latest_team', 'latest_team_count', 'team_status', 'smat_seasons', 'total_smat_matches', 'recent_seasons', 'recent_smat_matches', 'batter', 'scouting_score', 'match_key', 'exact_ipl_match', 'IPL_Player_Match', 'IPL_Match_Status', 'IPL_Match_Type', 'IPL_Exposure', 'Active_Seasons', 'Total_Matches', 'Total_Runs', 'Avg_Season_Average', 'Avg_Season_SR', 'Average_SR_SD', 'Latest_Season', 'name_key', 'Score_Average', 'Score_SR', 'Score_Runs', 'Score_Consistency', 'Score_Recent', 'Score_Experience', 'Final_Scouting_Score', 'Age']


,Emerging_Rank,Rank,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,batter,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,Average_SR_SD,Latest_Season,name_key,Score_Average,Score_SR,Score_Runs,Score_Consistency,Score_Recent,Score_Experience,Final_Scouting_Score,Age
0,1,4,p simran singh,2023/24,Punjab,1,Single team,6,35,2.00,13.00,P Simran Singh,87.70,psimransingh,False,Prabhsimran Singh,Probable IPL Played,Unique initial + surname,Pending identity verification,6,35,1002,42.72,146.53,27.45,2023/24,psimransingh,85.26,92.31,89.74,51.28,99.36,53.53,82.64,26.00
1,2,7,r parag,2023/24,Assam,1,Single team,6,28,2.00,6.00,R Parag,90.23,rparag,False,Riyan Parag,Probable IPL Played,Unique initial + surname,Pending identity verification,6,26,874,48.33,144.45,23.53,2023/24,rparag,93.59,89.10,80.13,58.97,74.68,53.53,80.69,25.00
2,3,12,rk singh,2024/25,Uttar Pradesh,1,Single team,8,35,3.00,7.00,RK Singh,93.57,rksingh,False,Ramandeep Singh; Ravi Singh; Rinku Singh,Manual Verification,Multiple possible matches,Pending identity verification,8,35,910,40.82,147.96,38.75,2024/25,rksingh,83.97,94.23,82.69,24.36,82.69,91.67,77.60,29.00
3,4,16,virat singh,2021/22,Jharkhand,1,Single team,6,36,0.00,0.00,Virat Singh,76.22,viratsingh,False,NaN,No IPL Match Found,No identity match,No appearance found in available IPL data,6,36,1236,46.94,129.46,20.37,2021/22,viratsingh,91.03,67.95,98.72,71.79,10.26,53.53,73.96,28.00
4,5,19,lalit yadav,2023/24,Delhi,1,Single team,7,39,2.00,7.00,Lalit Yadav,84.88,lalityadav,False,Lalit Yadav,IPL Played - Exact,Exact,Confirmed IPL appearance,7,27,529,39.12,154.73,31.17,2023/24,lalityadav,80.13,98.72,42.31,42.31,82.69,75.96,71.59,29.00


In [161]:
print(replacement_pool.columns.tolist())
print("Candidate count:", len(replacement_pool))
display(replacement_pool.head(10))

['Emerging_Rank', 'Rank', 'player_key', 'latest_smat_season', 'latest_team', 'latest_team_count', 'team_status', 'smat_seasons', 'total_smat_matches', 'recent_seasons', 'recent_smat_matches', 'batter', 'scouting_score', 'match_key', 'exact_ipl_match', 'IPL_Player_Match', 'IPL_Match_Status', 'IPL_Match_Type', 'IPL_Exposure', 'Active_Seasons', 'Total_Matches', 'Total_Runs', 'Avg_Season_Average', 'Avg_Season_SR', 'Average_SR_SD', 'Latest_Season', 'name_key', 'Score_Average', 'Score_SR', 'Score_Runs', 'Score_Consistency', 'Score_Recent', 'Score_Experience', 'Final_Scouting_Score', 'Age', 'name_clean']
Candidate count: 28


,Emerging_Rank,Rank,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,batter,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,Average_SR_SD,Latest_Season,name_key,Score_Average,Score_SR,Score_Runs,Score_Consistency,Score_Recent,Score_Experience,Final_Scouting_Score,Age,name_clean
1,2,7,r parag,2023/24,Assam,1,Single team,6,28,2.00,6.00,R Parag,90.23,rparag,False,Riyan Parag,Probable IPL Played,Unique initial + surname,Pending identity verification,6,26,874,48.33,144.45,23.53,2023/24,rparag,93.59,89.10,80.13,58.97,74.68,53.53,80.69,25.00,r parag
2,3,12,rk singh,2024/25,Uttar Pradesh,1,Single team,8,35,3.00,7.00,RK Singh,93.57,rksingh,False,Ramandeep Singh; Ravi Singh; Rinku Singh,Manual Verification,Multiple possible matches,Pending identity verification,8,35,910,40.82,147.96,38.75,2024/25,rksingh,83.97,94.23,82.69,24.36,82.69,91.67,77.60,29.00,rk singh
4,5,19,lalit yadav,2023/24,Delhi,1,Single team,7,39,2.00,7.00,Lalit Yadav,84.88,lalityadav,False,Lalit Yadav,IPL Played - Exact,Exact,Confirmed IPL appearance,7,27,529,39.12,154.73,31.17,2023/24,lalityadav,80.13,98.72,42.31,42.31,82.69,75.96,71.59,29.00,lalit yadav
5,6,22,rd gaikwad,2021/22,Maharashtra,1,Single team,5,34,0.00,0.00,RD Gaikwad,77.75,rdgaikwad,False,Ruturaj Gaikwad,Probable IPL Played,Unique initial + surname,Pending identity verification,5,34,1132,34.72,136.55,14.59,2021/22,rdgaikwad,69.23,78.85,95.83,84.62,10.26,30.13,71.41,29.00,rd gaikwad
6,7,25,shubman gill,2022/23,Punjab,1,Single team,4,18,1.00,5.00,Shubman Gill,81.27,shubmangill,False,Shubman Gill,IPL Played - Exact,Exact,Confirmed IPL appearance,4,18,635,39.69,137.23,19.13,2022/23,shubmangill,80.77,81.41,58.97,74.36,64.74,13.78,70.66,27.00,shubman gill
7,8,26,mk lomror,2022/23,Rajasthan,1,Single team,6,31,1.00,2.00,MK Lomror,64.98,mklomror,False,Mahipal Lomror,Probable IPL Played,Unique initial + surname,Pending identity verification,6,31,814,42.86,122.96,10.27,2022/23,mklomror,85.90,50.64,77.56,95.51,33.65,53.53,70.02,27.00,mk lomror
8,9,31,abdul samad,2023/24,Jammu & Kashmir,1,Single team,5,17,1.00,2.00,Abdul Samad,67.42,abdulsamad,False,Abdul Samad,IPL Played - Exact,Exact,Confirmed IPL appearance,5,16,428,51.23,138.93,15.19,2023/24,abdulsamad,95.51,84.62,30.77,82.69,33.65,30.13,68.46,25.00,abdul samad
9,10,32,pp shaw,2024/25,Mumbai,1,Single team,5,25,2.00,9.00,PP Shaw,82.51,ppshaw,False,Prithvi Shaw,Probable IPL Played,Unique initial + surname,Pending identity verification,5,25,718,28.85,157.86,29.50,2024/25,ppshaw,50.00,99.36,65.38,46.79,91.35,30.13,68.08,27.00,pp shaw
11,12,61,shivam chauhan,2021/22,Haryana,1,Single team,4,22,0.00,0.00,Shivam Chauhan,66.34,shivamchauhan,False,Saurav Chauhan,Probable IPL Played,Unique initial + surname,Pending identity verification,4,22,627,31.84,122.29,2.19,2021/22,shivamchauhan,60.90,47.44,57.69,100.00,10.26,13.78,55.34,29.00,shivam chauhan
14,15,64,urvil patel,2023/24,Gujarat,1,Single team,7,31,2.00,8.00,Urvil Patel,72.56,urvilpatel,False,Urvil Patel,IPL Played - Exact,Exact,Confirmed IPL appearance,7,31,611,19.27,146.02,33.90,2023/24,urvilpatel,9.62,91.03,54.49,38.46,88.14,75.96,54.44,28.00,urvil patel


In [162]:
# Build player participation records
ipl_participation = pd.concat([
    ipl_identity[["match_id", "ipl_batter"]].rename(
        columns={"ipl_batter": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_non_striker"]].rename(
        columns={"ipl_non_striker": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_bowler"]].rename(
        columns={"ipl_bowler": "IPL_Player"}
    ),
    ipl_identity[["match_id", "ipl_dismissed_player"]].rename(
        columns={"ipl_dismissed_player": "IPL_Player"}
    )
], ignore_index=True).dropna()

ipl_participation["IPL_Player"] = (
    ipl_participation["IPL_Player"].astype(str).str.strip()
)

ipl_counts = (
    ipl_participation.groupby("IPL_Player")["match_id"]
    .nunique()
    .reset_index(name="IPL_Matches")
)

# Map only identifiable name variations
aliases = {
    "R Parag": "Riyan Parag",
    "RD Gaikwad": "Ruturaj Gaikwad",
    "MK Lomror": "Mahipal Lomror",
    "PP Shaw": "Prithvi Shaw"
}

replacement_pool = replacement_pool.drop(
    columns=["IPL_Player", "IPL_Matches"],
    errors="ignore"
).copy()

replacement_pool["IPL_Player"] = (
    replacement_pool["batter"].replace(aliases)
)

replacement_pool = replacement_pool.merge(
    ipl_counts,
    on="IPL_Player",
    how="left",
    validate="many_to_one"
)

replacement_pool["IPL_Matches"] = (
    replacement_pool["IPL_Matches"].fillna(0).astype(int)
)

# Display candidates for review
display(
    replacement_pool[[
        "batter", "latest_team", "Age",
        "Total_Matches", "Total_Runs",
        "Avg_Season_Average", "Avg_Season_SR",
        "IPL_Player", "IPL_Matches",
        "Final_Scouting_Score"
    ]].sort_values("Final_Scouting_Score", ascending=False).head(15)
)

,batter,latest_team,Age,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,IPL_Player,IPL_Matches,Final_Scouting_Score
0,R Parag,Assam,25.00,26,874,48.33,144.45,Riyan Parag,64,80.69
1,RK Singh,Uttar Pradesh,29.00,35,910,40.82,147.96,RK Singh,0,77.60
2,Lalit Yadav,Delhi,29.00,27,529,39.12,154.73,Lalit Yadav,18,71.59
3,RD Gaikwad,Maharashtra,29.00,34,1132,34.72,136.55,Ruturaj Gaikwad,62,71.41
4,Shubman Gill,Punjab,27.00,18,635,39.69,137.23,Shubman Gill,76,70.66
5,MK Lomror,Rajasthan,27.00,31,814,42.86,122.96,Mahipal Lomror,27,70.02
6,Abdul Samad,Jammu & Kashmir,25.00,16,428,51.23,138.93,Abdul Samad,42,68.46
7,PP Shaw,Mumbai,27.00,25,718,28.85,157.86,Prithvi Shaw,26,68.08
8,Shivam Chauhan,Haryana,29.00,22,627,31.84,122.29,Shivam Chauhan,0,55.34
9,Urvil Patel,Gujarat,28.00,31,611,19.27,146.02,Urvil Patel,10,54.44


In [163]:
# Select Shivam Chauhan from the replacement pool
candidate = replacement_pool[
    replacement_pool["batter"].str.strip().str.lower()
    == "shivam chauhan"
].copy()

if len(candidate) != 1:
    raise ValueError("Expected exactly one Shivam Chauhan candidate.")

# Align columns with the existing final shortlist
candidate = candidate.reindex(columns=final_batters.columns)

# Add candidate and recalculate ranks
final_batters = pd.concat(
    [final_batters, candidate],
    ignore_index=True
)

final_batters = final_batters.sort_values(
    "Final_Scouting_Score",
    ascending=False
).reset_index(drop=True)

final_batters["Final_Rank"] = range(1, len(final_batters) + 1)

# Save updated shortlist
final_batters.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Batters.csv",
    index=False
)

display(final_batters[[
    "Final_Rank", "batter", "Total_Matches",
    "Total_Runs", "Avg_Season_Average",
    "Avg_Season_SR", "IPL_Matches",
    "Final_Scouting_Score"
]])

print("Final batters:", len(final_batters))

,Final_Rank,batter,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,IPL_Matches,Final_Scouting_Score
0,1,Virat Singh,36,1236,46.94,129.46,0,73.96
1,2,Atharva Taide,25,631,38.51,124.99,10,62.72
2,3,Shivam Chauhan,22,627,31.84,122.29,0,55.34
3,4,Shivam Chaudhary,23,586,28.24,124.84,0,54.78
4,5,Amandeep Khare,28,758,30.67,111.14,0,54.60
5,6,Rajat Dey,21,488,28.22,130.99,0,53.03
6,7,Shubham Pundir,21,452,35.65,111.03,0,51.68
7,8,Rohan Kunnummal,13,361,36.73,113.38,0,47.18
8,9,Vishvaraj Jadeja,18,315,28.38,116.43,0,41.33
9,10,S Gani,11,252,30.60,115.86,0,40.50


Final batters: 10


In [165]:
# ==========================================
# PRESSURE-X: FINAL SHORTLIST VALIDATION
# ==========================================

def validate_shortlist(df, player_col, category, stat_col):
    print(f"\n{'='*55}")
    print(f"{category.upper()} VALIDATION")
    print(f"{'='*55}")

    duplicates = df[df[player_col].duplicated(keep=False)]

    important_cols = [
        player_col, "Final_Scouting_Score",
        "Total_Matches", stat_col
    ]
    available_cols = [c for c in important_cols if c in df.columns]

    print("Total selected:", len(df))
    print("Duplicate players:", len(duplicates))
    print("\nMissing values:")
    print(df[available_cols].isnull().sum())

    print(
        "\nSorted by score:",
        df["Final_Scouting_Score"].is_monotonic_decreasing
    )

    display(df)

    return {
        "Category": category,
        "Selected": len(df),
        "Duplicates": len(duplicates),
        "Missing_Values": int(df[available_cols].isnull().sum().sum())
    }


bat_validation = validate_shortlist(
    final_batters, "batter", "Batters", "Total_Runs"
)

bowl_validation = validate_shortlist(
    final_bowlers, "bowler_name", "Bowlers", "Total_Wickets"
)

validation_summary = pd.DataFrame([
    bat_validation, bowl_validation
])

print("\nFINAL VALIDATION SUMMARY")
display(validation_summary)

print("Total shortlisted:", len(final_batters) + len(final_bowlers))


BATTERS VALIDATION
Total selected: 10
Duplicate players: 0

Missing values:
batter                  0
Final_Scouting_Score    0
Total_Matches           0
Total_Runs              0
dtype: int64

Sorted by score: True


,Final_Rank,Rank,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,batter,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Runs,Avg_Season_Average,Avg_Season_SR,Average_SR_SD,Latest_Season,name_key,Score_Average,Score_SR,Score_Runs,Score_Consistency,Score_Recent,Score_Experience,Final_Scouting_Score,Age,Scouting_Priority,Review_Flag,IPL_Player,IPL_Matches
0,1,16,virat singh,2021/22,Jharkhand,1,Single team,6,36,0.00,0.00,Virat Singh,76.22,viratsingh,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,6,36,1236,46.94,129.46,20.37,2021/22,viratsingh,91.03,67.95,98.72,71.79,10.26,53.53,73.96,28.00,High,Statistically promising,Virat Singh,0
1,2,44,atharva taide,2024/25,Vidarbha,1,Single team,5,25,2.00,6.00,Atharva Taide,74.06,atharvataide,False,Atharwa Taide,Probable IPL Played,Unique initial + surname,Pending identity verification,5,25,631,38.51,124.99,23.80,2024/25,atharvataide,78.21,55.13,58.33,58.33,74.68,30.13,62.72,26.00,High,Statistically promising,Atharwa Taide,10
2,3,61,shivam chauhan,2021/22,Haryana,1,Single team,4,22,0.00,0.00,Shivam Chauhan,66.34,shivamchauhan,False,Saurav Chauhan,Probable IPL Played,Unique initial + surname,Pending identity verification,4,22,627,31.84,122.29,2.19,2021/22,shivamchauhan,60.90,47.44,57.69,100.00,10.26,13.78,55.34,29.00,NaN,NaN,Shivam Chauhan,0
3,4,62,shivam chaudhary,2023/24,Railways,1,Single team,5,23,2.00,7.00,Shivam Chaudhary,62.91,shivamchaudhary,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,23,586,28.24,124.84,21.94,2023/24,shivamchaudhary,45.51,53.85,51.28,66.03,82.69,30.13,54.78,29.00,High,Statistically promising,Shivam Chaudhary,0
4,5,63,amandeep khare,2023/24,Chhattisgarh,1,Single team,7,28,2.00,4.00,Amandeep Khare,40.19,amandeepkhare,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,7,28,758,30.67,111.14,21.39,2023/24,amandeepkhare,56.41,26.28,71.15,69.87,54.17,75.96,54.60,29.00,High,Review: Strike rate / sample,Amandeep Khare,0
5,6,72,rajat dey,2023/24,Tripura,1,Single team,6,21,2.00,5.00,Rajat Dey,75.94,rajatdey,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,6,21,488,28.22,130.99,27.26,2023/24,rajatdey,44.87,69.23,37.82,51.92,64.74,53.53,53.03,29.00,High,Statistically promising,Rajat Dey,0
6,7,77,shubham pundir,2023/24,Jammu & Kashmir,1,Single team,6,21,1.00,4.00,Shubham Pundir,40.96,shubhampundir,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,6,21,452,35.65,111.03,15.48,2023/24,shubhampundir,74.36,25.00,32.69,81.41,54.17,53.53,51.68,28.00,Review,Review: Strike rate / sample,Shubham Pundir,0
7,8,93,rohan kunnummal,2023/24,Kerala,1,Single team,5,13,2.00,2.00,Rohan Kunnummal,44.79,rohankunnummal,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,13,361,36.73,113.38,17.63,2023/24,rohankunnummal,75.64,30.13,21.15,77.56,33.65,30.13,47.18,28.00,Review,Review: Strike rate / sample,Rohan Kunnummal,0
8,9,105,vishvaraj jadeja,2024/25,Saurashtra,1,Single team,6,18,3.00,5.00,Vishvaraj Jadeja,38.32,vishvarajjadeja,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,6,18,315,28.38,116.43,23.88,2024/25,vishvarajjadeja,46.79,36.54,13.46,57.69,64.74,53.53,41.33,28.00,Review,Review: Strike rate / sample,Vishvaraj Jadeja,0
9,10,108,s gani,2022/23,Bihar,1,Single team,3,11,1.00,3.00,S Gani,47.84,sgani,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,3,11,252,30.60,115.86,13.47,2022/23,sgani,55.13,33.97,1.60,88.46,43.91,4.81,40.50,27.00,Review,Review: Strike rate / sample,S Gani,0



BOWLERS VALIDATION
Total selected: 10
Duplicate players: 0

Missing values:
bowler_name             0
Final_Scouting_Score    0
Total_Matches           0
Total_Wickets           0
dtype: int64

Sorted by score: True


,Final_Rank,Rank,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,bowler_name,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Wickets,Avg_Season_Economy,Avg_Season_Bowling_Average,Avg_Season_Bowling_SR,Economy_SD,Latest_Season,name_key,Score_Average,Score_Economy,Score_Wickets,Score_SR,Score_Consistency,Score_Recent,Final_Scouting_Score,Age,Scouting_Priority,Review_Flag,IPL_Player,IPL_Matches
0,1,11,khrievitso kense,2023/24,Nagaland,1,Single team,3,9,1.00,2.00,Khrievitso Kense,85.53,khrievitsokense,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,3,9,13,5.39,13.50,15.13,0.35,2023/24,khrievitsokense,97.67,99.22,5.43,89.15,100.00,31.01,75.23,22.00,High,Review: Wicket sample,Khrievitso Kense,0
1,2,25,na rathva,2023/24,Baroda,1,Single team,6,30,2.00,8.00,NA Rathva,87.61,narathva,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,27,27,7.20,18.13,15.63,1.99,2023/24,narathva,89.92,59.69,62.02,86.82,25.58,84.88,69.63,27.00,High,Statistically promising,NA Rathva,0
2,3,37,akash choudhary,2023/24,Meghalaya,1,Single team,4,18,1.00,2.00,Akash Choudhary,75.61,akashchoudhary,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,18,19,6.42,21.15,17.29,1.91,2023/24,akashchoudhary,75.19,88.37,31.78,77.52,27.91,31.01,63.22,27.00,High,Statistically promising,Akash Choudhary,0
3,4,48,pradipta pramanik,2024/25,Bengal,1,Single team,4,10,1.00,2.00,Pradipta Pramanik,66.15,pradiptapramanik,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,10,12,7.22,11.40,11.80,3.39,2024/25,pradiptapramanik,99.22,58.14,1.94,100.00,1.55,31.01,56.43,28.00,High,Review: Wicket sample,Pradipta Pramanik,0
4,5,99,kanishk seth,2024/25,Bengal,1,Single team,4,14,1.00,2.00,Kanishk Seth,28.09,kanishkseth,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,14,16,8.56,27.08,19.00,0.93,2024/25,kanishkseth,50.39,13.18,15.50,64.34,79.84,31.01,38.18,29.00,High,Review: Economy / average,Kanishk Seth,0
5,6,100,mukhtar hussain,2023/24,Assam,1,Single team,5,19,2.00,6.00,Mukhtar Hussain,47.55,mukhtarhussain,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,19,22,8.21,30.86,21.71,1.57,2023/24,mukhtarhussain,35.66,20.93,41.86,50.39,42.64,71.71,37.93,27.00,Review,Review: Economy / average,Mukhtar Hussain,0
6,7,110,mrinmoy dutta,2023/24,Assam,1,Single team,5,14,2.00,6.00,Mrinmoy Dutta,6.64,mrinmoydutta,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,14,14,9.18,27.85,19.37,2.78,2023/24,mrinmoydutta,47.29,3.10,8.53,62.79,10.85,71.71,28.39,27.00,Review,Review: Economy / average,Mrinmoy Dutta,0
7,8,120,ajay sarkar,2023/24,Tripura,1,Single team,7,24,2.00,5.00,Ajay Sarkar,28.76,ajaysarkar,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,7,24,16,7.74,38.13,33.15,1.98,2023/24,ajaysarkar,17.83,36.43,15.50,10.08,26.36,62.02,23.91,29.00,Review,Review: Economy / average,Ajay Sarkar,0
8,9,121,harishankar reddy,2023/24,Andhra,1,Single team,7,20,2.00,3.00,Harishankar Reddy,35.25,harishankarreddy,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,7,20,24,9.60,36.34,25.12,3.19,2023/24,harishankarreddy,18.60,1.55,48.06,31.01,4.65,43.02,21.92,28.00,Review,Review: Economy / average,Harishankar Reddy,0
9,10,125,tahmeed rahman,2023/24,Nagaland,1,Single team,5,21,2.00,6.00,Tahmeed Rahman,28.33,tahmeedrahman,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,21,12,7.97,38.68,30.60,2.61,2023/24,tahmeedrahman,16.28,29.46,1.94,16.28,13.18,71.71,19.17,28.00,Review,Review: Economy / average,Tahmeed Rahman,0



FINAL VALIDATION SUMMARY


,Category,Selected,Duplicates,Missing_Values
0,Batters,10,0,0
1,Bowlers,10,0,0


Total shortlisted: 20


In [166]:
# ==========================================
# PRESSURE-X: FINAL REPORT EXPORT
# ==========================================

from pathlib import Path
import pandas as pd

# Output directory
OUTPUT_DIR = DATA_DIR

# Refresh final CSV files
final_batters.to_csv(
    OUTPUT_DIR / "PRESSURE_X_Final_10_Batters.csv",
    index=False
)

final_bowlers.to_csv(
    OUTPUT_DIR / "PRESSURE_X_Final_10_Bowlers.csv",
    index=False
)

# Executive summary
executive_summary = pd.DataFrame({
    "Metric": [
        "Project",
        "Objective",
        "Target Team",
        "Target Season",
        "Total Shortlisted",
        "Emerging Batters",
        "Emerging Bowlers",
        "Data Source",
        "IPL Audit Coverage",
        "Primary Batting Prospect",
        "Primary Bowling Prospect"
    ],
    "Value": [
        "PRESSURE-X | RR SupeRR Selector Season 2",
        "Identify emerging T20 cricket talent",
        "Rajasthan Royals",
        "IPL 2027",
        20,
        10,
        10,
        "Historical domestic T20 / SMAT data",
        "Available IPL data: 2022–2026",
        "Virat Singh",
        "Khrievitso Kense"
    ]
})

# Methodology and limitations
methodology = pd.DataFrame({
    "Section": [
        "Batting Score",
        "Bowling Score",
        "Emerging Player Filter",
        "Batting Threshold",
        "Bowling Threshold",
        "Age Filter",
        "IPL Exposure",
        "Data Limitation",
        "Scouting Limitation"
    ],
    "Description": [
        "Average 25%, Strike Rate 25%, Runs 20%, Consistency 15%, Recent Matches 10%, Active Seasons 5%.",
        "Average 25%, Economy 25%, Wickets 20%, Bowling Strike Rate 15%, Economy Consistency 10%, Recent Matches 5%.",
        "Candidates filtered using available domestic T20 performance and player eligibility criteria.",
        "At least 8 matches and 250 runs.",
        "At least 8 matches and 12 wickets.",
        "Age 29 or below in the candidate pool.",
        "Appearance audit covers available IPL data from 2022–2026; it does not establish complete career exposure.",
        "SMAT delivery data available through 2024/25. Do not claim 2025/26 performance.",
        "Statistical rankings indicate performance, not guaranteed IPL suitability. Further video, role and availability checks are required."
    ]
})

# Create workbook
output_file = OUTPUT_DIR / "PRESSURE_X_Final_Scouting_Report.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    executive_summary.to_excel(
        writer, sheet_name="Executive Summary", index=False
    )

    final_batters.to_excel(
        writer, sheet_name="Batters", index=False
    )

    final_bowlers.to_excel(
        writer, sheet_name="Bowlers", index=False
    )

    methodology.to_excel(
        writer, sheet_name="Methodology", index=False
    )

    # Include supporting analysis if available
    if "shortlisted_phase_stats" in globals() and not shortlisted_phase_stats.empty:
        shortlisted_phase_stats.to_excel(
            writer, sheet_name="Bowling Phases", index=False
        )

    if "recruitment_matrix" in globals() and not recruitment_matrix.empty:
        recruitment_matrix.to_excel(
            writer, sheet_name="Bowling Recruitment", index=False
        )

    # Basic spreadsheet formatting
    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for column_cells in worksheet.columns:
            max_length = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in column_cells
            )
            worksheet.column_dimensions[
                column_cells[0].column_letter
            ].width = min(max_length + 2, 35)

print("FINAL REPORT GENERATED")
print("File:", output_file)
print("Batters:", len(final_batters))
print("Bowlers:", len(final_bowlers))
print("Total:", len(final_batters) + len(final_bowlers))
print("Sheets:", pd.ExcelFile(output_file).sheet_names)

FINAL REPORT GENERATED
File: C:\Users\pavan\PRESSURE_X_Final_Scouting_Report.xlsx
Batters: 10
Bowlers: 10
Total: 20
Sheets: ['Executive Summary', 'Batters', 'Bowlers', 'Methodology', 'Bowling Phases', 'Bowling Recruitment']


In [167]:
# ==========================================
# PRESSURE-X: IPL IDENTITY REVIEW
# ==========================================

ipl_review = pd.DataFrame([
    ["Virat Singh", 3, "Verified historical IPL appearance", "2021"],
    ["Atharva Taide", 10, "Recorded 2022-2026; career total to verify", "2022-2026"],
    ["Shivam Chauhan", None, "Possible Saurav Chauhan match; unverified", "Pending"]
], columns=[
    "Player",
    "Verified_or_Recorded_Matches",
    "Verification_Status",
    "Coverage"
])

display(ipl_review)

ipl_review.to_csv(
    DATA_DIR / "PRESSURE_X_IPL_Identity_Review.csv",
    index=False
)

,Player,Verified_or_Recorded_Matches,Verification_Status,Coverage
0,Virat Singh,3.00,Verified historical IPL appearance,2021
1,Atharva Taide,10.00,Recorded 2022-2026; career total to verify,2022-2026
2,Shivam Chauhan,NaN,Possible Saurav Chauhan match; unverified,Pending


In [171]:
# ==========================================
# PRESSURE-X: UPDATE IPL AUDIT IN FINAL REPORT
# ==========================================

ipl_audit = {
    "Virat Singh": {
        "Known_IPL_Matches": 3,
        "IPL_Audit_Status": "Verified: 2021 appearances",
        "IPL_Audit_Coverage": "2021"
    },
    "Atharva Taide": {
        "Known_IPL_Matches": 10,
        "IPL_Audit_Status": "Recorded: career total unverified",
        "IPL_Audit_Coverage": "2022-2026"
    },
    "Shivam Chauhan": {
        "Known_IPL_Matches": np.nan,
        "IPL_Audit_Status": "Identity verification pending",
        "IPL_Audit_Coverage": "Unverified"
    }
}

for col in [
    "Known_IPL_Matches",
    "IPL_Audit_Status",
    "IPL_Audit_Coverage"
]:
    final_batters[col] = final_batters["batter"].map(
        {name: values[col] for name, values in ipl_audit.items()}
    )

# Save updated batter shortlist
final_batters.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Batters.csv",
    index=False
)

# Update Batters sheet in existing workbook
with pd.ExcelWriter(
    DATA_DIR / "PRESSURE_X_Final_Scouting_Report.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    final_batters.to_excel(
        writer,
        sheet_name="Batters",
        index=False
    )

display(final_batters[[
    "Final_Rank",
    "batter",
    "Total_Matches",
    "Total_Runs",
    "Known_IPL_Matches",
    "IPL_Audit_Status",
    "IPL_Audit_Coverage"
]])

print("IPL audit added to batter CSV and Excel workbook.")

,Final_Rank,batter,Total_Matches,Total_Runs,Known_IPL_Matches,IPL_Audit_Status,IPL_Audit_Coverage
0,1,Virat Singh,36,1236,3.00,Verified: 2021 appearances,2021
1,2,Atharva Taide,25,631,10.00,Recorded: career total unverified,2022-2026
2,3,Shivam Chauhan,22,627,NaN,Identity verification pending,Unverified
3,4,Shivam Chaudhary,23,586,NaN,NaN,NaN
4,5,Amandeep Khare,28,758,NaN,NaN,NaN
5,6,Rajat Dey,21,488,NaN,NaN,NaN
6,7,Shubham Pundir,21,452,NaN,NaN,NaN
7,8,Rohan Kunnummal,13,361,NaN,NaN,NaN
8,9,Vishvaraj Jadeja,18,315,NaN,NaN,NaN
9,10,S Gani,11,252,NaN,NaN,NaN


IPL audit added to batter CSV and Excel workbook.


In [170]:
mask = final_batters["batter"].eq("Shivam Chauhan")

final_batters.loc[mask, "Known_IPL_Matches"] = 0
final_batters.loc[mask, "IPL_Audit_Status"] = "Verified: No IPL appearances found"
final_batters.loc[mask, "IPL_Audit_Coverage"] = "Career profile"

final_batters.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Batters.csv",
    index=False
)

with pd.ExcelWriter(
    DATA_DIR / "PRESSURE_X_Final_Scouting_Report.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    final_batters.to_excel(writer, sheet_name="Batters", index=False)

print("Shivam Chauhan identity audit updated.")

Shivam Chauhan identity audit updated.


In [172]:
# ==========================================
# PRESSURE-X: UPDATE IPL AUDIT IN FINAL REPORT
# ==========================================

ipl_audit = {
    "Virat Singh": {
        "Known_IPL_Matches": 3,
        "IPL_Audit_Status": "Verified: 2021 appearances",
        "IPL_Audit_Coverage": "2021"
    },
    "Atharva Taide": {
        "Known_IPL_Matches": 10,
        "IPL_Audit_Status": "Recorded: career total unverified",
        "IPL_Audit_Coverage": "2022-2026"
    },
    "Shivam Chauhan": {
        "Known_IPL_Matches": np.nan,
        "IPL_Audit_Status": "Identity verification pending",
        "IPL_Audit_Coverage": "Unverified"
    }
}

for col in [
    "Known_IPL_Matches",
    "IPL_Audit_Status",
    "IPL_Audit_Coverage"
]:
    final_batters[col] = final_batters["batter"].map(
        {name: values[col] for name, values in ipl_audit.items()}
    )

# Save updated batter shortlist
final_batters.to_csv(
    DATA_DIR / "PRESSURE_X_Final_10_Batters.csv",
    index=False
)

# Update Batters sheet in existing workbook
with pd.ExcelWriter(
    DATA_DIR / "PRESSURE_X_Final_Scouting_Report.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    final_batters.to_excel(
        writer,
        sheet_name="Batters",
        index=False
    )

display(final_batters[[
    "Final_Rank",
    "batter",
    "Total_Matches",
    "Total_Runs",
    "Known_IPL_Matches",
    "IPL_Audit_Status",
    "IPL_Audit_Coverage"
]])

print("IPL audit added to batter CSV and Excel workbook.")

,Final_Rank,batter,Total_Matches,Total_Runs,Known_IPL_Matches,IPL_Audit_Status,IPL_Audit_Coverage
0,1,Virat Singh,36,1236,3.00,Verified: 2021 appearances,2021
1,2,Atharva Taide,25,631,10.00,Recorded: career total unverified,2022-2026
2,3,Shivam Chauhan,22,627,NaN,Identity verification pending,Unverified
3,4,Shivam Chaudhary,23,586,NaN,NaN,NaN
4,5,Amandeep Khare,28,758,NaN,NaN,NaN
5,6,Rajat Dey,21,488,NaN,NaN,NaN
6,7,Shubham Pundir,21,452,NaN,NaN,NaN
7,8,Rohan Kunnummal,13,361,NaN,NaN,NaN
8,9,Vishvaraj Jadeja,18,315,NaN,NaN,NaN
9,10,S Gani,11,252,NaN,NaN,NaN


IPL audit added to batter CSV and Excel workbook.


In [173]:
# ==========================================
# PRESSURE-X: FINAL SCOUTING RECOMMENDATIONS
# ==========================================

import pandas as pd

# Select priority prospects using existing rankings
priority_batters = final_batters.head(3).copy()
priority_bowlers = final_bowlers.head(4).copy()

priority_batters["Category"] = "Priority Batter"
priority_bowlers["Category"] = "Priority Bowler"

priority_batters["Recommendation"] = "Detailed scouting and video assessment"
priority_bowlers["Recommendation"] = "Detailed scouting and video assessment"

# Remaining candidates
other_batters = final_batters.iloc[3:].copy()
other_bowlers = final_bowlers.iloc[4:].copy()

other_batters["Category"] = "Secondary Batter"
other_bowlers["Category"] = "Secondary Bowler"

other_batters["Recommendation"] = "Monitor performance and reassess"
other_bowlers["Recommendation"] = "Monitor performance and reassess"

recommendations = pd.concat(
    [priority_batters, priority_bowlers,
     other_batters, other_bowlers],
    ignore_index=True
)

# Keep the report compact
recommendation_columns = [
    "Category",
    "batter" if "batter" in recommendations.columns else "bowler_name",
    "Final_Rank",
    "Final_Scouting_Score",
    "Recommendation"
]

recommendation_columns = [
    c for c in recommendation_columns if c in recommendations.columns
]

recommendations = recommendations[recommendation_columns]

limitations = pd.DataFrame({
    "Limitation": [
        "Historical SMAT delivery data ends in 2024/25.",
        "IPL career appearance verification is incomplete for some candidates.",
        "Statistical rankings do not establish role suitability or future IPL success.",
        "Candidates require video analysis, role assessment and availability checks."
    ]
})

output_file = DATA_DIR / "PRESSURE_X_Final_Scouting_Report.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    recommendations.to_excel(
        writer, sheet_name="Scouting Recommendations", index=False
    )
    limitations.to_excel(
        writer, sheet_name="Limitations", index=False
    )

print("FINAL RECOMMENDATIONS ADDED")
print("Priority batters:", len(priority_batters))
print("Priority bowlers:", len(priority_bowlers))
print("Total candidates:", len(recommendations))
print("Workbook:", output_file)
print("Sheets:", pd.ExcelFile(output_file).sheet_names)

FINAL RECOMMENDATIONS ADDED
Priority batters: 3
Priority bowlers: 4
Total candidates: 20
Workbook: C:\Users\pavan\PRESSURE_X_Final_Scouting_Report.xlsx
Sheets: ['Executive Summary', 'Batters', 'Bowlers', 'Methodology', 'Bowling Phases', 'Bowling Recruitment', 'Scouting Recommendations', 'Limitations']


In [174]:
# Select and display the top 10 batters
top_10_batters = (
    final_batters
    .sort_values("Final_Rank")
    .head(10)
    [["Final_Rank", "batter", "Total_Matches",
      "Total_Runs", "Final_Scouting_Score"]]
    .reset_index(drop=True)
)

display(top_10_batters)

# Export to CSV
top_10_batters.to_csv(
    "PRESSURE_X_Final_10_Batters.csv",
    index=False
)

print("Top 10 batters exported successfully!")

,Final_Rank,batter,Total_Matches,Total_Runs,Final_Scouting_Score
0,1,Virat Singh,36,1236,73.96
1,2,Atharva Taide,25,631,62.72
2,3,Shivam Chauhan,22,627,55.34
3,4,Shivam Chaudhary,23,586,54.78
4,5,Amandeep Khare,28,758,54.60
5,6,Rajat Dey,21,488,53.03
6,7,Shubham Pundir,21,452,51.68
7,8,Rohan Kunnummal,13,361,47.18
8,9,Vishvaraj Jadeja,18,315,41.33
9,10,S Gani,11,252,40.50


Top 10 batters exported successfully!


In [175]:
# Select and display the top 10 bowlers
top_10_bowlers = (
    final_bowlers
    .sort_values("Final_Rank")
    .head(10)
    [["Final_Rank", "bowler_name", "Total_Matches",
      "Total_Wickets", "Final_Scouting_Score"]]
    .reset_index(drop=True)
)

display(top_10_bowlers)

# Export to CSV
top_10_bowlers.to_csv(
    "PRESSURE_X_Final_10_Bowlers.csv",
    index=False
)

print("Top 10 bowlers exported successfully!")

,Final_Rank,bowler_name,Total_Matches,Total_Wickets,Final_Scouting_Score
0,1,Khrievitso Kense,9,13,75.23
1,2,NA Rathva,27,27,69.63
2,3,Akash Choudhary,18,19,63.22
3,4,Pradipta Pramanik,10,12,56.43
4,5,Kanishk Seth,14,16,38.18
5,6,Mukhtar Hussain,19,22,37.93
6,7,Mrinmoy Dutta,14,14,28.39
7,8,Ajay Sarkar,24,16,23.91
8,9,Harishankar Reddy,20,24,21.92
9,10,Tahmeed Rahman,21,12,19.17


Top 10 bowlers exported successfully!


In [176]:
# Display bowling phase analysis for shortlisted bowlers

phase_table = (
    recruitment_matrix[
        ["bowler_name", "Best_Phase", "Balls",
         "Wickets", "Economy", "Phase_Score"]
    ]
    .rename(columns={
        "bowler_name": "Bowler",
        "Best_Phase": "Best phase",
        "Balls": "Balls",
        "Wickets": "Wkts",
        "Economy": "Econ",
        "Phase_Score": "Phase score"
    })
    .reset_index(drop=True)
)

display(phase_table)

# Export table
phase_table.to_csv(
    "PRESSURE_X_Bowling_Phase_Analysis.csv",
    index=False
)

print("Bowling phase analysis exported successfully!")

KeyError: "['Balls', 'Wickets', 'Economy'] not in index"

In [177]:
print("Recruitment Matrix Columns:")
print(recruitment_matrix.columns.tolist())

display(recruitment_matrix.head())

Recruitment Matrix Columns:
['Final_Rank', 'Rank', 'player_key', 'latest_smat_season', 'latest_team', 'latest_team_count', 'team_status', 'smat_seasons', 'total_smat_matches', 'recent_seasons', 'recent_smat_matches', 'bowler_name', 'scouting_score', 'match_key', 'exact_ipl_match', 'IPL_Player_Match', 'IPL_Match_Status', 'IPL_Match_Type', 'IPL_Exposure', 'Active_Seasons', 'Total_Matches', 'Total_Wickets', 'Avg_Season_Economy', 'Avg_Season_Bowling_Average', 'Avg_Season_Bowling_SR', 'Economy_SD', 'Latest_Season', 'name_key', 'Score_Average', 'Score_Economy', 'Score_Wickets', 'Score_SR', 'Score_Consistency', 'Score_Recent', 'Final_Scouting_Score', 'Age', 'Scouting_Priority', 'Review_Flag', 'Best_Phase', 'Best_Phase_Balls', 'Best_Phase_Wickets', 'Best_Phase_Economy', 'Phase_Score']


,Final_Rank,Rank,player_key,latest_smat_season,latest_team,latest_team_count,team_status,smat_seasons,total_smat_matches,recent_seasons,recent_smat_matches,bowler_name,scouting_score,match_key,exact_ipl_match,IPL_Player_Match,IPL_Match_Status,IPL_Match_Type,IPL_Exposure,Active_Seasons,Total_Matches,Total_Wickets,Avg_Season_Economy,Avg_Season_Bowling_Average,Avg_Season_Bowling_SR,Economy_SD,Latest_Season,name_key,Score_Average,Score_Economy,Score_Wickets,Score_SR,Score_Consistency,Score_Recent,Final_Scouting_Score,Age,Scouting_Priority,Review_Flag,Best_Phase,Best_Phase_Balls,Best_Phase_Wickets,Best_Phase_Economy,Phase_Score
0,1,11,khrievitso kense,2023/24,Nagaland,1,Single team,3,9,1.00,2.00,Khrievitso Kense,85.53,khrievitsokense,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,3,9,13,5.39,13.50,15.13,0.35,2023/24,khrievitsokense,97.67,99.22,5.43,89.15,100.00,31.01,75.23,22.00,High,Review: Wicket sample,Middle Overs,167,12,5.60,80.00
1,2,25,na rathva,2023/24,Baroda,1,Single team,6,30,2.00,8.00,NA Rathva,87.61,narathva,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,5,27,27,7.20,18.13,15.63,1.99,2023/24,narathva,89.92,59.69,62.02,86.82,25.58,84.88,69.63,27.00,High,Statistically promising,Middle Overs,318,15,6.87,70.00
2,3,37,akash choudhary,2023/24,Meghalaya,1,Single team,4,18,1.00,2.00,Akash Choudhary,75.61,akashchoudhary,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,18,19,6.42,21.15,17.29,1.91,2023/24,akashchoudhary,75.19,88.37,31.78,77.52,27.91,31.01,63.22,27.00,High,Statistically promising,Powerplay,210,9,6.54,75.00
3,4,48,pradipta pramanik,2024/25,Bengal,1,Single team,4,10,1.00,2.00,Pradipta Pramanik,66.15,pradiptapramanik,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,10,12,7.22,11.40,11.80,3.39,2024/25,pradiptapramanik,99.22,58.14,1.94,100.00,1.55,31.01,56.43,28.00,High,Review: Wicket sample,Middle Overs,132,8,6.32,65.00
4,5,99,kanishk seth,2024/25,Bengal,1,Single team,4,14,1.00,2.00,Kanishk Seth,28.09,kanishkseth,False,,No IPL Match Found,No identity match,No appearance found in available IPL data,4,14,16,8.56,27.08,19.00,0.93,2024/25,kanishkseth,50.39,13.18,15.50,64.34,79.84,31.01,38.18,29.00,High,Review: Economy / average,Powerplay,192,7,8.00,20.00


In [178]:
# Bowling Phase Analysis Table

phase_table = (
    recruitment_matrix[
        [
            "bowler_name",
            "Best_Phase",
            "Best_Phase_Balls",
            "Best_Phase_Wickets",
            "Best_Phase_Economy",
            "Phase_Score"
        ]
    ]
    .rename(columns={
        "bowler_name": "Bowler",
        "Best_Phase": "Best phase",
        "Best_Phase_Balls": "Balls",
        "Best_Phase_Wickets": "Wkts",
        "Best_Phase_Economy": "Econ",
        "Phase_Score": "Phase score"
    })
    .sort_values("Phase score", ascending=False)
    .reset_index(drop=True)
)

display(phase_table)

# Export CSV
phase_table.to_csv(
    "PRESSURE_X_Bowling_Phase_Analysis.csv",
    index=False
)

print("Bowling Phase Analysis exported successfully!")

,Bowler,Best phase,Balls,Wkts,Econ,Phase score
0,Khrievitso Kense,Middle Overs,167,12,5.60,80.00
1,Akash Choudhary,Powerplay,210,9,6.54,75.00
2,NA Rathva,Middle Overs,318,15,6.87,70.00
3,Pradipta Pramanik,Middle Overs,132,8,6.32,65.00
4,Ajay Sarkar,Powerplay,204,9,7.18,65.00
5,Mukhtar Hussain,Death Overs,96,8,9.06,55.00
6,Harishankar Reddy,Death Overs,131,9,10.58,55.00
7,Mrinmoy Dutta,Middle Overs,60,3,4.40,52.50
8,Tahmeed Rahman,Middle Overs,279,11,8.28,40.00
9,Kanishk Seth,Powerplay,192,7,8.00,20.00


Bowling Phase Analysis exported successfully!


In [179]:
# PRESSURE-X: Final Quality Check

import pandas as pd
from pathlib import Path

reports = {
    "Top 10 Batters": top_10_batters,
    "Top 10 Bowlers": top_10_bowlers,
    "Bowling Phase Analysis": phase_table
}

print("=" * 55)
print("       PRESSURE-X FINAL VALIDATION REPORT")
print("=" * 55)

for name, data in reports.items():
    print(f"\n{name}")
    print("-" * 40)
    print("Rows:", len(data))
    print("Columns:", len(data.columns))
    print("Missing values:", data.isnull().sum().sum())
    print("Duplicate rows:", data.duplicated().sum())

    if "Bowler" in data.columns:
        print("Duplicate players:", data["Bowler"].duplicated().sum())
    elif "batter" in data.columns:
        print("Duplicate players:", data["batter"].duplicated().sum())
    elif "bowler_name" in data.columns:
        print("Duplicate players:", data["bowler_name"].duplicated().sum())

    print("Status:", "PASS" if len(data) == 10 else "CHECK")

# Verify exported files
files = [
    "PRESSURE_X_Final_10_Batters.csv",
    "PRESSURE_X_Final_10_Bowlers.csv",
    "PRESSURE_X_Bowling_Phase_Analysis.csv"
]

print("\n" + "=" * 55)
print("EXPORTED FILE CHECK")
print("=" * 55)

for file in files:
    print(f"{file}: {'FOUND' if Path(file).exists() else 'NOT FOUND'}")

print("\nFinal validation completed.")

       PRESSURE-X FINAL VALIDATION REPORT

Top 10 Batters
----------------------------------------
Rows: 10
Columns: 5
Missing values: 0
Duplicate rows: 0
Duplicate players: 0
Status: PASS

Top 10 Bowlers
----------------------------------------
Rows: 10
Columns: 5
Missing values: 0
Duplicate rows: 0
Duplicate players: 0
Status: PASS

Bowling Phase Analysis
----------------------------------------
Rows: 10
Columns: 6
Missing values: 0
Duplicate rows: 0
Duplicate players: 0
Status: PASS

EXPORTED FILE CHECK
PRESSURE_X_Final_10_Batters.csv: FOUND
PRESSURE_X_Final_10_Bowlers.csv: FOUND
PRESSURE_X_Bowling_Phase_Analysis.csv: FOUND

Final validation completed.
